# 🌿 JungleSurvivor v2 — Kaggle 完整 Notebook**叢林求生離線 AI 助手 — 結構化特徵辨識 Pipeline**### 架構- **Gemma 4** 只做「看圖填表」— 從照片萃取結構化植物特徵 (JSON)- **演算法引擎** 做比對、計分、排序 — 確定性結果，零 LLM 算力- **50 種台灣常見植物** 知識庫，含危險/可食/藥用分類- **混淆物種警告** + 野外實測指引### 測試項目1. 單張照片辨識2. 多張照片特徵合併3. 使用者手動修正特徵4. 危險物種警告 + 混淆對提示5. Gradio 互動介面

In [ ]:
# Cell 1: 安裝依賴
# Pillow 不要升級 — Kaggle 預裝版本與 torchvision 相容
!pip install -q --upgrade transformers accelerate gradio

import json, re, os, sys, time, torch
import requests
from io import BytesIO
from pathlib import Path
from PIL import Image
from dataclasses import dataclass, field
from typing import Any, Optional, Callable
import heapq

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

In [ ]:
# Cell 2: 全域設定

MODEL_ID = "google/gemma-4-E4B-it"
MAX_NEW_TOKENS = 4096
DTYPE = torch.bfloat16 if torch.cuda.is_available() else torch.float32

print(f"✅ Model: {MODEL_ID}")
print(f"✅ Dtype: {DTYPE}")

## 🧮 核心引擎 — Scoring & Matching確定性演算法：特徵比對、加權計分、Top-N 排序、Early Stopping 剪枝。

In [ ]:
# Cell 3: Scoring Engine
"""
JungleSurvivor v2 — Scoring Engine

Implements the deterministic scoring algorithm from Plan Section 7:
  - 7.1: Only positive scoring (match → +weight, mismatch → 0)
  - 7.2: Single-value scoring
  - 7.3: Array scoring with max(|observed|, |kb|) denominator
  - 7.4: Confidence = score / effective_total
"""

from dataclasses import dataclass, field
from typing import Any


SKIP_VALUES = frozenset({"not_visible", "uncertain", "not_checkable"})


@dataclass
class ScoreResult:
    species_id: str
    species_name: str
    category: str
    danger_level: str | None
    score: float
    effective_total: float
    confidence: float  # 0-100
    matched_features: list[dict] = field(default_factory=list)


def should_skip(value: Any) -> bool:
    """Check if an observed value should be skipped (not_visible, uncertain, etc.)."""
    if value is None:
        return True
    if isinstance(value, str) and value in SKIP_VALUES:
        return True
    if isinstance(value, list) and len(value) == 0:
        return True
    return False


def score_single(observed_value: str, kb_value: str, weight: int) -> float:
    """Score a single-value attribute. Plan 7.2."""
    if should_skip(observed_value):
        return 0.0
    if observed_value == kb_value:
        return float(weight)
    return 0.0


def score_array(observed_values: list[str], kb_values: list[str], weight: int) -> float:
    """
    Score an array-type attribute. Plan 7.3.
    Formula: weight * |intersection| / max(|observed|, |kb|)
    """
    if should_skip(observed_values) or not kb_values:
        return 0.0

    obs_set = set(observed_values)
    kb_set = set(kb_values)
    intersection = obs_set & kb_set
    denominator = max(len(obs_set), len(kb_set))

    if denominator == 0:
        return 0.0

    return weight * len(intersection) / denominator


def score_attribute(observed_value: Any, kb_value: Any, weight: int, attr_type: str) -> float:
    """Score a single attribute based on its type."""
    if attr_type == "array":
        obs = observed_value if isinstance(observed_value, list) else [observed_value]
        kb = kb_value if isinstance(kb_value, list) else [kb_value]
        return score_array(obs, kb, weight)
    elif attr_type == "boolean":
        return 0.0
    else:
        return score_single(str(observed_value), str(kb_value), weight)


def compute_effective_total(
    schema: dict,
    observed_features: dict,
    species_features: dict,
    has_photo: bool = True,
) -> float:
    """
    Compute effective_total for confidence calculation. Plan 7.4.

    effective_total = sum of weights for:
      - All photo_observable attributes (if photo uploaded)
      - All non-photo attributes that user manually provided
    """
    total = 0.0
    schema_clean = {k: v for k, v in schema.items() if not k.startswith("_")}

    for section_name, section_schema in schema_clean.items():
        sp_section = species_features.get(section_name)
        if sp_section is None:
            continue

        obs_section = observed_features.get(section_name, {})

        for attr_name, attr_def in section_schema.items():
            if attr_def["type"] == "boolean":
                continue

            sp_attr = sp_section.get(attr_name)
            if sp_attr is None:
                continue

            is_photo_obs = attr_def.get("photo_observable", True)
            obs_value = obs_section.get(attr_name)

            if is_photo_obs and has_photo:
                total += sp_attr.get("weight", 1)
            elif not is_photo_obs and obs_value is not None and not should_skip(obs_value):
                total += sp_attr.get("weight", 1)

    return total


def score_species(
    observed_features: dict,
    species: dict,
    schema: dict,
    has_photo: bool = True,
) -> ScoreResult:
    """
    Score a single species against observed features.
    Returns ScoreResult with score, effective_total, and confidence.
    """
    sp_features = species["features"]
    schema_clean = {k: v for k, v in schema.items() if not k.startswith("_")}

    score = 0.0
    matched = []

    for section_name, section_schema in schema_clean.items():
        sp_section = sp_features.get(section_name)
        if sp_section is None:
            continue

        obs_section = observed_features.get(section_name, {})

        for attr_name, attr_def in section_schema.items():
            if attr_def["type"] == "boolean":
                continue

            sp_attr = sp_section.get(attr_name)
            if sp_attr is None:
                continue

            obs_value = obs_section.get(attr_name)
            if should_skip(obs_value):
                continue

            kb_value = sp_attr["value"]
            weight = sp_attr.get("weight", 1)

            attr_score = score_attribute(obs_value, kb_value, weight, attr_def["type"])
            score += attr_score

            if attr_score > 0:
                matched.append({
                    "section": section_name,
                    "attribute": attr_name,
                    "observed": obs_value,
                    "kb_value": kb_value,
                    "score": attr_score,
                    "weight": weight,
                })

    effective_total = compute_effective_total(schema, observed_features, sp_features, has_photo)
    confidence = (score / effective_total * 100) if effective_total > 0 else 0.0

    return ScoreResult(
        species_id=species["id"],
        species_name=species.get("common_names", {}).get("zh-TW", species["id"]),
        category=species.get("category", "unknown"),
        danger_level=species.get("danger_level"),
        score=score,
        effective_total=effective_total,
        confidence=round(confidence, 1),
        matched_features=matched,
    )


## 🔍 LLM 特徵萃取器從 LLM 回應中提取 JSON → 驗證 → 清理，容錯截斷修復。

In [ ]:
# Cell 4: Feature Extractor
"""
JungleSurvivor v2 — LLM Feature Extractor

Implements Plan Section 5:
  - 5.1: LLM only does "fill in the blanks" from photo
  - 5.2: Per-photo extraction with immediate feedback
  - 5.3: Prompt template auto-generated from KB enums
  - 5.4: JSON validation + error tolerance
"""

import json
import re
from pathlib import Path
from dataclasses import dataclass, field
from typing import Any, Optional


SKIP_VALUES = frozenset({"not_visible", "uncertain", "not_checkable"})

_ZH_STRIP_RE = re.compile(r'^([a-zA-Z0-9_<>.\-]+)\(.*\)$')

def _strip_zh_annotation(val: str) -> str:
    """Strip Chinese annotation from values like 'glossy(光滑反光(像打蠟))' → 'glossy'."""
    m = _ZH_STRIP_RE.match(val)
    return m.group(1) if m else val


@dataclass
class ExtractionResult:
    raw_response: str
    features: Optional[dict]
    success: bool
    error: Optional[str] = None
    warnings: list[str] = field(default_factory=list)


# ── JSON Extraction (reusing v1 repair logic) ──

def _extract_json_from_codeblock(text: str) -> Optional[str]:
    m = re.search(r'```(?:json)?\s*\n?([\s\S]*?)```', text)
    if m:
        return m.group(1).strip()
    m = re.search(r'```(?:json)?\s*\n?([\s\S]+)', text)
    if m:
        return m.group(1).strip()
    return None


def _extract_json_fallback(text: str) -> Optional[str]:
    start = text.find('{')
    if start < 0:
        return None
    candidate = text[start:]
    end = candidate.rfind('}')
    if end >= 0:
        return candidate[:end + 1].strip()
    return candidate.strip()


def _repair_truncated_json(json_str: str) -> Optional[str]:
    candidate = json_str.rstrip()
    while candidate.endswith(',') or candidate.endswith(':'):
        candidate = candidate[:-1].rstrip()

    opens = candidate.count('{') - candidate.count('}')
    brackets = candidate.count('[') - candidate.count(']')

    if opens <= 0 and brackets <= 0:
        return None

    candidate += ']' * max(0, brackets) + '}' * max(0, opens)
    return candidate


def _try_parse(json_str: str) -> Optional[dict]:
    clean = json_str.strip()
    try:
        return json.loads(clean)
    except json.JSONDecodeError:
        pass

    repaired = _repair_truncated_json(clean)
    if repaired:
        try:
            return json.loads(repaired)
        except json.JSONDecodeError:
            pass
        cleaned = re.sub(r',\s*([}\]])', r'\1', repaired)
        try:
            return json.loads(cleaned)
        except json.JSONDecodeError:
            pass

    cleaned = re.sub(r',\s*([}\]])', r'\1', clean)
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        pass

    return None


def extract_json(text: str) -> Optional[dict]:
    """Extract JSON from LLM response using multiple strategies."""
    for extractor in [_extract_json_from_codeblock, _extract_json_fallback]:
        candidate = extractor(text)
        if candidate:
            result = _try_parse(candidate)
            if result:
                return result
    return None


# ── Feature Validation ──

def validate_and_clean_features(
    raw_features: dict,
    schema: dict,
    enums: dict,
) -> tuple[dict, list[str]]:
    """
    Validate LLM-extracted features against schema.
    Plan 5.4: invalid values → "uncertain", missing sections → skip.
    Returns cleaned features dict and list of warnings.
    """
    cleaned = {}
    warnings = []
    schema_clean = {k: v for k, v in schema.items() if not k.startswith("_")}

    # Top-level overall features may be mixed with nested sections
    overall_raw = {}
    sections_raw = {}
    for key, val in raw_features.items():
        if key in schema_clean and key != "overall":
            if isinstance(val, dict):
                sections_raw[key] = val
            else:
                warnings.append(f"Section '{key}' expected dict, got {type(val).__name__}")
        elif key in schema_clean.get("overall", {}):
            overall_raw[key] = val
        elif key == "overall" and isinstance(val, dict):
            overall_raw.update(val)
        else:
            overall_raw[key] = val

    # Process overall section
    if overall_raw:
        sections_raw["overall"] = overall_raw

    for section_name, section_schema in schema_clean.items():
        raw_section = sections_raw.get(section_name)
        if raw_section is None:
            continue

        cleaned_section = {}
        section_enums = enums.get(section_name, {})

        for attr_name, attr_def in section_schema.items():
            raw_val = raw_section.get(attr_name)
            if raw_val is None:
                continue

            attr_type = attr_def["type"]
            valid_values = section_enums.get(attr_name, attr_def.get("values", []))
            valid_strs = [str(v) for v in valid_values]

            if attr_type == "boolean":
                if isinstance(raw_val, bool):
                    cleaned_section[attr_name] = raw_val
                elif str(raw_val).lower() in ("true", "false"):
                    cleaned_section[attr_name] = str(raw_val).lower() == "true"
            elif attr_type == "array":
                if isinstance(raw_val, str):
                    raw_val = [raw_val]
                if isinstance(raw_val, list):
                    valid_items = []
                    for item in raw_val:
                        s = _strip_zh_annotation(str(item))
                        if s in valid_strs:
                            valid_items.append(s)
                        elif s not in SKIP_VALUES:
                            warnings.append(
                                f"{section_name}.{attr_name}: '{s}' not in enum, dropped"
                            )
                    if valid_items:
                        cleaned_section[attr_name] = valid_items
            else:  # single
                s = _strip_zh_annotation(str(raw_val))
                if s in valid_strs:
                    cleaned_section[attr_name] = s
                elif s in SKIP_VALUES:
                    cleaned_section[attr_name] = s
                else:
                    cleaned_section[attr_name] = "uncertain"
                    warnings.append(
                        f"{section_name}.{attr_name}: '{s}' not in enum, set to 'uncertain'"
                    )

        if cleaned_section:
            cleaned[section_name] = cleaned_section

    # Enforce photo_observable constraints:
    # Attributes that cannot be determined from photos are forced to "not_checkable"
    # regardless of what the LLM outputs. User manual input bypasses this function.
    for section_name, section_schema in schema_clean.items():
        if section_name not in cleaned:
            continue
        for attr_name, attr_def in section_schema.items():
            if attr_def.get("photo_observable", True):
                continue
            if attr_name not in cleaned[section_name]:
                continue
            val = cleaned[section_name][attr_name]
            if val not in ("not_checkable", "not_visible", "uncertain"):
                cleaned[section_name][attr_name] = "not_checkable"
                warnings.append(
                    f"{section_name}.{attr_name}: '{val}' → 'not_checkable' (non-photo attribute)"
                )

    # If a section has visible=false, strip all other attributes from that section.
    # The LLM may hallucinate features for parts it cannot see.
    for section_name in list(cleaned.keys()):
        section = cleaned[section_name]
        if section.get("visible") is False:
            kept = {"visible": False}
            removed = [k for k in section if k != "visible"]
            if removed:
                warnings.append(
                    f"{section_name}: visible=false, removed hallucinated attrs: {removed}"
                )
            cleaned[section_name] = kept

    return cleaned, warnings


def parse_llm_response(
    response: str,
    schema: dict,
    enums: dict,
) -> ExtractionResult:
    """
    Full pipeline: extract JSON → validate → clean features.
    """
    raw_json = extract_json(response)
    if raw_json is None:
        return ExtractionResult(
            raw_response=response,
            features=None,
            success=False,
            error="無法從 LLM 回應中提取 JSON",
        )

    features, warnings = validate_and_clean_features(raw_json, schema, enums)

    if not features:
        return ExtractionResult(
            raw_response=response,
            features=None,
            success=False,
            error="JSON 提取成功但無有效特徵",
            warnings=warnings,
        )

    return ExtractionResult(
        raw_response=response,
        features=features,
        success=True,
        warnings=warnings,
    )


## 🔀 多照片特徵合併 + 使用者輸入覆蓋

In [ ]:
# Cell 5: Feature Merger
"""
JungleSurvivor v2 — Feature Merger

Implements Plan Section 6:
  - 6.1: Multi-photo feature union merge
  - 6.2: User manual input (final override via tag/chip UI)
  - 6.3: Free text fields preserved as-is
"""

from typing import Any

SKIP_VALUES = frozenset({"not_visible", "uncertain", "not_checkable"})


def _is_skip(val: Any) -> bool:
    return val is None or (isinstance(val, str) and val in SKIP_VALUES)


def merge_single_values(existing: Any, new: Any) -> Any:
    """
    Merge two single-value attributes. Plan 6.1:
      - not_visible + valid → valid
      - uncertain + definite → definite
      - different definite values → convert to array (both kept)
      - same value → keep single
    """
    if _is_skip(existing):
        return new
    if _is_skip(new):
        return existing
    if existing == new:
        return existing
    # Different values → convert to array
    if isinstance(existing, list):
        if new not in existing:
            return existing + [new]
        return existing
    return [existing, new]


def merge_array_values(existing: Any, new: Any) -> list:
    """
    Merge two array-type attributes. Plan 6.1: take union.
    """
    if existing is None or (isinstance(existing, list) and len(existing) == 0):
        return new if isinstance(new, list) else [new] if new else []
    if new is None or (isinstance(new, list) and len(new) == 0):
        return existing if isinstance(existing, list) else [existing]

    ex_list = existing if isinstance(existing, list) else [existing]
    new_list = new if isinstance(new, list) else [new]

    merged = list(ex_list)
    for v in new_list:
        if v not in merged and not _is_skip(v):
            merged.append(v)
    # Remove skip values if any real values exist
    real_vals = [v for v in merged if not _is_skip(v)]
    return real_vals if real_vals else merged


def merge_features(
    existing: dict,
    new_features: dict,
    schema: dict,
) -> dict:
    """
    Merge new photo features into existing accumulated features.
    Plan 6.1: multi-photo union merge.

    Args:
        existing: accumulated features so far (may be empty {})
        new_features: features from the latest photo
        schema: feature_schema.json content

    Returns:
        merged features dict
    """
    schema_clean = {k: v for k, v in schema.items() if not k.startswith("_")}
    merged = {}

    # Collect all section names from both
    all_sections = set()
    all_sections.update(existing.keys())
    all_sections.update(new_features.keys())

    for section_name in all_sections:
        ex_section = existing.get(section_name, {})
        new_section = new_features.get(section_name, {})
        section_schema = schema_clean.get(section_name, {})

        merged_section = {}
        all_attrs = set()
        all_attrs.update(ex_section.keys())
        all_attrs.update(new_section.keys())

        for attr_name in all_attrs:
            ex_val = ex_section.get(attr_name)
            new_val = new_section.get(attr_name)

            attr_def = section_schema.get(attr_name)
            if attr_def and attr_def["type"] == "array":
                merged_section[attr_name] = merge_array_values(ex_val, new_val)
            elif attr_def and attr_def["type"] == "boolean":
                merged_section[attr_name] = new_val if new_val is not None else ex_val
            else:
                if ex_val is None:
                    merged_section[attr_name] = new_val
                elif new_val is None:
                    merged_section[attr_name] = ex_val
                else:
                    merged_section[attr_name] = merge_single_values(ex_val, new_val)

        if merged_section:
            merged[section_name] = merged_section

    return merged


def apply_user_input(
    features: dict,
    user_features: dict,
) -> dict:
    """
    Apply user manual input. Plan 6.2: user values are the final truth.
    The tag/chip UI means what the user sees is what gets used.

    user_features has the same structure as features.
    Any value present in user_features completely replaces the AI value.
    """
    result = {}
    all_sections = set()
    all_sections.update(features.keys())
    all_sections.update(user_features.keys())

    for section_name in all_sections:
        ai_section = features.get(section_name, {})
        user_section = user_features.get(section_name, {})

        merged_section = dict(ai_section)
        for attr_name, user_val in user_section.items():
            if user_val is not None:
                merged_section[attr_name] = user_val

        if merged_section:
            result[section_name] = merged_section

    return result


def get_feature_summary(features: dict, schema: dict) -> dict:
    """
    Generate a summary of filled vs unfilled features for UI display.
    Returns dict with sections and their completion status.
    """
    schema_clean = {k: v for k, v in schema.items() if not k.startswith("_")}
    summary = {}

    for section_name, section_schema in schema_clean.items():
        section = features.get(section_name, {})
        total = 0
        filled = 0
        missing = []

        for attr_name, attr_def in section_schema.items():
            if attr_def["type"] == "boolean":
                continue
            total += 1
            val = section.get(attr_name)
            if val is not None and not _is_skip(val):
                filled += 1
            else:
                missing.append(attr_name)

        summary[section_name] = {
            "total": total,
            "filled": filled,
            "missing": missing,
        }

    return summary


## 🎯 Species Matcher (Early Stopping)Branch-and-bound 剪枝，效率比暴力搜尋高 2-3 倍。

In [ ]:
# Cell 6: Matcher
"""
JungleSurvivor v2 — Species Matcher with Early Stopping

Implements Plan Section 7.5: Branch-and-bound pruning.
Compares observed features against all KB species and returns Top N.
"""

import heapq
import json
from pathlib import Path
from typing import Any



def match_top_n(
    observed_features: dict,
    plants: list[dict],
    schema: dict,
    has_photo: bool = True,
    top_n: int = 3,
    use_early_stopping: bool = True,
) -> list[ScoreResult]:
    """
    Match observed features against all species in KB, return Top N.

    Uses Early Stopping (Plan 7.5): if remaining max possible score
    cannot beat current Nth best, skip that species early.
    """
    schema_clean = {k: v for k, v in schema.items() if not k.startswith("_")}

    if not use_early_stopping:
        results = []
        for species in plants:
            result = score_species(observed_features, species, schema, has_photo)
            results.append(result)
        results.sort(key=lambda r: (-r.score, -r.confidence, r.species_id))
        return results[:top_n]

    # Early stopping version
    # min-heap of (score, tiebreak, result) — keeps track of the current top N scores
    # tiebreak uses (-confidence, species_id) to match brute-force sort
    top_heap: list[tuple[float, float, str, ScoreResult]] = []
    min_top_score = 0.0

    for idx, species in enumerate(plants):
        sp_features = species["features"]

        # Collect all scorable attributes and their max possible contribution
        attrs_to_score = []
        for section_name, section_schema in schema_clean.items():
            sp_section = sp_features.get(section_name)
            if sp_section is None:
                continue
            obs_section = observed_features.get(section_name, {})

            for attr_name, attr_def in section_schema.items():
                if attr_def["type"] == "boolean":
                    continue
                sp_attr = sp_section.get(attr_name)
                if sp_attr is None:
                    continue
                obs_value = obs_section.get(attr_name)
                if should_skip(obs_value):
                    continue

                weight = sp_attr.get("weight", 1)
                attrs_to_score.append((section_name, attr_name, attr_def, sp_attr, obs_value, weight))

        total_possible = sum(a[5] for a in attrs_to_score)

        # If total possible cannot beat current min_top_score, skip entirely
        if len(top_heap) >= top_n and total_possible < min_top_score:
            continue

        # Score with early stopping
        score = 0.0
        compared_weight = 0.0
        pruned = False

        for section_name, attr_name, attr_def, sp_attr, obs_value, weight in attrs_to_score:
            attr_score = score_attribute(obs_value, sp_attr["value"], weight, attr_def["type"])
            score += attr_score
            compared_weight += weight
            remaining_max = total_possible - compared_weight

            if len(top_heap) >= top_n and (score + remaining_max) < min_top_score:
                pruned = True
                break

        if pruned:
            continue

        effective_total = compute_effective_total(schema, observed_features, sp_features, has_photo)
        confidence = (score / effective_total * 100) if effective_total > 0 else 0.0

        result = ScoreResult(
            species_id=species["id"],
            species_name=species.get("common_names", {}).get("zh-TW", species["id"]),
            category=species.get("category", "unknown"),
            danger_level=species.get("danger_level"),
            score=score,
            effective_total=effective_total,
            confidence=round(confidence, 1),
        )

        heap_item = (score, confidence, result.species_id, result)
        if len(top_heap) < top_n:
            heapq.heappush(top_heap, heap_item)
            if len(top_heap) == top_n:
                min_top_score = top_heap[0][0]
        elif score >= min_top_score:
            heapq.heapreplace(top_heap, heap_item)
            min_top_score = top_heap[0][0]

    # Extract results sorted consistently with brute-force
    results = [item[3] for item in sorted(top_heap, key=lambda x: (-x[0], -x[1], x[2]))]
    return results


## ⚠️ 後處理 — 警告等級 & 混淆物種嚴格優先順序：RED > ORANGE > YELLOW > GREEN > GREY

In [ ]:
# Cell 7: Postprocessor
"""
JungleSurvivor v2 — Post-processor

Implements Plan Section 8:
  - 8.1: Warning level determination (strict priority: RED > ORANGE > YELLOW > GREEN > GREY)
  - 8.2: Unknown species handling
  - 8.3: Confusion pair detection and field test guidance
  - 8.4: Top 3 result formatting
"""

from dataclasses import dataclass, field
from typing import Optional



@dataclass
class WarningInfo:
    level: str  # RED, ORANGE, YELLOW, GREEN, GREY
    color: str  # emoji indicator
    message: str


@dataclass
class ConfusionPairAlert:
    pair_id: str
    species_a_id: str
    species_b_id: str
    lethality: str
    key_differences: list[dict]


@dataclass
class ProcessedResult:
    top_results: list[ScoreResult]
    warning: WarningInfo
    confusion_alerts: list[ConfusionPairAlert] = field(default_factory=list)
    is_unknown: bool = False
    unknown_message: Optional[str] = None


def determine_warning_level(top3: list[ScoreResult]) -> WarningInfo:
    """
    Plan 8.1: Strict priority warning level determination.
    Evaluates from most dangerous to safest; returns on first match.
    """
    if not top3:
        return WarningInfo(
            level="GREY",
            color="",
            message="無法判定，請提供更多資訊。",
        )

    top1 = top3[0]

    # Priority 1: RED — Top 1 is dangerous with high confidence
    if top1.confidence >= 60 and top1.category == "dangerous":
        return WarningInfo(
            level="RED",
            color="🔴",
            message="⚠️ 高度危險！此植物極可能有毒，請勿觸碰或食用。",
        )

    # Priority 2: ORANGE — Top 3 has dangerous species with >= 40% confidence
    dangerous_in_top3 = [
        r for r in top3
        if r.category == "dangerous" and r.confidence >= 40
    ]
    if dangerous_in_top3:
        return WarningInfo(
            level="ORANGE",
            color="🟠",
            message="⚠️ 候選結果中有危險物種，請謹慎對待，建議進一步確認。",
        )

    # Priority 3: YELLOW — handled externally via confusion pair check
    # (will be set after confusion pair detection)

    # Priority 4: GREY — all confidence too low
    if top1.confidence < 40:
        return WarningInfo(
            level="GREY",
            color="⚪",
            message="❓ 無法確定此物種，知識庫中未找到高度匹配的紀錄。建議不要食用或觸碰。",
        )

    # Priority 5: GREEN — Top 1 is safe with high confidence
    if top1.confidence >= 60 and top1.category in ("edible", "medicinal"):
        return WarningInfo(
            level="GREEN",
            color="🟢",
            message="✅ 此植物可能可安全使用，但仍建議確認後再行動。",
        )

    # Priority 6: GREY — medium confidence, uncertain
    return WarningInfo(
        level="GREY",
        color="⚪",
        message="❓ 辨識結果信心度不高，建議補充更多照片或特徵資訊。",
    )


def check_confusion_pairs(
    top3: list[ScoreResult],
    confusion_db: dict,
) -> list[ConfusionPairAlert]:
    """
    Plan 8.3: Check if any Top 3 species are in known confusion pairs.
    Also check if a species' confusion partner isn't in Top 3 — still warn.
    """
    alerts = []
    top3_ids = {r.species_id for r in top3}
    pairs = confusion_db.get("confusion_pairs", [])

    for pair in pairs:
        a_id = pair["species_a"]
        b_id = pair["species_b"]

        if a_id in top3_ids or b_id in top3_ids:
            alerts.append(ConfusionPairAlert(
                pair_id=pair.get("id", ""),
                species_a_id=a_id,
                species_b_id=b_id,
                lethality=pair.get("lethality", "unknown"),
                key_differences=pair.get("key_differences", []),
            ))

    return alerts


def process_results(
    top3: list[ScoreResult],
    confusion_db: dict,
    plants: list[dict],
) -> ProcessedResult:
    """
    Full post-processing pipeline. Plan Section 8.

    1. Determine warning level
    2. Check confusion pairs
    3. If confusion found, may upgrade to YELLOW
    4. Handle unknown species (low confidence)
    """
    # Step 1: Warning level
    warning = determine_warning_level(top3)

    # Step 2: Confusion pairs
    confusion_alerts = check_confusion_pairs(top3, confusion_db)

    # Step 3: If confusion found and current warning < YELLOW, upgrade
    if confusion_alerts and warning.level in ("GREEN", "GREY"):
        has_critical = any(a.lethality in ("critical", "high") for a in confusion_alerts)
        if has_critical:
            warning = WarningInfo(
                level="YELLOW",
                color="🟡",
                message="⚠️ 候選結果中有外觀相似的安全/危險物種對，請仔細比對區分特徵。",
            )
        elif confusion_alerts:
            warning = WarningInfo(
                level="YELLOW",
                color="🟡",
                message="⚠️ 候選結果中存在容易混淆的物種，建議進一步確認。",
            )

    # Step 4: Unknown species check
    is_unknown = False
    unknown_message = None
    if top3 and top3[0].confidence < 60:
        is_unknown = True
        unknown_message = (
            "無法確定此物種。根據觀察到的特徵，"
            "這類植物在此地區的知識庫中未有高度匹配的紀錄。"
            "建議不要食用或觸碰。"
        )

    return ProcessedResult(
        top_results=top3,
        warning=warning,
        confusion_alerts=confusion_alerts,
        is_unknown=is_unknown,
        unknown_message=unknown_message,
    )


def format_result_display(
    processed: ProcessedResult,
    plants: list[dict],
) -> str:
    """
    Plan 8.4: Format Top 3 results for display.
    Returns formatted text string.
    """
    lines = []

    # Warning banner
    lines.append(f"{processed.warning.color} {processed.warning.message}")
    lines.append("")

    # Species lookup
    plants_by_id = {sp["id"]: sp for sp in plants}

    rank_emoji = ["🥇", "🥈", "🥉"]

    for i, result in enumerate(processed.top_results):
        emoji = rank_emoji[i] if i < 3 else f"#{i+1}"
        lines.append(f"{emoji} #{i+1} {result.species_name}（信心度 {result.confidence}%）")

        # Category indicator
        cat_map = {
            "dangerous": "⚠️ 有毒！",
            "edible": "🍃 可食用",
            "medicinal": "💊 藥用",
        }
        lines.append(f"   {cat_map.get(result.category, '❓ 未分類')}")

        # Diagnostic features from KB
        sp_data = plants_by_id.get(result.species_id)
        if sp_data and "human_readable" in sp_data:
            hr = sp_data["human_readable"]
            diag = hr.get("diagnostic_features", [])
            if diag:
                lines.append("")
                lines.append("   【知識庫描述 — 請對照實物確認】")
                for feat in diag[:5]:
                    lines.append(f"   ✦ {feat}")

            if result.category == "dangerous":
                tox = hr.get("toxicity", "")
                if tox:
                    lines.append(f"   🚨 毒性：{tox}")


        # Usage / survival info
        if sp_data and "usage" in sp_data:
            usage = sp_data["usage"]
            lines.append("")
            lines.append("   【生存用途資訊】")
            _type_zh = {"edible_leaf": "可食葉", "edible_root": "可食根莖",
                        "edible_fruit": "可食果實", "wound_care": "傷口處理",
                        "anti_inflammatory": "消炎解毒"}
            u_types = usage.get("type", [])
            if u_types:
                _labels = [_type_zh.get(t, t) for t in u_types]
                lines.append(f"   \U0001f4cb 用途分類：{chr(12289).join(_labels)}")
            if usage.get("edible_parts"):
                lines.append(f"   \U0001f33f 可食部位：{chr(12289).join(usage['edible_parts'])}")
            if usage.get("preparation"):
                lines.append(f"   \U0001f52a 處理方式：{usage['preparation']}")
            if usage.get("medicinal_effects"):
                lines.append(f"   \U0001f48a 藥效：{chr(12289).join(usage['medicinal_effects'])}")
            if usage.get("warnings"):
                lines.append(f"   \u26a0\ufe0f 注意：{usage['warnings']}")
        lines.append("")

    # Confusion pair alerts
    if processed.confusion_alerts:
        for alert in processed.confusion_alerts:
            sp_a = plants_by_id.get(alert.species_a_id)
            sp_b = plants_by_id.get(alert.species_b_id)
            name_a = sp_a["common_names"]["zh-TW"] if sp_a else alert.species_a_id
            name_b = sp_b["common_names"]["zh-TW"] if sp_b else alert.species_b_id

            lines.append(f"⚠️ {name_a} 與 {name_b} 是已知混淆物種對！")
            for diff in alert.key_differences:
                lines.append(f"   🔍 {diff.get('test', '')}")
            lines.append("")

    return "\n".join(lines)

## 📚 知識庫 (嵌入式)50 種台灣常見植物 + 完整結構化特徵 + 混淆物種對。所有資料內嵌於 Notebook 中，不需要外部檔案。

In [ ]:
# Cell 8: 知識庫資料 (嵌入)

FEATURE_SCHEMA = json.loads(r'''{"_version": "2.1", "_description": "JungleSurvivor v2 完整特徵 schema — 定義每個屬性的 enum 候選值、類型、photo_observable", "overall": {"growth_form": {"type": "single", "photo_observable": true, "values": ["herb", "shrub", "tree", "vine", "fern", "fungus", "grass", "succulent", "aquatic", "moss", "palm_like"]}, "height_estimate": {"type": "single", "photo_observable": true, "values": ["<30cm", "30-100cm", "1-2m", "2-5m", ">5m"]}, "latex": {"type": "single", "photo_observable": false, "values": ["none", "yes_white", "yes_yellow", "yes_red", "yes_clear"]}, "smell": {"type": "single", "photo_observable": false, "values": ["none", "aromatic", "pungent", "fishy", "minty", "spicy", "rotten", "sweet"]}, "habitat": {"type": "single", "photo_observable": true, "values": ["roadside", "forest_floor", "streamside", "cliff_rock", "open_field", "wetland", "epiphytic", "parasitic", "urban", "coastal"]}, "water_droplet_test": {"type": "single", "photo_observable": false, "values": ["beading", "flat"]}}, "leaf": {"visible": {"type": "boolean", "photo_observable": true, "values": [true, false]}, "leaf_type": {"type": "single", "photo_observable": true, "values": ["simple", "trifoliate", "pinnate_compound", "bipinnate_compound", "palmate_compound"]}, "shape": {"type": "single", "photo_observable": true, "values": ["heart", "arrow", "oval", "elliptic", "lanceolate", "linear", "needle", "scale", "round", "spatulate", "obovate", "rhombic", "fan", "kidney"]}, "edge": {"type": "single", "photo_observable": true, "values": ["entire", "serrated", "double_serrated", "crenate", "lobed", "deeply_lobed", "wavy", "spiny"]}, "tip": {"type": "single", "photo_observable": true, "values": ["acute", "acuminate", "rounded", "emarginate", "truncate"]}, "base": {"type": "single", "photo_observable": true, "values": ["cordate", "cuneate", "rounded", "truncate", "sagittate", "peltate"]}, "colors": {"type": "array", "photo_observable": true, "values": ["light_green", "green", "dark_green", "yellow_green", "red", "purple", "variegated", "silver", "brown", "red_underside"]}, "color_pattern": {"type": "single", "photo_observable": true, "values": ["solid", "gradient", "spotted", "striped", "bicolor"]}, "surface_top": {"type": "single", "photo_observable": true, "values": ["glossy", "matte", "hairy", "rough", "waxy", "velvety", "pubescent", "scaly", "sandpaper"]}, "surface_bottom": {"type": "single", "photo_observable": true, "values": ["glossy", "matte", "hairy", "rough", "waxy", "velvety", "pubescent", "scaly", "sandpaper"]}, "arrangement": {"type": "single", "photo_observable": true, "values": ["alternate", "opposite", "whorled", "rosette", "basal_rosette", "clustered", "spiral", "two_ranked", "bird_nest_radial"]}, "size": {"type": "single", "photo_observable": true, "values": ["tiny_<2cm", "small_2-5cm", "medium_5-15cm", "large_15-50cm", "very_large_>50cm"]}, "venation": {"type": "single", "photo_observable": true, "values": ["parallel", "pinnate", "palmate", "reticulate", "single_midrib"]}, "texture": {"type": "single", "photo_observable": true, "values": ["papery", "leathery", "fleshy", "membranous", "succulent"]}, "petiole_attach": {"type": "single", "photo_observable": true, "values": ["normal", "peltate_shield", "sheathing", "sessile"]}}, "stem": {"visible": {"type": "boolean", "photo_observable": true, "values": [true, false]}, "type": {"type": "single", "photo_observable": true, "values": ["erect", "creeping", "climbing", "twining", "prostrate", "rhizome", "pseudostem"]}, "cross_section": {"type": "single", "photo_observable": true, "values": ["round", "square", "triangular", "flattened", "ridged"]}, "surface": {"type": "single", "photo_observable": true, "values": ["smooth", "hairy", "thorny", "ridged", "waxy", "scaly", "bark_rough", "bark_smooth"]}, "colors": {"type": "array", "photo_observable": true, "values": ["green", "brown", "purple", "red", "gray", "white_powdery"]}, "interior": {"type": "single", "photo_observable": false, "values": ["solid", "hollow", "pithy"]}, "has_thorns": {"type": "single", "photo_observable": true, "values": ["yes", "no"]}}, "flower": {"visible": {"type": "boolean", "photo_observable": true, "values": [true, false]}, "colors": {"type": "array", "photo_observable": true, "values": ["white", "yellow", "orange", "red", "pink", "purple", "blue", "green", "brown"]}, "color_pattern": {"type": "single", "photo_observable": true, "values": ["solid", "gradient", "spotted", "striped", "center_different", "bicolor"]}, "petal_count": {"type": "single", "photo_observable": true, "values": ["0", "3", "4", "5", "6", "many", "fused_tubular"]}, "symmetry": {"type": "single", "photo_observable": true, "values": ["radial", "bilateral", "irregular"]}, "size": {"type": "single", "photo_observable": true, "values": ["tiny_<5mm", "small_5-15mm", "medium_15-30mm", "large_30-60mm", "very_large_>60mm"]}, "arrangement": {"type": "single", "photo_observable": true, "values": ["solitary", "raceme", "spike", "umbel", "head_composite", "panicle", "cyme", "catkin", "spathe_spadix"]}, "position": {"type": "single", "photo_observable": true, "values": ["terminal", "axillary", "basal", "cauliflorous"]}, "orientation": {"type": "single", "photo_observable": true, "values": ["upright", "horizontal", "drooping"]}, "special_shape": {"type": "single", "photo_observable": true, "values": ["none", "lip_labellum", "spur", "spathe", "butterfly_shape", "bell_tubular", "trumpet"]}, "fragrant": {"type": "single", "photo_observable": false, "values": ["yes", "no"]}}, "fruit": {"visible": {"type": "boolean", "photo_observable": true, "values": [true, false]}, "type": {"type": "single", "photo_observable": true, "values": ["berry", "drupe", "capsule", "pod_legume", "achene", "samara", "nut", "aggregate", "fig", "cone", "spore"]}, "colors": {"type": "array", "photo_observable": true, "values": ["green", "yellow", "orange", "red", "purple", "black", "brown", "white_waxy"]}, "size": {"type": "single", "photo_observable": true, "values": ["tiny_<5mm", "small_5-15mm", "medium_15-30mm", "large_>30mm"]}, "surface": {"type": "single", "photo_observable": true, "values": ["smooth", "hairy", "spiny", "warty", "waxy", "hooked_bristles"]}}, "root": {"visible": {"type": "boolean", "photo_observable": true, "values": [true, false]}, "type": {"type": "single", "photo_observable": false, "values": ["fibrous", "taproot", "tuberous", "rhizome", "bulb", "corm", "aerial_root", "storage_tuber"]}}}''')

DERIVED_ENUMS = json.loads(r'''{"overall": {"growth_form": ["aquatic", "fern", "fungus", "grass", "herb", "palm_like", "shrub", "tree", "vine", "not_visible", "uncertain"], "height_estimate": ["1-2m", "2-5m", "30-100cm", "<30cm", ">5m", "not_visible", "uncertain"], "latex": ["none", "yes_clear", "yes_white", "not_visible", "uncertain", "not_checkable"], "smell": ["aromatic", "fishy", "none", "pungent", "not_visible", "uncertain", "not_checkable"], "habitat": ["coastal", "epiphytic", "forest_floor", "open_field", "roadside", "streamside", "urban", "wetland", "not_visible", "uncertain"], "water_droplet_test": ["beading", "flat", "not_visible", "uncertain", "not_checkable"]}, "leaf": {"leaf_type": ["bipinnate_compound", "pinnate_compound", "simple", "trifoliate", "not_visible", "uncertain"], "shape": ["arrow", "elliptic", "fan", "heart", "lanceolate", "linear", "oval", "rhombic", "round", "not_visible", "uncertain"], "edge": ["crenate", "deeply_lobed", "entire", "lobed", "serrated", "not_visible", "uncertain"], "tip": ["acuminate", "acute", "rounded", "not_visible", "uncertain"], "base": ["cordate", "cuneate", "rounded", "not_visible", "uncertain"], "colors": ["brown", "dark_green", "green", "purple", "red_underside", "variegated", "not_visible", "uncertain"], "color_pattern": ["bicolor", "solid", "spotted", "not_visible", "uncertain"], "surface_top": ["glossy", "hairy", "matte", "rough", "scaly", "velvety", "waxy", "not_visible", "uncertain"], "surface_bottom": ["glossy", "hairy", "matte", "not_visible", "uncertain"], "arrangement": ["alternate", "basal_rosette", "bird_nest_radial", "clustered", "opposite", "whorled", "not_visible", "uncertain"], "size": ["large_15-50cm", "medium_5-15cm", "small_2-5cm", "tiny_<2cm", "very_large_>50cm", "not_visible", "uncertain"], "venation": ["palmate", "parallel", "pinnate", "reticulate", "single_midrib", "not_visible", "uncertain"], "texture": ["fleshy", "leathery", "papery", "not_visible", "uncertain"], "petiole_attach": ["normal", "peltate_shield", "sessile", "not_visible", "uncertain"]}, "stem": {"type": ["climbing", "creeping", "erect", "rhizome", "twining", "not_visible", "uncertain"], "cross_section": ["round", "square", "not_visible", "uncertain"], "surface": ["bark_rough", "bark_smooth", "hairy", "scaly", "smooth", "thorny", "not_visible", "uncertain"], "colors": ["brown", "gray", "green", "purple", "red", "not_visible", "uncertain"], "interior": ["hollow", "pithy", "solid", "not_visible", "uncertain", "not_checkable"], "has_thorns": ["no", "yes", "not_visible", "uncertain"]}, "flower": {"colors": ["brown", "green", "orange", "pink", "purple", "red", "white", "yellow", "not_visible", "uncertain"], "color_pattern": ["center_different", "solid", "not_visible", "uncertain"], "petal_count": ["0", "4", "5", "many", "not_visible", "uncertain"], "symmetry": ["bilateral", "radial", "not_visible", "uncertain"], "size": ["large_30-60mm", "medium_15-30mm", "small_5-15mm", "tiny_<5mm", "very_large_>60mm", "not_visible", "uncertain"], "arrangement": ["cyme", "head_composite", "panicle", "raceme", "solitary", "spathe_spadix", "spike", "umbel", "not_visible", "uncertain"], "position": ["axillary", "terminal", "not_visible", "uncertain"], "orientation": ["drooping", "upright", "not_visible", "uncertain"], "special_shape": ["bell_tubular", "butterfly_shape", "lip_labellum", "none", "spathe", "trumpet", "not_visible", "uncertain"], "fragrant": ["no", "yes", "not_visible", "uncertain", "not_checkable"]}, "fruit": {"type": ["achene", "aggregate", "berry", "capsule", "drupe", "pod_legume", "spore", "not_visible", "uncertain"], "colors": ["brown", "green", "orange", "purple", "red", "white_waxy", "yellow", "not_visible", "uncertain"], "size": ["large_>30mm", "medium_15-30mm", "small_5-15mm", "tiny_<5mm", "not_visible", "uncertain"], "surface": ["smooth", "spiny", "not_visible", "uncertain"]}, "root": {"type": ["aerial_root", "fibrous", "rhizome", "storage_tuber", "taproot", "not_visible", "uncertain", "not_checkable"]}}''')

DERIVED_WEIGHTS = json.loads(r'''{"overall": {"growth_form": {"herb": 1, "tree": 3, "fern": 3, "shrub": 3, "fungus": 3, "palm_like": 6, "vine": 4, "aquatic": 6, "grass": 6}, "height_estimate": {"1-2m": 2, ">5m": 3, "30-100cm": 1, "2-5m": 3, "<30cm": 3}, "latex": {"yes_white": 2, "yes_clear": 6, "none": 1}, "smell": {"none": 1, "aromatic": 2, "fishy": 6, "pungent": 5}, "habitat": {"forest_floor": 2, "coastal": 6, "wetland": 6, "epiphytic": 5, "roadside": 2, "streamside": 3, "open_field": 3, "urban": 3}, "water_droplet_test": {"flat": 1, "beading": 6}}, "leaf": {"leaf_type": {"simple": 1, "pinnate_compound": 3, "bipinnate_compound": 3, "trifoliate": 6}, "shape": {"heart": 3, "lanceolate": 2, "oval": 1, "linear": 5, "arrow": 6, "elliptic": 4, "round": 4, "rhombic": 6, "fan": 6}, "edge": {"entire": 1, "serrated": 2, "lobed": 3, "deeply_lobed": 6, "crenate": 5}, "tip": {"acute": 3, "acuminate": 1, "rounded": 3}, "base": {"cordate": 3, "cuneate": 1, "rounded": 3}, "colors": {"dark_green": 2, "green": 1, "red_underside": 5, "purple": 4, "variegated": 6, "brown": 4}, "color_pattern": {"solid": 1, "bicolor": 3, "spotted": 6}, "surface_top": {"glossy": 2, "velvety": 6, "matte": 1, "rough": 4, "hairy": 3, "scaly": 4, "waxy": 6}, "surface_bottom": {"matte": 1, "hairy": 4, "glossy": 6}, "arrangement": {"clustered": 2, "alternate": 1, "bird_nest_radial": 6, "basal_rosette": 5, "opposite": 4, "whorled": 6}, "size": {"very_large_>50cm": 3, "large_15-50cm": 2, "medium_5-15cm": 1, "tiny_<2cm": 6, "small_2-5cm": 4}, "venation": {"pinnate": 1, "single_midrib": 5, "parallel": 4, "palmate": 3, "reticulate": 3}, "texture": {"leathery": 2, "papery": 1, "fleshy": 3}, "petiole_attach": {"normal": 1, "peltate_shield": 6, "sessile": 6}}, "stem": {"type": {"erect": 1, "rhizome": 5, "twining": 6, "climbing": 6, "creeping": 4}, "cross_section": {"round": 1, "square": 4}, "surface": {"smooth": 1, "bark_smooth": 6, "scaly": 3, "thorny": 5, "hairy": 4, "bark_rough": 5}, "colors": {"green": 1, "purple": 3, "brown": 2, "gray": 4, "red": 5}, "interior": {"pithy": 3, "solid": 1, "hollow": 3}, "has_thorns": {"no": 1, "yes": 4}}, "flower": {"colors": {"yellow": 2, "green": 2, "white": 2, "pink": 3, "brown": 6, "purple": 3, "red": 3, "orange": 5}, "color_pattern": {"solid": 1, "center_different": 6}, "petal_count": {"0": 4, "5": 1, "4": 5, "many": 4}, "symmetry": {"radial": 1, "bilateral": 6}, "size": {"large_30-60mm": 5, "medium_15-30mm": 1, "tiny_<5mm": 4, "small_5-15mm": 5, "very_large_>60mm": 5}, "arrangement": {"spathe_spadix": 4, "cyme": 4, "spike": 3, "solitary": 3, "panicle": 4, "umbel": 4, "head_composite": 3, "raceme": 4}, "position": {"terminal": 1, "axillary": 6}, "orientation": {"upright": 1, "drooping": 6}, "special_shape": {"spathe": 4, "none": 1, "trumpet": 4, "butterfly_shape": 5, "bell_tubular": 5, "lip_labellum": 6}, "fragrant": {"no": 1, "yes": 3}}, "fruit": {"type": {"berry": 3, "drupe": 3, "spore": 2, "capsule": 1, "achene": 3, "pod_legume": 6, "aggregate": 5}, "colors": {"red": 3, "green": 1, "brown": 2, "yellow": 4, "orange": 4, "purple": 3, "white_waxy": 5}, "size": {"small_5-15mm": 1, "large_>30mm": 4, "tiny_<5mm": 2, "medium_15-30mm": 6}, "surface": {"smooth": 1, "spiny": 5}}, "root": {"type": {"rhizome": 3, "taproot": 5, "storage_tuber": 4, "fibrous": 1, "aerial_root": 6}}}''')

CONFUSION_DB = json.loads(r'''{"_description": "混淆物種對關係表 — 不儲存物種特徵，只記錄哪些物種容易混淆及區分方法", "confusion_pairs": [{"id": "taro_vs_alocasia", "species_a": "alocasia_odora", "species_b": "colocasia_esculenta", "lethality": "high", "key_differences": [{"feature": "leaf.surface_top", "species_a_value": "glossy", "species_b_value": "velvety", "test": "用手指摸葉面：光滑發亮像打蠟 = 姑婆芋（有毒），有天鵝絨般絨毛 = 芋頭（可食）"}, {"feature": "overall.water_droplet_test", "species_a_value": "flat", "species_b_value": "beading", "test": "在葉面滴水：水珠攤平 = 姑婆芋（有毒），水珠成珠滾動 = 芋頭（可食）"}, {"feature": "leaf.petiole_attach", "species_a_value": "normal", "species_b_value": "peltate_shield", "test": "觀察葉柄接合處：接在葉片邊緣 = 姑婆芋，接在葉片內側（盾狀）= 芋頭"}]}, {"id": "parasol_vs_green_spored", "species_a": "chlorophyllum_molybdites", "species_b": "macrolepiota_procera", "lethality": "medium", "key_differences": [{"feature": "spore_print_color", "species_a_value": "gray_green", "species_b_value": "white", "test": "取孢子印：灰綠色 = 綠褶菇（有毒），白色 = 高大環柄菇（可食）"}, {"feature": "gill_color_mature", "species_a_value": "green", "species_b_value": "white", "test": "觀察成熟菌褶：成熟後變灰綠色 = 綠褶菇（有毒），始終白色 = 高大環柄菇（可食）"}]}, {"id": "birds_nest_fern_vs_bracken", "species_a": "pteridium_aquilinum", "species_b": "asplenium_nidus", "lethality": "low", "key_differences": [{"feature": "leaf.leaf_type", "species_a_value": "bipinnate_compound", "species_b_value": "simple", "test": "觀察葉片結構：細碎羽狀複葉 = 蕨（有害），大型單葉 = 山蘇（可食）"}, {"feature": "leaf.arrangement", "species_a_value": "alternate", "species_b_value": "bird_nest_radial", "test": "觀察生長排列：從地面直立散開 = 蕨，從中心輻射呈鳥巢狀 = 山蘇"}]}, {"id": "termite_mushroom_vs_white_toxic", "species_a": "amanita_phalloides", "species_b": "termitomyces_sp", "lethality": "critical", "key_differences": [{"feature": "volva_presence", "species_a_value": "present", "species_b_value": "absent", "test": "檢查菌柄基部：有杯狀菌托（蛋殼狀包覆）= 毒鵝膏（致命！），無菌托 = 雞肉絲菇"}, {"feature": "habitat_association", "species_a_value": "tree_root", "species_b_value": "termite_mound", "test": "觀察生長位置：生長在樹根附近 = 可能是毒鵝膏，生長在白蟻巢上 = 雞肉絲菇"}]}, {"id": "bidens_vs_parthenium", "species_a": "parthenium_hysterophorus", "species_b": "bidens_pilosa_var_radiata", "lethality": "low", "key_differences": [{"feature": "flower.size", "species_a_value": "tiny_<5mm", "species_b_value": "medium_15-30mm", "test": "觀察花朵大小：花極小且無明顯白色舌狀花 = 銀膠菊（有害），花較大有白色舌狀花 = 大花咸豐草（可食）"}, {"feature": "overall.smell", "species_a_value": "pungent", "species_b_value": "none", "test": "搓揉莖葉聞氣味：有刺激性異味 = 銀膠菊（有害），無特殊氣味 = 大花咸豐草"}]}, {"id": "pokeweed_vs_amaranth", "species_a": "phytolacca_americana", "species_b": "amaranthus_viridis", "lethality": "high", "key_differences": [{"feature": "stem.colors", "species_a_value": ["purple"], "species_b_value": ["green"], "test": "觀察莖部：粗壯且帶紫紅色 = 美洲商陸（有毒），較細且綠色 = 野莧菜（可食）"}, {"feature": "fruit.type", "species_a_value": "berry", "species_b_value": "achene", "test": "觀察果實：紫黑色漿果串 = 美洲商陸（有毒），乾燥小粒 = 野莧菜（可食）"}]}, {"id": "calla_vs_taro", "species_a": "zantedeschia_aethiopica", "species_b": "colocasia_esculenta", "lethality": "medium", "key_differences": [{"feature": "flower.special_shape", "species_a_value": "spathe", "species_b_value": "spathe", "test": "觀察佛焰苞：大型純白色佛焰苞 = 海芋（有毒），較不顯眼的黃綠色苞 = 芋頭"}, {"feature": "leaf.shape", "species_a_value": "arrow", "species_b_value": "heart", "test": "觀察葉形：箭形較尖 = 海芋，心形較圓 = 芋頭"}]}, {"id": "lantana_vs_mulberry", "species_a": "lantana_camara", "species_b": "morus_australis", "lethality": "medium", "key_differences": [{"feature": "fruit.type", "species_a_value": "berry", "species_b_value": "aggregate", "test": "觀察果實形態：小球狀聚生 = 馬纓丹（有毒），長圓形聚合果 = 小葉桑（可食）"}, {"feature": "overall.smell", "species_a_value": "pungent", "species_b_value": "none", "test": "搓葉片：有刺鼻臭味 = 馬纓丹（有毒），無特殊氣味 = 小葉桑"}]}, {"id": "foxglove_vs_plantain", "species_a": "digitalis_purpurea", "species_b": "plantago_major", "lethality": "critical", "key_differences": [{"feature": "leaf.surface_top", "species_a_value": "hairy", "species_b_value": "matte", "test": "觸摸葉面：有明顯絨毛 = 毛地黃（致命！），光滑或微粗 = 車前草（可食）"}, {"feature": "leaf.venation", "species_a_value": "pinnate", "species_b_value": "parallel", "test": "觀察葉脈：羽狀脈 = 毛地黃（致命！），明顯平行脈 = 車前草（可食）"}]}, {"id": "vegetable_fern_vs_bracken", "species_a": "pteridium_aquilinum", "species_b": "diplazium_esculentum", "lethality": "low", "key_differences": [{"feature": "leaf.shape", "species_a_value": "oval", "species_b_value": "lanceolate", "test": "觀察整體葉形：三角形寬展 = 蕨（有害），披針形較窄 = 過貓（可食）"}, {"feature": "overall.habitat", "species_a_value": "open_field", "species_b_value": "streamside", "test": "觀察生長環境：開闊山坡 = 蕨（有害），溪邊潮濕處 = 過貓（可食）"}]}, {"id": "prickly_ash_vs_chinaberry", "species_a": "melia_azedarach", "species_b": "zanthoxylum_ailanthoides", "lethality": "medium", "key_differences": [{"feature": "overall.smell", "species_a_value": "none", "species_b_value": "spicy", "test": "搓揉葉片聞味：無花椒味 = 苦楝（有毒），有濃烈花椒辛香味 = 食茱萸（可食）"}, {"feature": "stem.has_thorns", "species_a_value": "no", "species_b_value": "yes", "test": "觀察枝幹：無刺 = 苦楝（有毒），有明顯尖刺 = 食茱萸（可食）"}]}, {"id": "cudweed_vs_parthenium", "species_a": "parthenium_hysterophorus", "species_b": "pseudognaphalium_affine", "lethality": "low", "key_differences": [{"feature": "leaf.surface_top", "species_a_value": "hairy", "species_b_value": "hairy", "test": "觀察絨毛：稀疏粗糙毛 = 銀膠菊（有害），全株密被白色綿毛 = 鼠麴草（可食）"}, {"feature": "flower.colors", "species_a_value": ["white"], "species_b_value": ["yellow"], "test": "觀察花色：白色小花 = 銀膠菊（有害），黃色小花密集 = 鼠麴草（可食）"}]}]}''')

PROMPT_TEMPLATE = r'''你是植物特徵辨識器。請仔細觀察照片中的植物，
只根據你在照片中「實際看到」的視覺特徵填寫 JSON。

【核心規則】
1. 只根據照片的視覺資訊填寫，不要根據你對植物物種的知識填寫。
   即使你認出這可能是某種植物，也必須只描述照片中看到的特徵。
2. 看不到的部位填 "not_visible"，不確定的屬性填 "uncertain"。
3. 嚴禁猜測看不到的部位。
4. 輸出純 JSON，不要任何解釋文字。
5. 如果某個部位（花/果實/根）完全不在照片中，該部位只填 "visible": false，
   不要填寫該部位的其他任何屬性。範例："flower": {"visible": false}

【照片無法判斷的屬性 — 一律填 "not_checkable"】
以下屬性不可能從照片判斷，不論你多有信心，一律填 "not_checkable"：
- latex（汁液 — 需折斷莖部才能觀察）
- smell（氣味 — 需要實際聞到）
- water_droplet_test（水珠測試 — 需要現場實測）
- stem.interior（莖內部 — 需要切開才能觀察）
- flower.fragrant（花香 — 需要實際聞到）
- root.type（根型 — 根部通常在地下）

【範例 — 一張路邊草本植物只拍到葉和莖的照片】
{
  "growth_form": "herb",
  "height_estimate": "30-100cm",
  "latex": "not_checkable",
  "smell": "not_checkable",
  "habitat": "roadside",
  "water_droplet_test": "not_checkable",
  "leaf": {
    "leaf_type": "simple",
    "shape": "oval",
    "edge": "serrated",
    "tip": "acute",
    "base": "rounded",
    "colors": ["green"],
    "color_pattern": "solid",
    "surface_top": "matte",
    "surface_bottom": "hairy",
    "arrangement": "alternate",
    "size": "medium_5-15cm",
    "venation": "pinnate",
    "texture": "papery",
    "petiole_attach": "normal"
  },
  "stem": {
    "type": "erect",
    "cross_section": "round",
    "surface": "hairy",
    "colors": ["green"],
    "interior": "not_checkable",
    "has_thorns": "no"
  },
  "flower": {
    "visible": false
  },
  "fruit": {
    "visible": false
  },
  "root": {
    "visible": false,
    "type": "not_checkable"
  }
}

【請填寫以下 JSON — 每個值必須從括號內的選項中選擇】
{
  "growth_form": 從 ["aquatic(水生)", "fern(蕨類)", "fungus(菇菌)", "grass(禾草)", "herb(草本)", "palm_like(棕櫚狀)", "shrub(灌木)", "tree(喬木)", "vine(藤本)", "not_visible", "uncertain"] 選,
  "height_estimate": 從 ["1-2m(1-2公尺)", "2-5m(2-5公尺)", "30-100cm(30-100cm)", "<30cm(矮於30cm)", ">5m(高於5公尺)", "not_visible", "uncertain"] 選,
  "latex": 從 ["none", "yes_clear", "yes_white", "not_visible", "uncertain", "not_checkable"] 選,
  "smell": 從 ["aromatic", "fishy", "none", "pungent", "not_visible", "uncertain", "not_checkable"] 選,
  "habitat": 從 ["coastal(海岸)", "epiphytic(附生)", "forest_floor(林下地面)", "open_field(空曠地)", "roadside(路邊)", "streamside(溪邊)", "urban(都市)", "wetland(濕地)", "not_visible", "uncertain"] 選,
  "water_droplet_test": 從 ["beading", "flat", "not_visible", "uncertain", "not_checkable"] 選,
  "leaf": {
    "leaf_type": 從 ["bipinnate_compound(二回羽狀複葉)", "pinnate_compound(羽狀複葉)", "simple(單葉)", "trifoliate(三出複葉)", "not_visible", "uncertain"] 選,
    "shape": 從 ["arrow(箭形)", "elliptic(橢圓形)", "fan(扇形)", "heart(心形)", "lanceolate(披針形)", "linear(線形)", "oval(卵形)", "rhombic(菱形)", "round(圓形)", "not_visible", "uncertain"] 選,
    "edge": 從 ["crenate(圓齒)", "deeply_lobed(深裂)", "entire(全緣(光滑))", "lobed(淺裂)", "serrated(鋸齒)", "not_visible", "uncertain"] 選,
    "tip": 從 ["acuminate(漸尖(尖端拉長如尾巴))", "acute(銳尖(尖端直接收窄))", "rounded(圓鈍)", "not_visible", "uncertain"] 選,
    "base": 從 ["cordate(心形基部(有凹口))", "cuneate(楔形(V字收窄))", "rounded(圓形)", "not_visible", "uncertain"] 選,
    "colors": [從 ["brown(棕)", "dark_green(深綠)", "green(綠)", "purple(紫)", "red_underside(背紅)", "variegated(斑葉)", "not_visible", "uncertain"] 中選一到多個],
    "color_pattern": 從 ["bicolor(雙色)", "solid(純色)", "spotted(斑點)", "not_visible", "uncertain"] 選,
    "surface_top": 從 ["glossy(光滑反光(像打蠟))", "hairy(有毛)", "matte(霧面(不反光))", "rough(粗糙)", "scaly(鱗片狀)", "velvety(絨面(如天鵝絨))", "waxy(蠟質層)", "not_visible", "uncertain"] 選,
    "surface_bottom": 從 ["glossy(光滑反光)", "hairy(有毛)", "matte(霧面)", "not_visible", "uncertain"] 選,
    "arrangement": 從 ["alternate(互生)", "basal_rosette(基生蓮座)", "bird_nest_radial(鳥巢放射狀)", "clustered(叢生)", "opposite(對生)", "whorled(輪生)", "not_visible", "uncertain"] 選,
    "size": 從 ["large_15-50cm(大(15-50cm))", "medium_5-15cm(中(5-15cm))", "small_2-5cm(小(2-5cm))", "tiny_<2cm(極小(<2cm))", "very_large_>50cm(很大(>50cm))", "not_visible", "uncertain"] 選,
    "venation": 從 ["palmate(掌狀脈)", "parallel(平行脈)", "pinnate(羽狀脈)", "reticulate(網狀脈)", "single_midrib(單中脈)", "not_visible", "uncertain"] 選,
    "texture": 從 ["fleshy(肉質)", "leathery(革質(厚硬))", "papery(紙質)", "not_visible", "uncertain"] 選,
    "petiole_attach": 從 ["normal(接在葉片邊緣)", "peltate_shield(盾狀著生(接在葉片內側))", "sessile(無柄直接著生)", "not_visible", "uncertain"] 選
  },
  "stem": {
    "type": 從 ["climbing(攀緣)", "creeping(匍匐)", "erect(直立)", "rhizome(根莖)", "twining(纏繞)", "not_visible", "uncertain"] 選,
    "cross_section": 從 ["round", "square", "not_visible", "uncertain"] 選,
    "surface": 從 ["bark_rough(粗糙樹皮)", "bark_smooth(光滑樹皮)", "hairy(有毛)", "scaly(鱗片)", "smooth(光滑)", "thorny(有刺)", "not_visible", "uncertain"] 選,
    "colors": [從 ["brown(棕)", "gray(灰)", "green(綠)", "purple(紫)", "red(紅)", "not_visible", "uncertain"] 中選一到多個],
    "interior": 從 ["hollow", "pithy", "solid", "not_visible", "uncertain", "not_checkable"] 選,
    "has_thorns": 從 ["no", "yes", "not_visible", "uncertain"] 選
  },
  "flower": {
    "colors": [從 ["brown(棕)", "green(綠)", "orange(橙)", "pink(粉紅)", "purple(紫)", "red(紅)", "white(白)", "yellow(黃)", "not_visible", "uncertain"] 中選一到多個],
    "color_pattern": 從 ["center_different(中心異色)", "solid(純色)", "not_visible", "uncertain"] 選,
    "petal_count": 從 ["0", "4", "5", "many", "not_visible", "uncertain"] 選,
    "symmetry": 從 ["bilateral", "radial", "not_visible", "uncertain"] 選,
    "size": 從 ["large_30-60mm", "medium_15-30mm", "small_5-15mm", "tiny_<5mm", "very_large_>60mm", "not_visible", "uncertain"] 選,
    "arrangement": 從 ["cyme(聚繖)", "head_composite(頭狀(菊科))", "panicle(圓錐)", "raceme(總狀)", "solitary(單生)", "spathe_spadix(佛焰苞+肉穗(天南星科))", "spike(穗狀)", "umbel(繖形)", "not_visible", "uncertain"] 選,
    "position": 從 ["axillary", "terminal", "not_visible", "uncertain"] 選,
    "orientation": 從 ["drooping", "upright", "not_visible", "uncertain"] 選,
    "special_shape": 從 ["bell_tubular(鐘形/筒形)", "butterfly_shape(蝶形(豆科))", "lip_labellum(唇瓣(蘭科))", "none(無)", "spathe(佛焰苞)", "trumpet(喇叭形)", "not_visible", "uncertain"] 選,
    "fragrant": 從 ["no", "yes", "not_visible", "uncertain", "not_checkable"] 選
  },
  "fruit": {
    "type": 從 ["achene(瘦果)", "aggregate(聚合果)", "berry(漿果)", "capsule(蒴果)", "drupe(核果)", "pod_legume(莢果)", "spore(孢子)", "not_visible", "uncertain"] 選,
    "colors": [從 ["brown(棕)", "green(綠)", "orange(橙)", "purple(紫)", "red(紅)", "white_waxy(白蠟)", "yellow(黃)", "not_visible", "uncertain"] 中選一到多個],
    "size": 從 ["large_>30mm", "medium_15-30mm", "small_5-15mm", "tiny_<5mm", "not_visible", "uncertain"] 選,
    "surface": 從 ["smooth(光滑)", "spiny(有刺)", "not_visible", "uncertain"] 選
  },
  "root": {
    "type": 從 ["aerial_root", "fibrous", "rhizome", "storage_tuber", "taproot", "not_visible", "uncertain", "not_checkable"] 選
  }
}'''

print(f"✅ Schema sections: {list(k for k in FEATURE_SCHEMA if not k.startswith('_'))}")
print(f"✅ Confusion pairs: {len(CONFUSION_DB.get('confusion_pairs', []))}")
print(f"✅ Prompt template: {len(PROMPT_TEMPLATE)} chars")

In [ ]:
# Cell 9: 植物知識庫 (50 species)

PLANTS_DB = json.loads(r'''[{"id":"alocasia_odora","scientific_name":"Alocasia odora","common_names":{"zh-TW":"姑婆芋","en":"Giant Elephant Ear"},"category":"dangerous","danger_level":"high","features":{"overall":{"growth_form":{"value":"herb","weight":1},"height_estimate":{"value":"1-2m","weight":2},"latex":{"value":"yes_white","weight":2},"smell":{"value":"none","weight":1},"habitat":{"value":"forest_floor","weight":2},"water_droplet_test":{"value":"flat","weight":1}},"leaf":{"leaf_type":{"value":"simple","weight":1},"shape":{"value":"heart","weight":3},"edge":{"value":"entire","weight":1},"tip":{"value":"acute","weight":3},"base":{"value":"cordate","weight":3},"colors":{"value":["dark_green"],"weight":2},"color_pattern":{"value":"solid","weight":1},"surface_top":{"value":"glossy","weight":2},"surface_bottom":{"value":"matte","weight":1},"arrangement":{"value":"clustered","weight":2},"size":{"value":"very_large_>50cm","weight":3},"venation":{"value":"pinnate","weight":1},"texture":{"value":"leathery","weight":2},"petiole_attach":{"value":"normal","weight":1}},"stem":{"type":{"value":"erect","weight":1},"cross_section":{"value":"round","weight":1},"surface":{"value":"smooth","weight":1},"colors":{"value":["green","purple"],"weight":3},"interior":{"value":"pithy","weight":3},"has_thorns":{"value":"no","weight":1}},"flower":{"colors":{"value":["yellow","green"],"weight":2},"color_pattern":{"value":"solid","weight":1},"petal_count":{"value":"0","weight":4},"symmetry":{"value":"radial","weight":1},"size":{"value":"large_30-60mm","weight":5},"arrangement":{"value":"spathe_spadix","weight":4},"position":{"value":"terminal","weight":1},"orientation":{"value":"upright","weight":1},"special_shape":{"value":"spathe","weight":4},"fragrant":{"value":"no","weight":1}},"fruit":{"type":{"value":"berry","weight":3},"colors":{"value":["red"],"weight":3},"size":{"value":"small_5-15mm","weight":1},"surface":{"value":"smooth","weight":1}},"root":{"type":{"value":"rhizome","weight":3}}},"human_readable":{"toxicity":"全株有毒，汁液含草酸鈣針晶，誤食致口腔灼傷、腹瀉、喉嚨腫脹","symptoms":["口腔灼傷","舌頭麻痺","腹瀉","喉嚨腫脹","嘔吐"],"first_aid":"立即用大量清水漱口，勿催吐，儘速就醫。皮膚接觸汁液時以大量清水沖洗。","affected_parts":["全株","汁液"],"diagnostic_features":["葉面光滑發亮，像打了蠟","水珠在葉面攤平不成珠狀","葉尖朝上或水平指向","葉柄接在葉片邊緣（非盾狀著生）","折斷莖葉有白色乳汁，碰到皮膚會灼傷"]},"distribution":{"altitude_range":[0,1500],"climate_zones":["subtropical","tropical"]},"total_weight":80,"photo_observable_weight":69},{"id":"cerbera_manghas","scientific_name":"Cerbera manghas","common_names":{"zh-TW":"海檬果","en":"Sea Mango"},"category":"dangerous","danger_level":"critical","features":{"overall":{"growth_form":{"value":"tree","weight":3},"height_estimate":{"value":">5m","weight":3},"latex":{"value":"yes_white","weight":2},"smell":{"value":"aromatic","weight":2},"habitat":{"value":"coastal","weight":6},"water_droplet_test":{"value":"flat","weight":1}},"leaf":{"leaf_type":{"value":"simple","weight":1},"shape":{"value":"lanceolate","weight":2},"edge":{"value":"entire","weight":1},"tip":{"value":"acuminate","weight":1},"base":{"value":"cuneate","weight":1},"colors":{"value":["dark_green"],"weight":2},"color_pattern":{"value":"solid","weight":1},"surface_top":{"value":"glossy","weight":2},"surface_bottom":{"value":"matte","weight":1},"arrangement":{"value":"alternate","weight":1},"size":{"value":"large_15-50cm","weight":2},"venation":{"value":"pinnate","weight":1},"texture":{"value":"leathery","weight":2},"petiole_attach":{"value":"normal","weight":1}},"stem":{"type":{"value":"erect","weight":1},"cross_section":{"value":"round","weight":1},"surface":{"value":"bark_smooth","weight":6},"colors":{"value":["brown","gray"],"weight":4},"interior":{"value":"solid","weight":1},"has_thorns":{"value":"no","weight":1}},"flower":{"colors":{"value":["white","pink"],"weight":3},"color_pattern":{"value":"center_different","weight":6},"petal_count":{"value":"5","weight":1},"symmetry":{"value":"radial","weight":1},"size":{"value":"large_30-60mm","weight":5},"arrangement":{"value":"cyme","weight":4},"position":{"value":"terminal","weight":1},"orientation":{"value":"upright","weight":1},"special_shape":{"value":"none","weight":1},"fragrant":{"value":"yes","weight":3}},"fruit":{"type":{"value":"drupe","weight":3},"colors":{"value":["green","red"],"weight":3},"size":{"value":"large_>30mm","weight":4},"surface":{"value":"smooth","weight":1}},"root":{"type":{"value":"taproot","weight":5}}},"human_readable":{"toxicity":"全株有毒，種子劇毒，含海檬果毒素（cerberin），誤食可致心臟麻痺死亡","symptoms":["嘔吐","腹痛","心律不整","心臟麻痺","可致死"],"first_aid":"立即催吐（若意識清醒），儘速送醫，告知醫師疑似海檬果中毒。此為心臟毒性，需緊急處理。","affected_parts":["全株","種子（劇毒）","乳汁"],"diagnostic_features":["倒披針形革質葉，有光澤","白色五瓣花，中心粉紅色，有香味","卵圓形核果，直徑 5-7cm，青綠轉暗紅","折斷枝條有白色乳汁","常見於海岸、公園景觀樹"]},"distribution":{"altitude_range":[0,100],"climate_zones":["subtropical","tropical"]},"total_weight":92,"photo_observable_weight":78},{"id":"colocasia_esculenta","scientific_name":"Colocasia esculenta","common_names":{"zh-TW":"芋頭","en":"Taro"},"category":"edible","features":{"overall":{"growth_form":{"value":"herb","weight":1},"height_estimate":{"value":"1-2m","weight":2},"latex":{"value":"yes_clear","weight":6},"smell":{"value":"none","weight":1},"habitat":{"value":"wetland","weight":6},"water_droplet_test":{"value":"beading","weight":6}},"leaf":{"leaf_type":{"value":"simple","weight":1},"shape":{"value":"heart","weight":3},"edge":{"value":"entire","weight":1},"tip":{"value":"acuminate","weight":1},"base":{"value":"cordate","weight":3},"colors":{"value":["green"],"weight":1},"color_pattern":{"value":"solid","weight":1},"surface_top":{"value":"velvety","weight":6},"surface_bottom":{"value":"matte","weight":1},"arrangement":{"value":"clustered","weight":2},"size":{"value":"large_15-50cm","weight":2},"venation":{"value":"pinnate","weight":1},"texture":{"value":"leathery","weight":2},"petiole_attach":{"value":"peltate_shield","weight":6}},"stem":{"type":{"value":"erect","weight":1},"cross_section":{"value":"round","weight":1},"surface":{"value":"smooth","weight":1},"colors":{"value":["green"],"weight":1},"interior":{"value":"pithy","weight":3},"has_thorns":{"value":"no","weight":1}},"flower":{"colors":{"value":["yellow","green"],"weight":2},"color_pattern":{"value":"solid","weight":1},"petal_count":{"value":"0","weight":4},"symmetry":{"value":"radial","weight":1},"size":{"value":"medium_15-30mm","weight":1},"arrangement":{"value":"spathe_spadix","weight":4},"position":{"value":"axillary","weight":6},"orientation":{"value":"upright","weight":1},"special_shape":{"value":"spathe","weight":4},"fragrant":{"value":"no","weight":1}},"fruit":{"type":{"value":"berry","weight":3},"colors":{"value":["green"],"weight":1},"size":{"value":"small_5-15mm","weight":1},"surface":{"value":"smooth","weight":1}},"root":{"type":{"value":"storage_tuber","weight":4}}},"usage":{"type":["edible_root"],"edible_parts":["塊莖（須煮熟）","嫩莖（須煮熟）"],"preparation":"必須完全煮熟！生芋頭含草酸鈣。蒸煮 20-30 分鐘至熟透。","season":"全年可採，秋冬塊莖最大","warnings":"極易與有毒的姑婆芋混淆！必須做水珠測試確認。生食有毒。"},"human_readable":{"description_zh":"多年生草本，有大型心形葉片。葉面有天鵝絨般的微絨毛質感，水珠在葉面呈圓珠狀滾動。地下有圓形塊莖可食用，但必須煮熟。","diagnostic_features":["葉面有天鵝絨般微絨毛，不光滑","水珠在葉面呈圓珠狀滾動（荷葉效應）","葉尖通常朝下垂","葉柄接在葉片邊緣內側（盾狀著生）","地下有圓形塊莖"]},"distribution":{"altitude_range":[0,1000],"climate_zones":["subtropical","tropical"]},"total_weight":96,"photo_observable_weight":75},{"id":"asplenium_nidus","scientific_name":"Asplenium nidus","common_names":{"zh-TW":"山蘇（鳥巢蕨）","en":"Bird's Nest Fern"},"category":"edible","features":{"overall":{"growth_form":{"value":"fern","weight":3},"height_estimate":{"value":"30-100cm","weight":1},"latex":{"value":"none","weight":1},"smell":{"value":"none","weight":1},"habitat":{"value":"epiphytic","weight":5},"water_droplet_test":{"value":"flat","weight":1}},"leaf":{"leaf_type":{"value":"simple","weight":1},"shape":{"value":"lanceolate","weight":2},"edge":{"value":"entire","weight":1},"tip":{"value":"acuminate","weight":1},"base":{"value":"rounded","weight":3},"colors":{"value":["green"],"weight":1},"color_pattern":{"value":"solid","weight":1},"surface_top":{"value":"glossy","weight":2},"surface_bottom":{"value":"matte","weight":1},"arrangement":{"value":"bird_nest_radial","weight":6},"size":{"value":"very_large_>50cm","weight":3},"venation":{"value":"single_midrib","weight":5},"texture":{"value":"leathery","weight":2},"petiole_attach":{"value":"sessile","weight":6}},"stem":{"type":{"value":"rhizome","weight":5},"cross_section":{"value":"round","weight":1},"surface":{"value":"scaly","weight":3},"colors":{"value":["brown"],"weight":2},"interior":{"value":"pithy","weight":3},"has_thorns":{"value":"no","weight":1}},"flower":null,"fruit":{"type":{"value":"spore","weight":2},"colors":{"value":["brown"],"weight":2},"size":{"value":"tiny_<5mm","weight":2},"surface":{"value":"smooth","weight":1}},"root":{"type":{"value":"fibrous","weight":1}}},"usage":{"type":["edible_leaf"],"edible_parts":["嫩捲葉（中心尚未展開的幼葉）"],"preparation":"川燙後即可食用，也可快炒。川燙 1-2 分鐘。可加薑絲、蒜末調味。","season":"全年可採，春夏嫩芽最多","warnings":"確認是山蘇而非其他蕨類。只食用嫩捲葉。每株最多採 1-2 支嫩芽。"},"human_readable":{"description_zh":"附生於大樹幹或岩壁的蕨類，大型單葉從中心向外輻射展開形成鳥巢狀。葉片長披針形，光滑革質，有明顯的黑褐色中肋。","diagnostic_features":["葉片從中心向外輻射展開，形成鳥巢狀排列","大型單葉，長披針形，光滑革質","明顯的黑褐色中肋","附生在樹幹上或岩石上","葉背有線狀孢子囊群"]},"distribution":{"altitude_range":[200,1500],"climate_zones":["subtropical","tropical"]},"total_weight":70,"photo_observable_weight":63},{"id":"plantago_major","scientific_name":"Plantago major","common_names":{"zh-TW":"車前草","en":"Broadleaf Plantain"},"category":"medicinal","features":{"overall":{"growth_form":{"value":"herb","weight":1},"height_estimate":{"value":"30-100cm","weight":1},"latex":{"value":"none","weight":1},"smell":{"value":"none","weight":1},"habitat":{"value":"roadside","weight":2},"water_droplet_test":{"value":"flat","weight":1}},"leaf":{"leaf_type":{"value":"simple","weight":1},"shape":{"value":"oval","weight":1},"edge":{"value":"entire","weight":1},"tip":{"value":"rounded","weight":3},"base":{"value":"rounded","weight":3},"colors":{"value":["green"],"weight":1},"color_pattern":{"value":"solid","weight":1},"surface_top":{"value":"matte","weight":1},"surface_bottom":{"value":"matte","weight":1},"arrangement":{"value":"basal_rosette","weight":5},"size":{"value":"large_15-50cm","weight":2},"venation":{"value":"parallel","weight":4},"texture":{"value":"papery","weight":1},"petiole_attach":{"value":"normal","weight":1}},"stem":{"type":{"value":"erect","weight":1},"cross_section":{"value":"round","weight":1},"surface":{"value":"smooth","weight":1},"colors":{"value":["green"],"weight":1},"interior":{"value":"solid","weight":1},"has_thorns":{"value":"no","weight":1}},"flower":{"colors":{"value":["green","brown"],"weight":6},"color_pattern":{"value":"solid","weight":1},"petal_count":{"value":"4","weight":5},"symmetry":{"value":"radial","weight":1},"size":{"value":"tiny_<5mm","weight":4},"arrangement":{"value":"spike","weight":3},"position":{"value":"terminal","weight":1},"orientation":{"value":"upright","weight":1},"special_shape":{"value":"none","weight":1},"fragrant":{"value":"no","weight":1}},"fruit":{"type":{"value":"capsule","weight":1},"colors":{"value":["brown"],"weight":2},"size":{"value":"tiny_<5mm","weight":2},"surface":{"value":"smooth","weight":1}},"root":{"type":{"value":"fibrous","weight":1}}},"usage":{"type":["wound_care","anti_inflammatory","edible_leaf"],"edible_parts":["嫩葉"],"medicinal_effects":["消炎","抗菌","止血","利尿"],"preparation":"搗碎鮮葉敷於傷口可止血消炎。嫩葉可川燙後涼拌食用。","season":"春夏最佳，全年可採","warnings":"避免採集路邊受農藥或汽車廢氣污染的植株。"},"human_readable":{"description_zh":"極常見的低矮草本，所有葉從基部長出呈蓮座狀貼地展開。橢圓形至卵形葉片，葉面有明顯的 3-7 條平行脈。穗狀花序直立。","diagnostic_features":["所有葉從基部長出，蓮座狀貼地展開","葉面有明顯的 3-7 條平行脈（關鍵特徵）","全緣或微波狀葉緣","穗狀花序直立，小花密集","路邊、荒地極常見"]},"distribution":{"altitude_range":[0,2500],"climate_zones":["subtropical","temperate","tropical"]},"total_weight":70,"photo_observable_weight":64},{"id":"houttuynia_cordata","scientific_name":"Houttuynia cordata","common_names":{"zh-TW":"魚腥草（折耳根）","en":"Fish Mint"},"category":"medicinal","features":{"overall":{"growth_form":{"value":"herb","weight":1},"height_estimate":{"value":"30-100cm","weight":1},"latex":{"value":"none","weight":1},"smell":{"value":"fishy","weight":6},"habitat":{"value":"streamside","weight":3},"water_droplet_test":{"value":"flat","weight":1}},"leaf":{"leaf_type":{"value":"simple","weight":1},"shape":{"value":"heart","weight":3},"edge":{"value":"entire","weight":1},"tip":{"value":"acuminate","weight":1},"base":{"value":"cordate","weight":3},"colors":{"value":["green","red_underside"],"weight":5},"color_pattern":{"value":"bicolor","weight":3},"surface_top":{"value":"matte","weight":1},"surface_bottom":{"value":"matte","weight":1},"arrangement":{"value":"alternate","weight":1},"size":{"value":"medium_5-15cm","weight":1},"venation":{"value":"palmate","weight":3},"texture":{"value":"papery","weight":1},"petiole_attach":{"value":"normal","weight":1}},"stem":{"type":{"value":"erect","weight":1},"cross_section":{"value":"round","weight":1},"surface":{"value":"smooth","weight":1},"colors":{"value":["red","purple"],"weight":5},"interior":{"value":"hollow","weight":3},"has_thorns":{"value":"no","weight":1}},"flower":{"colors":{"value":["white"],"weight":2},"color_pattern":{"value":"solid","weight":1},"petal_count":{"value":"4","weight":5},"symmetry":{"value":"radial","weight":1},"size":{"value":"small_5-15mm","weight":5},"arrangement":{"value":"spike","weight":3},"position":{"value":"terminal","weight":1},"orientation":{"value":"upright","weight":1},"special_shape":{"value":"none","weight":1},"fragrant":{"value":"no","weight":1}},"fruit":{"type":{"value":"capsule","weight":1},"colors":{"value":["brown"],"weight":2},"size":{"value":"tiny_<5mm","weight":2},"surface":{"value":"smooth","weight":1}},"root":{"type":{"value":"rhizome","weight":3}}},"usage":{"type":["wound_care","anti_inflammatory","edible_leaf"],"edible_parts":["嫩莖葉","地下匍匐莖"],"medicinal_effects":["清熱解毒","抗菌","消炎","利尿"],"preparation":"鮮葉搗汁外敷或水煎內服。涼拌或川燙食用。","season":"全年可採，春夏最嫩","warnings":"味道強烈但安全無毒。孕婦大量食用應諮詢醫師。"},"human_readable":{"description_zh":"多年生草本，高 15-40cm。全株搓揉具強烈魚腥味（關鍵辨識特徵）。心形葉互生，葉背常帶紫紅。莖紅紫色。穗狀花序頂生，下方有 4 枚白色花瓣狀苞片呈十字展開。","diagnostic_features":["全株搓揉有強烈魚腥味（最關鍵特徵）","心形葉片，葉背常帶紫紅色","莖呈紅紫色，有縱向條紋","穗狀花序下方有 4 枚白色十字狀苞片","常成片蔓生於溪邊潮濕處"]},"distribution":{"altitude_range":[0,2000],"climate_zones":["subtropical","temperate"]},"total_weight":81,"photo_observable_weight":66},{"id":"datura_stramonium","scientific_name":"Datura stramonium","common_names":{"zh-TW":"曼陀羅","en":"Jimsonweed"},"category":"dangerous","danger_level":"critical","features":{"overall":{"growth_form":{"value":"herb","weight":1},"height_estimate":{"value":"1-2m","weight":2},"latex":{"value":"none","weight":1},"smell":{"value":"pungent","weight":5},"habitat":{"value":"roadside","weight":2},"water_droplet_test":{"value":"flat","weight":1}},"leaf":{"leaf_type":{"value":"simple","weight":1},"shape":{"value":"oval","weight":1},"edge":{"value":"serrated","weight":2},"tip":{"value":"acute","weight":3},"base":{"value":"cuneate","weight":1},"colors":{"value":["green"],"weight":1},"color_pattern":{"value":"solid","weight":1},"surface_top":{"value":"rough","weight":4},"surface_bottom":{"value":"matte","weight":1},"arrangement":{"value":"alternate","weight":1},"size":{"value":"large_15-50cm","weight":2},"venation":{"value":"reticulate","weight":3},"texture":{"value":"papery","weight":1},"petiole_attach":{"value":"normal","weight":1}},"stem":{"type":{"value":"erect","weight":1},"cross_section":{"value":"round","weight":1},"surface":{"value":"smooth","weight":1},"colors":{"value":["purple"],"weight":3},"interior":{"value":"solid","weight":1},"has_thorns":{"value":"no","weight":1}},"flower":{"colors":{"value":["white","purple"],"weight":3},"color_pattern":{"value":"solid","weight":1},"petal_count":{"value":"5","weight":1},"symmetry":{"value":"radial","weight":1},"size":{"value":"very_large_>60mm","weight":5},"arrangement":{"value":"solitary","weight":3},"position":{"value":"terminal","weight":1},"orientation":{"value":"upright","weight":1},"special_shape":{"value":"trumpet","weight":4},"fragrant":{"value":"no","weight":1}},"fruit":{"type":{"value":"capsule","weight":1},"colors":{"value":["green"],"weight":1},"size":{"value":"large_>30mm","weight":4},"surface":{"value":"spiny","weight":5}},"root":{"type":{"value":"fibrous","weight":1}}},"human_readable":{"toxicity":"全株劇毒，含莨菪鹼（scopolamine）、東莨菪鹼（atropine），誤食致譫妄、幻覺、死亡","symptoms":["瞳孔放大","口乾","心跳加速","幻覺","譫妄","抽搐","昏迷","可致死"],"first_aid":"立即催吐（若意識清醒且在 1 小時內），儘速送醫。注意維持呼吸道暢通。","affected_parts":["全株","種子（濃度最高）","葉"],"diagnostic_features":["卵形，長 8-20cm，邊緣有粗鋸齒或不規則裂片","略粗糙，暗綠色，搓揉後有臭味","長 3-6cm","直立草本或亞灌木，高 50-150cm，莖綠色或紫色，光滑","白色或淡紫色喇叭形大花，長 6-10cm，單生於枝腋"]},"distribution":{"altitude_range":[0,1000],"climate_zones":["subtropical","temperate","tropical"]},"total_weight":76,"photo_observable_weight":66},{"id":"urtica_thunbergiana","scientific_name":"Urtica thunbergiana","common_names":{"zh-TW":"咬人貓","en":"Stinging Nettle (Taiwan)"},"category":"dangerous","danger_level":"medium","features":{"overall":{"growth_form":{"value":"herb","weight":1},"height_estimate":{"value":"30-100cm","weight":1},"latex":{"value":"none","weight":1},"smell":{"value":"none","weight":1},"habitat":{"value":"forest_floor","weight":2},"water_droplet_test":{"value":"flat","weight":1}},"leaf":{"leaf_type":{"value":"simple","weight":1},"shape":{"value":"heart","weight":3},"edge":{"value":"serrated","weight":2},"tip":{"value":"acute","weight":3},"base":{"value":"cordate","weight":3},"colors":{"value":["green"],"weight":1},"color_pattern":{"value":"solid","weight":1},"surface_top":{"value":"hairy","weight":3},"surface_bottom":{"value":"matte","weight":1},"arrangement":{"value":"opposite","weight":4},"size":{"value":"medium_5-15cm","weight":1},"venation":{"value":"palmate","weight":3},"texture":{"value":"papery","weight":1},"petiole_attach":{"value":"normal","weight":1}},"stem":{"type":{"value":"erect","weight":1},"cross_section":{"value":"square","weight":4},"surface":{"value":"thorny","weight":5},"colors":{"value":["green"],"weight":1},"interior":{"value":"solid","weight":1},"has_thorns":{"value":"yes","weight":4}},"flower":{"colors":{"value":["green"],"weight":2},"color_pattern":{"value":"solid","weight":1},"petal_count":{"value":"0","weight":4},"symmetry":{"value":"radial","weight":1},"size":{"value":"tiny_<5mm","weight":4},"arrangement":{"value":"spike","weight":3},"position":{"value":"terminal","weight":1},"orientation":{"value":"upright","weight":1},"special_shape":{"value":"none","weight":1},"fragrant":{"value":"no","weight":1}},"fruit":{"type":{"value":"achene","weight":3},"colors":{"value":["green"],"weight":1},"size":{"value":"small_5-15mm","weight":1},"surface":{"value":"smooth","weight":1}},"root":{"type":{"value":"fibrous","weight":1}}},"human_readable":{"toxicity":"莖葉有透明焮刺毛，觸碰注入蟻酸等化學物質，引起劇烈灼痛，持續數小時","symptoms":["劇烈灼痛","皮膚紅腫","丘疹","疼痛持續數小時至一天"],"first_aid":"用膠帶黏除皮膚上的刺毛。冷水沖洗患處。可塗抹含類固醇的止癢藥膏。避免搔抓。","affected_parts":["莖","葉","焮刺毛"],"diagnostic_features":["闊卵形至心形，長 6-13cm，邊緣有粗鋸齒","兩面均有焮刺毛（透明針狀突起，肉眼可見）——關鍵特徵","網狀脈，3-5 條基出脈","長 3-8cm，有刺毛","直立草本，高 50-120cm，莖上也有刺毛，方形莖"]},"distribution":{"altitude_range":[500,3000],"climate_zones":["subtropical","temperate"]},"total_weight":77,"photo_observable_weight":71},{"id":"dendrocnide_meyeniana","scientific_name":"Dendrocnide meyeniana","common_names":{"zh-TW":"咬人狗","en":"Stinging Tree (Taiwan)"},"category":"dangerous","danger_level":"high","features":{"overall":{"growth_form":{"value":"shrub","weight":3},"height_estimate":{"value":"2-5m","weight":3},"latex":{"value":"none","weight":1},"smell":{"value":"none","weight":1},"habitat":{"value":"forest_floor","weight":2},"water_droplet_test":{"value":"flat","weight":1}},"leaf":{"leaf_type":{"value":"simple","weight":1},"shape":{"value":"oval","weight":1},"edge":{"value":"entire","weight":1},"tip":{"value":"acute","weight":3},"base":{"value":"cuneate","weight":1},"colors":{"value":["green"],"weight":1},"color_pattern":{"value":"solid","weight":1},"surface_top":{"value":"glossy","weight":2},"surface_bottom":{"value":"hairy","weight":4},"arrangement":{"value":"alternate","weight":1},"size":{"value":"large_15-50cm","weight":2},"venation":{"value":"palmate","weight":3},"texture":{"value":"papery","weight":1},"petiole_attach":{"value":"normal","weight":1}},"stem":{"type":{"value":"erect","weight":1},"cross_section":{"value":"round","weight":1},"surface":{"value":"thorny","weight":5},"colors":{"value":["green"],"weight":1},"interior":{"value":"solid","weight":1},"has_thorns":{"value":"yes","weight":4}},"flower":{"colors":{"value":["green"],"weight":2},"color_pattern":{"value":"solid","weight":1},"petal_count":{"value":"5","weight":1},"symmetry":{"value":"radial","weight":1},"size":{"value":"medium_15-30mm","weight":1},"arrangement":{"value":"cyme","weight":4},"position":{"value":"terminal","weight":1},"orientation":{"value":"upright","weight":1},"special_shape":{"value":"none","weight":1},"fragrant":{"value":"no","weight":1}},"fruit":{"type":{"value":"drupe","weight":3},"colors":{"value":["green"],"weight":1},"size":{"value":"small_5-15mm","weight":1},"surface":{"value":"smooth","weight":1}},"root":{"type":{"value":"fibrous","weight":1}}},"human_readable":{"toxicity":"焮刺毛含蟻酸等毒素，觸碰引起劇烈灼痛，疼痛可持續數週至數月","symptoms":["劇烈灼痛","持續疼痛數週至數月","皮膚紅腫","嚴重者可能過敏性休克"],"first_aid":"用膠帶反覆黏除刺毛。冷水沖洗。勿用手觸摸患處。疼痛持續超過一天應就醫。","affected_parts":["全株表面（刺毛）"],"diagnostic_features":["大型卵形至圓形，長 15-30cm，寬 10-20cm，全緣或微波狀","正面光滑或疏被短毛，背面密生刺毛（關鍵特徵）","明顯的三出脈","長 5-15cm","灌木至小喬木，高 2-5m，莖上有刺毛"]},"distribution":{"altitude_range":[0,800],"climate_zones":["subtropical","tropical"]},"total_weight":68,"photo_observable_weight":62},{"id":"chlorophyllum_molybdites","scientific_name":"Chlorophyllum molybdites","common_names":{"zh-TW":"綠褶菇","en":"Green-spored Parasol"},"category":"dangerous","danger_level":"high","features":{"overall":{"growth_form":{"value":"fungus","weight":3},"height_estimate":{"value":"<30cm","weight":3},"latex":{"value":"none","weight":1},"smell":{"value":"none","weight":1},"habitat":{"value":"open_field","weight":3},"water_droplet_test":{"value":"flat","weight":1}},"leaf":{"leaf_type":{"value":"simple","weight":1},"shape":{"value":"oval","weight":1},"edge":{"value":"entire","weight":1},"tip":{"value":"rounded","weight":3},"base":{"value":"rounded","weight":3},"colors":{"value":["green"],"weight":1},"color_pattern":{"value":"solid","weight":1},"surface_top":{"value":"scaly","weight":4},"surface_bottom":{"value":"matte","weight":1},"arrangement":{"value":"clustered","weight":2},"size":{"value":"large_15-50cm","weight":2},"venation":{"value":"reticulate","weight":3},"texture":{"value":"fleshy","weight":3},"petiole_attach":{"value":"normal","weight":1}},"stem":{"type":{"value":"erect","weight":1},"cross_section":{"value":"round","weight":1},"surface":{"value":"smooth","weight":1},"colors":{"value":["green"],"weight":1},"interior":{"value":"solid","weight":1},"has_thorns":{"value":"no","weight":1}},"flower":null,"fruit":{"type":{"value":"spore","weight":2},"colors":{"value":["green"],"weight":1},"size":{"value":"tiny_<5mm","weight":2},"surface":{"value":"smooth","weight":1}},"root":{"type":{"value":"fibrous","weight":1}}},"human_readable":{"toxicity":"含多種毒素，誤食致嚴重腸胃炎，為台灣最常見的誤食毒菇","symptoms":["嚴重嘔吐","腹瀉","腹痛","脫水","症狀在食後 1-3 小時出現"],"first_aid":"催吐（若在食後 1 小時內），大量補充水分防止脫水，儘速就醫。保留殘餘菇體供醫師辨識。","affected_parts":["子實體（全株）"],"diagnostic_features":["傘蓋直徑 10-30cm，初球形後展平，白色至淡褐色，中央有褐色鱗片","初白色，成熟後轉為綠色（關鍵特徵）","菌柄長 10-25cm，白色，基部膨大，有可移動的菌環","綠色（關鍵特徵）","白色，受傷後不變色或微變褐色"]},"distribution":{"altitude_range":[0,500],"climate_zones":["subtropical","tropical"]},"total_weight":52,"photo_observable_weight":47},{"id":"amanita_phalloides","scientific_name":"Amanita phalloides","common_names":{"zh-TW":"鱗柄白毒鵝膏（死帽蕈近似種）","en":"Death Cap (related species)"},"category":"dangerous","danger_level":"critical","features":{"overall":{"growth_form":{"value":"fungus","weight":3},"height_estimate":{"value":"<30cm","weight":3},"latex":{"value":"none","weight":1},"smell":{"value":"none","weight":1},"habitat":{"value":"forest_floor","weight":2},"water_droplet_test":{"value":"flat","weight":1}},"leaf":{"leaf_type":{"value":"simple","weight":1},"shape":{"value":"oval","weight":1},"edge":{"value":"entire","weight":1},"tip":{"value":"rounded","weight":3},"base":{"value":"rounded","weight":3},"colors":{"value":["green"],"weight":1},"color_pattern":{"value":"solid","weight":1},"surface_top":{"value":"glossy","weight":2},"surface_bottom":{"value":"matte","weight":1},"arrangement":{"value":"clustered","weight":2},"size":{"value":"medium_5-15cm","weight":1},"venation":{"value":"reticulate","weight":3},"texture":{"value":"fleshy","weight":3},"petiole_attach":{"value":"normal","weight":1}},"stem":{"type":{"value":"erect","weight":1},"cross_section":{"value":"round","weight":1},"surface":{"value":"smooth","weight":1},"colors":{"value":["green"],"weight":1},"interior":{"value":"solid","weight":1},"has_thorns":{"value":"no","weight":1}},"flower":null,"fruit":{"type":{"value":"spore","weight":2},"colors":{"value":["green"],"weight":1},"size":{"value":"tiny_<5mm","weight":2},"surface":{"value":"smooth","weight":1}},"root":{"type":{"value":"fibrous","weight":1}}},"human_readable":{"toxicity":"含鵝膏毒素（amatoxin），致死量極低，誤食致肝腎衰竭，死亡率極高","symptoms":["潛伏期 6-24 小時（危險！初期無症狀）","嘔吐腹瀉（假性恢復期後加重）","肝腎衰竭","黃疸","多器官衰竭","死亡"],"first_aid":"立即送醫！告知疑似鵝膏毒素中毒。保留殘餘菇體。此毒素無特效解毒劑，需緊急支持性治療。時間是關鍵。","affected_parts":["子實體（全株）"],"diagnostic_features":["傘蓋直徑 5-15cm，白色至淡綠色或淡黃色，光滑，幼時半球形後展平","白色，離生，密集排列","菌柄長 8-15cm，白色，基部有明顯的菌托（杯狀，關鍵特徵）","菌柄上部有白色薄膜狀菌環","白色，無特殊氣味（危險！易被認為無毒）"]},"distribution":{"altitude_range":[500,2500],"climate_zones":["subtropical","temperate"]},"total_weight":48,"photo_observable_weight":43},{"id":"toxicodendron_vernicifluum","scientific_name":"Toxicodendron vernicifluum","common_names":{"zh-TW":"漆樹","en":"Chinese Lacquer Tree"},"category":"dangerous","danger_level":"medium","features":{"overall":{"growth_form":{"value":"tree","weight":3},"height_estimate":{"value":">5m","weight":3},"latex":{"value":"yes_white","weight":2},"smell":{"value":"none","weight":1},"habitat":{"value":"forest_floor","weight":2},"water_droplet_test":{"value":"flat","weight":1}},"leaf":{"leaf_type":{"value":"pinnate_compound","weight":3},"shape":{"value":"oval","weight":1},"edge":{"value":"entire","weight":1},"tip":{"value":"acuminate","weight":1},"base":{"value":"cuneate","weight":1},"colors":{"value":["purple"],"weight":4},"color_pattern":{"value":"solid","weight":1},"surface_top":{"value":"glossy","weight":2},"surface_bottom":{"value":"matte","weight":1},"arrangement":{"value":"alternate","weight":1},"size":{"value":"medium_5-15cm","weight":1},"venation":{"value":"pinnate","weight":1},"texture":{"value":"papery","weight":1},"petiole_attach":{"value":"normal","weight":1}},"stem":{"type":{"value":"erect","weight":1},"cross_section":{"value":"round","weight":1},"surface":{"value":"smooth","weight":1},"colors":{"value":["gray"],"weight":4},"interior":{"value":"solid","weight":1},"has_thorns":{"value":"no","weight":1}},"flower":{"colors":{"value":["yellow","green"],"weight":2},"color_pattern":{"value":"solid","weight":1},"petal_count":{"value":"5","weight":1},"symmetry":{"value":"radial","weight":1},"size":{"value":"medium_15-30mm","weight":1},"arrangement":{"value":"panicle","weight":4},"position":{"value":"terminal","weight":1},"orientation":{"value":"upright","weight":1},"special_shape":{"value":"none","weight":1},"fragrant":{"value":"no","weight":1}},"fruit":{"type":{"value":"drupe","weight":3},"colors":{"value":["yellow"],"weight":4},"size":{"value":"small_5-15mm","weight":1},"surface":{"value":"smooth","weight":1}},"root":{"type":{"value":"fibrous","weight":1}}},"human_readable":{"toxicity":"樹液含漆酚（urushiol），接觸致嚴重過敏性接觸性皮膚炎","symptoms":["嚴重皮膚紅腫","水泡","劇烈搔癢","過敏者可能全身性反應"],"first_aid":"立即用大量肥皂水清洗接觸部位（15分鐘內最有效）。勿搔抓水泡。塗抹含類固醇藥膏。嚴重者就醫。","affected_parts":["樹液（漆液）","枝葉"],"diagnostic_features":["奇數羽狀複葉，小葉 7-13 枚，每枚長 7-15cm，卵形至橢圓形","幼葉微被毛，成熟葉近光滑，秋季轉紅（漂亮但危險）","葉軸長 20-40cm","落葉喬木，高 10-20m，樹皮灰白色，切割後流出白色漆液（接觸即過敏）","圓錐花序，小型黃綠色花"]},"distribution":{"altitude_range":[200,2500],"climate_zones":["subtropical","temperate"]},"total_weight":65,"photo_observable_weight":58},{"id":"nerium_oleander","scientific_name":"Nerium oleander","common_names":{"zh-TW":"夾竹桃","en":"Oleander"},"category":"dangerous","danger_level":"critical","features":{"overall":{"growth_form":{"value":"shrub","weight":3},"height_estimate":{"value":"2-5m","weight":3},"latex":{"value":"yes_white","weight":2},"smell":{"value":"aromatic","weight":2},"habitat":{"value":"urban","weight":3},"water_droplet_test":{"value":"flat","weight":1}},"leaf":{"leaf_type":{"value":"simple","weight":1},"shape":{"value":"lanceolate","weight":2},"edge":{"value":"entire","weight":1},"tip":{"value":"acuminate","weight":1},"base":{"value":"cuneate","weight":1},"colors":{"value":["dark_green"],"weight":2},"color_pattern":{"value":"solid","weight":1},"surface_top":{"value":"glossy","weight":2},"surface_bottom":{"value":"matte","weight":1},"arrangement":{"value":"whorled","weight":6},"size":{"value":"medium_5-15cm","weight":1},"venation":{"value":"parallel","weight":4},"texture":{"value":"leathery","weight":2},"petiole_attach":{"value":"normal","weight":1}},"stem":{"type":{"value":"erect","weight":1},"cross_section":{"value":"round","weight":1},"surface":{"value":"smooth","weight":1},"colors":{"value":["gray"],"weight":4},"interior":{"value":"solid","weight":1},"has_thorns":{"value":"no","weight":1}},"flower":{"colors":{"value":["white","red","pink"],"weight":3},"color_pattern":{"value":"solid","weight":1},"petal_count":{"value":"5","weight":1},"symmetry":{"value":"radial","weight":1},"size":{"value":"medium_15-30mm","weight":1},"arrangement":{"value":"cyme","weight":4},"position":{"value":"terminal","weight":1},"orientation":{"value":"upright","weight":1},"special_shape":{"value":"trumpet","weight":4},"fragrant":{"value":"yes","weight":3}},"fruit":{"type":{"value":"capsule","weight":1},"colors":{"value":["green"],"weight":1},"size":{"value":"small_5-15mm","weight":1},"surface":{"value":"smooth","weight":1}},"root":{"type":{"value":"fibrous","weight":1}}},"human_readable":{"toxicity":"全株劇毒，含強心苷（oleandrin），誤食極少量即可致心臟停跳死亡","symptoms":["噁心嘔吐","腹痛","心律不整","心跳過慢","視覺異常","心臟停跳","可致死"],"first_aid":"立即催吐（若意識清醒），儘速送醫。此為心臟毒性，需持續心電圖監測。勿用夾竹桃枝條烤食物。","affected_parts":["全株（包括乾枯的葉和枝）","花蜜","燃燒煙霧也有毒"],"diagnostic_features":["線狀披針形，長 10-20cm，寬 1-2.5cm，革質，葉緣全緣略反捲","深綠色，光滑，葉背有明顯中脈","羽狀側脈密集，近平行","三葉輪生（關鍵特徵）","常綠灌木，高 2-5m，分枝多，枝條灰綠色"]},"distribution":{"altitude_range":[0,500],"climate_zones":["subtropical","tropical","temperate"]},"total_weight":74,"photo_observable_weight":64},{"id":"cycas_revoluta","scientific_name":"Cycas revoluta","common_names":{"zh-TW":"蘇鐵（鐵樹）","en":"Sago Palm"},"category":"dangerous","danger_level":"high","features":{"overall":{"growth_form":{"value":"palm_like","weight":6},"height_estimate":{"value":"1-2m","weight":2},"latex":{"value":"none","weight":1},"smell":{"value":"none","weight":1},"habitat":{"value":"urban","weight":3},"water_droplet_test":{"value":"flat","weight":1}},"leaf":{"leaf_type":{"value":"pinnate_compound","weight":3},"shape":{"value":"linear","weight":5},"edge":{"value":"entire","weight":1},"tip":{"value":"acuminate","weight":1},"base":{"value":"cuneate","weight":1},"colors":{"value":["dark_green"],"weight":2},"color_pattern":{"value":"solid","weight":1},"surface_top":{"value":"glossy","weight":2},"surface_bottom":{"value":"matte","weight":1},"arrangement":{"value":"clustered","weight":2},"size":{"value":"very_large_>50cm","weight":3},"venation":{"value":"single_midrib","weight":5},"texture":{"value":"leathery","weight":2},"petiole_attach":{"value":"normal","weight":1}},"stem":{"type":{"value":"erect","weight":1},"cross_section":{"value":"round","weight":1},"surface":{"value":"scaly","weight":3},"colors":{"value":["green"],"weight":1},"interior":{"value":"solid","weight":1},"has_thorns":{"value":"yes","weight":4}},"flower":{"colors":{"value":["white"],"weight":2},"color_pattern":{"value":"solid","weight":1},"petal_count":{"value":"5","weight":1},"symmetry":{"value":"radial","weight":1},"size":{"value":"medium_15-30mm","weight":1},"arrangement":{"value":"solitary","weight":3},"position":{"value":"terminal","weight":1},"orientation":{"value":"upright","weight":1},"special_shape":{"value":"none","weight":1},"fragrant":{"value":"no","weight":1}},"fruit":{"type":{"value":"capsule","weight":1},"colors":{"value":["orange","red"],"weight":4},"size":{"value":"large_>30mm","weight":4},"surface":{"value":"smooth","weight":1}},"root":{"type":{"value":"fibrous","weight":1}}},"human_readable":{"toxicity":"種子和嫩葉含蘇鐵苷（cycasin），誤食致肝臟損害，有致癌性","symptoms":["嘔吐","腹瀉","肝臟損害（延遲性）","黃疸","長期接觸有致癌風險"],"first_aid":"催吐（若在食後 1 小時內），儘速送醫。需追蹤肝功能指標。","affected_parts":["種子（毒性最高）","嫩葉","根"],"diagnostic_features":["大型羽狀複葉，長 50-150cm，小葉線形，硬挺，邊緣反捲","深綠色，革質，有光澤","小葉僅一條中脈","叢生於莖頂，向外輻射展開（棕櫚狀）","葉柄基部有刺"]},"distribution":{"altitude_range":[0,500],"climate_zones":["subtropical","tropical"]},"total_weight":79,"photo_observable_weight":73},{"id":"lantana_camara","scientific_name":"Lantana camara","common_names":{"zh-TW":"馬纓丹","en":"Lantana"},"category":"dangerous","danger_level":"high","features":{"overall":{"growth_form":{"value":"shrub","weight":3},"height_estimate":{"value":"30-100cm","weight":1},"latex":{"value":"none","weight":1},"smell":{"value":"pungent","weight":5},"habitat":{"value":"roadside","weight":2},"water_droplet_test":{"value":"flat","weight":1}},"leaf":{"leaf_type":{"value":"simple","weight":1},"shape":{"value":"oval","weight":1},"edge":{"value":"serrated","weight":2},"tip":{"value":"acuminate","weight":1},"base":{"value":"cuneate","weight":1},"colors":{"value":["dark_green"],"weight":2},"color_pattern":{"value":"solid","weight":1},"surface_top":{"value":"hairy","weight":3},"surface_bottom":{"value":"matte","weight":1},"arrangement":{"value":"opposite","weight":4},"size":{"value":"medium_5-15cm","weight":1},"venation":{"value":"reticulate","weight":3},"texture":{"value":"papery","weight":1},"petiole_attach":{"value":"normal","weight":1}},"stem":{"type":{"value":"erect","weight":1},"cross_section":{"value":"square","weight":4},"surface":{"value":"hairy","weight":4},"colors":{"value":["green"],"weight":1},"interior":{"value":"solid","weight":1},"has_thorns":{"value":"no","weight":1}},"flower":{"colors":{"value":["yellow","orange","red","pink"],"weight":5},"color_pattern":{"value":"solid","weight":1},"petal_count":{"value":"5","weight":1},"symmetry":{"value":"radial","weight":1},"size":{"value":"medium_15-30mm","weight":1},"arrangement":{"value":"umbel","weight":4},"position":{"value":"terminal","weight":1},"orientation":{"value":"upright","weight":1},"special_shape":{"value":"none","weight":1},"fragrant":{"value":"no","weight":1}},"fruit":{"type":{"value":"berry","weight":3},"colors":{"value":["purple","green"],"weight":3},"size":{"value":"small_5-15mm","weight":1},"surface":{"value":"smooth","weight":1}},"root":{"type":{"value":"fibrous","weight":1}}},"human_readable":{"toxicity":"全株具毒性，以未成熟綠色漿果毒性最顯著，含三萜類毒素 lantadene；誤食可引起嘔吐、腹瀉、肝損傷，嚴重可危及生命。成熟紫黑色果實仍存在風險，應一律避免採食。","symptoms":["嘔吐","腹瀉","腹痛","虛弱","肝功能異常（嚴重時）","呼吸困難（少見但嚴重）"],"first_aid":"勿再進食；若剛誤食可於意識清醒且醫護指示下處理。儘速就醫並告知誤食馬纓丹漿果。保留樣本照片或枝條以利辨識。大量清水漱口。","affected_parts":["全株","未熟綠色漿果（高風險）","成熟漿果","葉"],"diagnostic_features":["對生，兩兩相對於莖節上","卵形至闊卵形，長 3-8cm，先端鈍或銳尖，葉緣具粗鋸齒，鋸齒明顯易見","表面略粗糙，深綠色，常帶短柔毛；搓揉葉片會散出特殊刺鼻臭味（辨識線索）","羽狀網狀脈，側脈明顯向葉緣延伸","小型花密集組成頭狀或半球形繖房狀花序；花色隨開花進程變化，常見由黃轉橙再轉紅，亦有粉、橙紅等色，花期長"]},"distribution":{"altitude_range":[0,500],"climate_zones":["subtropical","tropical"]},"total_weight":74,"photo_observable_weight":64},{"id":"parthenium_hysterophorus","scientific_name":"Parthenium hysterophorus","common_names":{"zh-TW":"銀膠菊","en":"Parthenium Weed"},"category":"dangerous","danger_level":"medium","features":{"overall":{"growth_form":{"value":"herb","weight":1},"height_estimate":{"value":"30-100cm","weight":1},"latex":{"value":"none","weight":1},"smell":{"value":"none","weight":1},"habitat":{"value":"roadside","weight":2},"water_droplet_test":{"value":"flat","weight":1}},"leaf":{"leaf_type":{"value":"simple","weight":1},"shape":{"value":"lanceolate","weight":2},"edge":{"value":"lobed","weight":3},"tip":{"value":"acuminate","weight":1},"base":{"value":"cuneate","weight":1},"colors":{"value":["dark_green"],"weight":2},"color_pattern":{"value":"solid","weight":1},"surface_top":{"value":"hairy","weight":3},"surface_bottom":{"value":"matte","weight":1},"arrangement":{"value":"alternate","weight":1},"size":{"value":"medium_5-15cm","weight":1},"venation":{"value":"pinnate","weight":1},"texture":{"value":"papery","weight":1},"petiole_attach":{"value":"normal","weight":1}},"stem":{"type":{"value":"erect","weight":1},"cross_section":{"value":"round","weight":1},"surface":{"value":"smooth","weight":1},"colors":{"value":["purple"],"weight":3},"interior":{"value":"solid","weight":1},"has_thorns":{"value":"no","weight":1}},"flower":{"colors":{"value":["white"],"weight":2},"color_pattern":{"value":"solid","weight":1},"petal_count":{"value":"many","weight":4},"symmetry":{"value":"radial","weight":1},"size":{"value":"medium_15-30mm","weight":1},"arrangement":{"value":"head_composite","weight":3},"position":{"value":"terminal","weight":1},"orientation":{"value":"upright","weight":1},"special_shape":{"value":"none","weight":1},"fragrant":{"value":"no","weight":1}},"fruit":{"type":{"value":"achene","weight":3},"colors":{"value":["green"],"weight":1},"size":{"value":"small_5-15mm","weight":1},"surface":{"value":"smooth","weight":1}},"root":{"type":{"value":"fibrous","weight":1}}},"human_readable":{"toxicity":"入侵種。花粉、碎裂植株與汁液可誘發嚴重過敏性皮膚炎（發紅、腫、水泡、劇癢），亦可加重或誘發氣喘與呼吸道過敏。長期或重複暴露風險高。","symptoms":["皮膚紅腫","丘疹與水泡","劇烈搔癢","皮炎可久治不愈","咳嗽","喘鳴","鼻癢噴嚏"],"first_aid":"離開現場。以大量清水沖洗皮膚，更換衣物。皮膚炎可冷敷並依醫囑使用抗組織胺或類固醇藥膏；呼吸喘促或胸悶應立即就醫。從事清除工作應戴口罩、長袖與手套。","affected_parts":["花粉","葉與莖汁液","揚起的植株碎屑"],"diagnostic_features":["一年生直立草本，株高通常 30-100cm，多分枝，整株呈疏鬆灌叢狀","葉互生，下位葉柄較長，向上漸短","葉羽狀深裂，裂片狹窄、線形至披針形，整體外觀繁複如萹苣類或茴蒿類切葉","表面疏被短毛或近光滑，灰綠至深綠色，揉碎有草本植物氣味","花序為多數小型白色頭狀花密集排列於枝端或腋生圓錐狀分枝上，遠觀如小白菊或滿天星狀的白色花團"]},"distribution":{"altitude_range":[0,800],"climate_zones":["subtropical","tropical"]},"total_weight":58,"photo_observable_weight":52},{"id":"brugmansia_suaveolens","scientific_name":"Brugmansia suaveolens","common_names":{"zh-TW":"大花曼陀羅","en":"Angel's Trumpet"},"category":"dangerous","danger_level":"critical","features":{"overall":{"growth_form":{"value":"shrub","weight":3},"height_estimate":{"value":"2-5m","weight":3},"latex":{"value":"none","weight":1},"smell":{"value":"aromatic","weight":2},"habitat":{"value":"urban","weight":3},"water_droplet_test":{"value":"flat","weight":1}},"leaf":{"leaf_type":{"value":"simple","weight":1},"shape":{"value":"oval","weight":1},"edge":{"value":"entire","weight":1},"tip":{"value":"acuminate","weight":1},"base":{"value":"cuneate","weight":1},"colors":{"value":["dark_green"],"weight":2},"color_pattern":{"value":"solid","weight":1},"surface_top":{"value":"matte","weight":1},"surface_bottom":{"value":"matte","weight":1},"arrangement":{"value":"alternate","weight":1},"size":{"value":"large_15-50cm","weight":2},"venation":{"value":"pinnate","weight":1},"texture":{"value":"leathery","weight":2},"petiole_attach":{"value":"normal","weight":1}},"stem":{"type":{"value":"erect","weight":1},"cross_section":{"value":"round","weight":1},"surface":{"value":"smooth","weight":1},"colors":{"value":["green"],"weight":1},"interior":{"value":"solid","weight":1},"has_thorns":{"value":"no","weight":1}},"flower":{"colors":{"value":["white","yellow"],"weight":2},"color_pattern":{"value":"solid","weight":1},"petal_count":{"value":"5","weight":1},"symmetry":{"value":"radial","weight":1},"size":{"value":"very_large_>60mm","weight":5},"arrangement":{"value":"solitary","weight":3},"position":{"value":"terminal","weight":1},"orientation":{"value":"drooping","weight":6},"special_shape":{"value":"trumpet","weight":4},"fragrant":{"value":"yes","weight":3}},"fruit":{"type":{"value":"capsule","weight":1},"colors":{"value":["green"],"weight":1},"size":{"value":"small_5-15mm","weight":1},"surface":{"value":"smooth","weight":1}},"root":{"type":{"value":"fibrous","weight":1}}},"human_readable":{"toxicity":"全株劇毒，富含莨菪鹼、東莨菪鹼等莨菪烷類生物鹼；誤食或誤用入藥可致譫妄、幻覺、瞳孔放大、心跳與體溫異常、抽搐、昏迷及呼吸抑制，致死風險高。","symptoms":["口乾","瞳孔放大","視力模糊","心悸","譫妄","幻覺","躁動或嗜睡","抽搐","昏迷","呼吸衰竭"],"first_aid":"立即呼叫急救。勿自行催吐除非醫護指示。保持呼吸道暢通，側臥防嗆咳。儘速送醫並攜帶植株照片。醫院以支持療法為主，可能需鎮靜與監測。","affected_parts":["全株","花","葉","種子"],"diagnostic_features":["灌木至小喬木，高常見 2-5m，庭園與公園常修剪成樹形","單葉大型，寬橢圓形至卵形，長可達 15-30cm，葉基常楔形或圓鈍，葉尖長漸尖，全緣或微波狀緣","葉面深綠、質地薄革質至草質，葉背色較淡；主脈粗，側脈弧形上舉","葉柄長，略粗，支撐下垂的大葉","花極大，長筒狀喇叭形，長約 20-30cm，花冠先端開展；關鍵特徵為花朵明顯下垂（朝下懸掛），有濃郁甜香，花色以白或淡黃"]},"distribution":{"altitude_range":[0,800],"climate_zones":["subtropical","tropical"]},"total_weight":68,"photo_observable_weight":59},{"id":"phytolacca_americana","scientific_name":"Phytolacca americana","common_names":{"zh-TW":"美洲商陸","en":"American Pokeweed"},"category":"dangerous","danger_level":"high","features":{"overall":{"growth_form":{"value":"herb","weight":1},"height_estimate":{"value":"1-2m","weight":2},"latex":{"value":"none","weight":1},"smell":{"value":"none","weight":1},"habitat":{"value":"roadside","weight":2},"water_droplet_test":{"value":"flat","weight":1}},"leaf":{"leaf_type":{"value":"simple","weight":1},"shape":{"value":"oval","weight":1},"edge":{"value":"entire","weight":1},"tip":{"value":"acuminate","weight":1},"base":{"value":"cuneate","weight":1},"colors":{"value":["dark_green"],"weight":2},"color_pattern":{"value":"solid","weight":1},"surface_top":{"value":"matte","weight":1},"surface_bottom":{"value":"matte","weight":1},"arrangement":{"value":"alternate","weight":1},"size":{"value":"large_15-50cm","weight":2},"venation":{"value":"pinnate","weight":1},"texture":{"value":"papery","weight":1},"petiole_attach":{"value":"normal","weight":1}},"stem":{"type":{"value":"erect","weight":1},"cross_section":{"value":"round","weight":1},"surface":{"value":"smooth","weight":1},"colors":{"value":["purple"],"weight":3},"interior":{"value":"pithy","weight":3},"has_thorns":{"value":"no","weight":1}},"flower":{"colors":{"value":["white"],"weight":2},"color_pattern":{"value":"solid","weight":1},"petal_count":{"value":"5","weight":1},"symmetry":{"value":"radial","weight":1},"size":{"value":"medium_15-30mm","weight":1},"arrangement":{"value":"raceme","weight":4},"position":{"value":"terminal","weight":1},"orientation":{"value":"upright","weight":1},"special_shape":{"value":"none","weight":1},"fragrant":{"value":"no","weight":1}},"fruit":{"type":{"value":"berry","weight":3},"colors":{"value":["purple","green"],"weight":3},"size":{"value":"small_5-15mm","weight":1},"surface":{"value":"smooth","weight":1}},"root":{"type":{"value":"taproot","weight":5}}},"human_readable":{"toxicity":"全株有毒，以根部毒性最強；含商陸皂苷、商陸毒素等。誤食可致嚴重嘔吐、腹瀉、電解質失衡，乃至中樞抑制與循環衰竭。採野菜者最易誤認。","symptoms":["噁心嘔吐","腹痛腹瀉","大量腹瀉致脫水","頭暈","心跳變化","抽搐（嚴重）","呼吸抑制"],"first_aid":"禁止再進食。意識清楚時儘速就醫，勿自行催吐除非醫護建議。保留殘餘植物供辨識。注意補充水分與電解質需由醫療人員評估。","affected_parts":["根（最高風險）","莖","葉","漿果"],"diagnostic_features":["多年生直立草本，高可達 1-3m，莖粗壯，全株多汁","成熟莖常呈紅紫或紫紅色，幼莖綠色帶紫斑，易與某些可食莖葉植物混淆（危險）","大型互生葉，橢圓形至卵狀披針形，長 10-30cm，寬 5-15cm，先端漸尖，全緣","葉質薄至略厚，表面深綠，葉背較淡；葉柄長，著生於莖上呈螺旋互生","總狀花序腋生或頂生，小花白色，花梗與花軸隨發育伸長；花期長，花序直立或略下垂"]},"distribution":{"altitude_range":[0,1500],"climate_zones":["subtropical","temperate","tropical"]},"total_weight":61,"photo_observable_weight":49},{"id":"zantedeschia_aethiopica","scientific_name":"Zantedeschia aethiopica","common_names":{"zh-TW":"海芋","en":"Calla Lily"},"category":"dangerous","danger_level":"medium","features":{"overall":{"growth_form":{"value":"herb","weight":1},"height_estimate":{"value":"30-100cm","weight":1},"latex":{"value":"none","weight":1},"smell":{"value":"none","weight":1},"habitat":{"value":"streamside","weight":3},"water_droplet_test":{"value":"flat","weight":1}},"leaf":{"leaf_type":{"value":"simple","weight":1},"shape":{"value":"arrow","weight":6},"edge":{"value":"entire","weight":1},"tip":{"value":"acuminate","weight":1},"base":{"value":"cordate","weight":3},"colors":{"value":["dark_green"],"weight":2},"color_pattern":{"value":"solid","weight":1},"surface_top":{"value":"waxy","weight":6},"surface_bottom":{"value":"matte","weight":1},"arrangement":{"value":"clustered","weight":2},"size":{"value":"large_15-50cm","weight":2},"venation":{"value":"pinnate","weight":1},"texture":{"value":"leathery","weight":2},"petiole_attach":{"value":"normal","weight":1}},"stem":{"type":{"value":"erect","weight":1},"cross_section":{"value":"round","weight":1},"surface":{"value":"smooth","weight":1},"colors":{"value":["green"],"weight":1},"interior":{"value":"solid","weight":1},"has_thorns":{"value":"no","weight":1}},"flower":{"colors":{"value":["white","yellow"],"weight":2},"color_pattern":{"value":"solid","weight":1},"petal_count":{"value":"0","weight":4},"symmetry":{"value":"radial","weight":1},"size":{"value":"medium_15-30mm","weight":1},"arrangement":{"value":"spathe_spadix","weight":4},"position":{"value":"terminal","weight":1},"orientation":{"value":"upright","weight":1},"special_shape":{"value":"spathe","weight":4},"fragrant":{"value":"no","weight":1}},"fruit":{"type":{"value":"capsule","weight":1},"colors":{"value":["green"],"weight":1},"size":{"value":"small_5-15mm","weight":1},"surface":{"value":"smooth","weight":1}},"root":{"type":{"value":"storage_tuber","weight":4}}},"human_readable":{"toxicity":"含草酸鈣針晶及溶蛋白酵素等刺激性成分；誤食致口腔、舌頭、咽喉灼痛、流涎、吞嚥困難，可伴嘔吐。汁液接觸黏膜或敏感皮膚亦可刺激。","symptoms":["口腔灼痛","舌頭麻木","流涎","噁心嘔吐","吞嚥困難","皮膚紅癢（接觸性）"],"first_aid":"清水漱口但不要吞嚥大量生水造成嗆咳；移除口中殘渣。皮膚以大量清水沖洗。咽喉腫脹或呼吸不順立即就醫。避免與食用油鹼偏方自行處理。","affected_parts":["全株","汁液","地下莖","佛焰苞與肉穗"],"diagnostic_features":["多年生草本，高約 40-100cm，具地下莖，常成群落栽培","葉大型箭形至戟形，長 15-45cm，葉基深心形裂片，先端漸尖，全緣光滑","深綠色，具蠟質光澤，質地厚；主脈於背面隆起明顯","葉柄長而粗，淺綠色，具縱稜，支撐葉片挺立或略外展","關鍵特徵為純白色（或栽培變種其他色）大型佛焰苞，呈漏斗狀或喇叭狀包被中央之肉穗花序；肉穗花序黃色，粗短圓柱形，整體外觀優"]},"distribution":{"altitude_range":[200,1200],"climate_zones":["subtropical","temperate"]},"total_weight":72,"photo_observable_weight":63},{"id":"abrus_precatorius","scientific_name":"Abrus precatorius","common_names":{"zh-TW":"雞母珠（相思子）","en":"Rosary Pea"},"category":"dangerous","danger_level":"critical","features":{"overall":{"growth_form":{"value":"vine","weight":4},"height_estimate":{"value":"30-100cm","weight":1},"latex":{"value":"none","weight":1},"smell":{"value":"none","weight":1},"habitat":{"value":"roadside","weight":2},"water_droplet_test":{"value":"flat","weight":1}},"leaf":{"leaf_type":{"value":"pinnate_compound","weight":3},"shape":{"value":"elliptic","weight":4},"edge":{"value":"entire","weight":1},"tip":{"value":"rounded","weight":3},"base":{"value":"cuneate","weight":1},"colors":{"value":["green"],"weight":1},"color_pattern":{"value":"solid","weight":1},"surface_top":{"value":"matte","weight":1},"surface_bottom":{"value":"matte","weight":1},"arrangement":{"value":"alternate","weight":1},"size":{"value":"tiny_<2cm","weight":6},"venation":{"value":"pinnate","weight":1},"texture":{"value":"leathery","weight":2},"petiole_attach":{"value":"normal","weight":1}},"stem":{"type":{"value":"twining","weight":6},"cross_section":{"value":"round","weight":1},"surface":{"value":"smooth","weight":1},"colors":{"value":["green"],"weight":1},"interior":{"value":"solid","weight":1},"has_thorns":{"value":"no","weight":1}},"flower":{"colors":{"value":["red","pink","purple"],"weight":3},"color_pattern":{"value":"solid","weight":1},"petal_count":{"value":"5","weight":1},"symmetry":{"value":"bilateral","weight":6},"size":{"value":"small_5-15mm","weight":5},"arrangement":{"value":"raceme","weight":4},"position":{"value":"terminal","weight":1},"orientation":{"value":"upright","weight":1},"special_shape":{"value":"butterfly_shape","weight":5},"fragrant":{"value":"no","weight":1}},"fruit":{"type":{"value":"pod_legume","weight":6},"colors":{"value":["green"],"weight":1},"size":{"value":"small_5-15mm","weight":1},"surface":{"value":"smooth","weight":1}},"root":{"type":{"value":"fibrous","weight":1}}},"human_readable":{"toxicity":"種子劇毒，含雞母珠毒蛋白 abrin，毒性極強，咬碎或穿刺種皮後釋放；文獻記載極少量種子即可致命。全株其他部位亦不宜食用。","symptoms":["延遲性腸胃症狀","嚴重脫水","肝腎損傷","抽搐","休克","可致死"],"first_aid":"懷疑吞食種子視為急症，立即送醫。勿等待症狀。告知醫師時間與可能數量。避免催吐造成二次傷害；由醫療端處置。切勿把玩咬破種子。","affected_parts":["種子（劇毒）","莢果","其他部位不宜食用"],"diagnostic_features":["木質攀緣藤本，捲鬚或纏繞他物上升，可覆蓋灌叢與籬笆","偶數羽狀複葉，小葉 8-15 對，對生排列於葉軸","小葉長橢圓形，長約 1-2cm，寬 0.5-1cm，先端圓或微凹，全緣，質地薄革質，表面深綠光澤","蝶形花冠，花冠淡粉至紫紅色，腋生總狀花序，外形似小型豇豆類野花","莢果長圓形，成熟乾裂可見種子"]},"distribution":{"altitude_range":[0,800],"climate_zones":["subtropical","tropical"]},"total_weight":86,"photo_observable_weight":80},{"id":"ricinus_communis","scientific_name":"Ricinus communis","common_names":{"zh-TW":"蓖麻","en":"Castor Bean"},"category":"dangerous","danger_level":"critical","features":{"overall":{"growth_form":{"value":"shrub","weight":3},"height_estimate":{"value":"2-5m","weight":3},"latex":{"value":"none","weight":1},"smell":{"value":"none","weight":1},"habitat":{"value":"roadside","weight":2},"water_droplet_test":{"value":"flat","weight":1}},"leaf":{"leaf_type":{"value":"simple","weight":1},"shape":{"value":"round","weight":4},"edge":{"value":"deeply_lobed","weight":6},"tip":{"value":"acuminate","weight":1},"base":{"value":"cuneate","weight":1},"colors":{"value":["green"],"weight":1},"color_pattern":{"value":"solid","weight":1},"surface_top":{"value":"glossy","weight":2},"surface_bottom":{"value":"matte","weight":1},"arrangement":{"value":"alternate","weight":1},"size":{"value":"large_15-50cm","weight":2},"venation":{"value":"palmate","weight":3},"texture":{"value":"papery","weight":1},"petiole_attach":{"value":"normal","weight":1}},"stem":{"type":{"value":"erect","weight":1},"cross_section":{"value":"round","weight":1},"surface":{"value":"smooth","weight":1},"colors":{"value":["green"],"weight":1},"interior":{"value":"pithy","weight":3},"has_thorns":{"value":"no","weight":1}},"flower":{"colors":{"value":["yellow","green"],"weight":2},"color_pattern":{"value":"solid","weight":1},"petal_count":{"value":"5","weight":1},"symmetry":{"value":"radial","weight":1},"size":{"value":"medium_15-30mm","weight":1},"arrangement":{"value":"raceme","weight":4},"position":{"value":"terminal","weight":1},"orientation":{"value":"upright","weight":1},"special_shape":{"value":"none","weight":1},"fragrant":{"value":"no","weight":1}},"fruit":{"type":{"value":"capsule","weight":1},"colors":{"value":["green"],"weight":1},"size":{"value":"medium_15-30mm","weight":6},"surface":{"value":"spiny","weight":5}},"root":{"type":{"value":"fibrous","weight":1}}},"human_readable":{"toxicity":"種子含蓖麻毒蛋白 ricin 及蓖麻鹼等；嚼碎種子可致嚴重腸胃出血、器官衰竭，極少量即可致命。葉等亦可刺激。","symptoms":["噁心嘔吐","腹痛血便","脫水","低血壓","抽搐","尿量減少","多器官衰竭"],"first_aid":"立即急救送醫。勿延遲。提供吞食時間與是否咀嚼。避免自行灌腸或偏方。醫院以支持療法與監測為主。","affected_parts":["種子（劇毒）","幼苗與葉"],"diagnostic_features":["直立灌木或小喬木，高可達 2-5m，莖粗，髓疏鬆，幼莖常帶白粉或紫紅條紋","單葉掌狀深裂 5-11 裂，直徑 15-45cm，裂片狹披針形至卵狀，先端長尖，葉緣有鋸齒；整體如放射狀巨葉","葉柄極長，連接於葉片中央近圓心處，支撐大型葉於莖頂輪生或互生排列","表面亮綠，背面較淡；主脈掌狀自中心放射","頂生總狀或圓錐狀花序，單性同株，雄花在上、雌花在下，花部不顯眼，綠黃色"]},"distribution":{"altitude_range":[0,1200],"climate_zones":["subtropical","tropical","temperate"]},"total_weight":73,"photo_observable_weight":65},{"id":"triadica_sebifera","scientific_name":"Triadica sebifera","common_names":{"zh-TW":"烏桕","en":"Chinese Tallow Tree"},"category":"dangerous","danger_level":"medium","features":{"overall":{"growth_form":{"value":"tree","weight":3},"height_estimate":{"value":">5m","weight":3},"latex":{"value":"yes_white","weight":2},"smell":{"value":"none","weight":1},"habitat":{"value":"urban","weight":3},"water_droplet_test":{"value":"flat","weight":1}},"leaf":{"leaf_type":{"value":"simple","weight":1},"shape":{"value":"rhombic","weight":6},"edge":{"value":"entire","weight":1},"tip":{"value":"acuminate","weight":1},"base":{"value":"cuneate","weight":1},"colors":{"value":["dark_green","red_underside"],"weight":5},"color_pattern":{"value":"bicolor","weight":3},"surface_top":{"value":"matte","weight":1},"surface_bottom":{"value":"matte","weight":1},"arrangement":{"value":"alternate","weight":1},"size":{"value":"medium_5-15cm","weight":1},"venation":{"value":"pinnate","weight":1},"texture":{"value":"leathery","weight":2},"petiole_attach":{"value":"normal","weight":1}},"stem":{"type":{"value":"erect","weight":1},"cross_section":{"value":"round","weight":1},"surface":{"value":"bark_rough","weight":5},"colors":{"value":["green"],"weight":1},"interior":{"value":"solid","weight":1},"has_thorns":{"value":"no","weight":1}},"flower":{"colors":{"value":["yellow","green"],"weight":2},"color_pattern":{"value":"solid","weight":1},"petal_count":{"value":"5","weight":1},"symmetry":{"value":"radial","weight":1},"size":{"value":"medium_15-30mm","weight":1},"arrangement":{"value":"spike","weight":3},"position":{"value":"terminal","weight":1},"orientation":{"value":"upright","weight":1},"special_shape":{"value":"none","weight":1},"fragrant":{"value":"no","weight":1}},"fruit":{"type":{"value":"capsule","weight":1},"colors":{"value":["white_waxy"],"weight":5},"size":{"value":"small_5-15mm","weight":1},"surface":{"value":"smooth","weight":1}},"root":{"type":{"value":"fibrous","weight":1}}},"human_readable":{"toxicity":"樹皮、葉柄、乳汁等含刺激性成分；汁液或乳汁接觸皮膚可致紅腫、水泡及過敏性皮膚炎。誤食部位可致腸胃不適。","symptoms":["皮膚紅腫","搔癢","水泡","腸胃不適（誤食）"],"first_aid":"皮膚接觸立即肥皂水徹底清洗。勿抓破水泡。腫痛明顯或擴大就醫。誤食少量亦建議諮詢毒物科或急診。","affected_parts":["乳汁／汁液","種子外蠟層","葉及樹皮"],"diagnostic_features":["落葉喬木，高 8-15m 常見，冠幅開展，老樹樹皮灰褐縱裂","單葉互生，菱形至菱狀卵形，長 3-8cm，寬 3-9cm，先端尾狀尖，全緣；秋季葉色常轉紅或橙紅，為重要景觀辨識季相","幼葉可紅色，成葉薄革質，表面深綠，背面稍淡，乾燥時略呈紙質易透光","葉柄頂端近葉片基部處常具兩個小型腺體（關鍵特徵），腺體黃綠色小突起，觀察角度需逆光或近看","穗狀花序，雌雄同株，雄花序在上，雌花序在下，花黃綠色，風媒"]},"distribution":{"altitude_range":[0,1000],"climate_zones":["subtropical","tropical","temperate"]},"total_weight":71,"photo_observable_weight":64},{"id":"epipremnum_aureum","scientific_name":"Epipremnum aureum","common_names":{"zh-TW":"黃金葛","en":"Golden Pothos"},"category":"dangerous","danger_level":"low","features":{"overall":{"growth_form":{"value":"vine","weight":4},"height_estimate":{"value":"30-100cm","weight":1},"latex":{"value":"none","weight":1},"smell":{"value":"none","weight":1},"habitat":{"value":"epiphytic","weight":5},"water_droplet_test":{"value":"flat","weight":1}},"leaf":{"leaf_type":{"value":"simple","weight":1},"shape":{"value":"heart","weight":3},"edge":{"value":"entire","weight":1},"tip":{"value":"acuminate","weight":1},"base":{"value":"cordate","weight":3},"colors":{"value":["dark_green","variegated"],"weight":6},"color_pattern":{"value":"bicolor","weight":3},"surface_top":{"value":"glossy","weight":2},"surface_bottom":{"value":"matte","weight":1},"arrangement":{"value":"alternate","weight":1},"size":{"value":"medium_5-15cm","weight":1},"venation":{"value":"pinnate","weight":1},"texture":{"value":"leathery","weight":2},"petiole_attach":{"value":"normal","weight":1}},"stem":{"type":{"value":"climbing","weight":6},"cross_section":{"value":"round","weight":1},"surface":{"value":"smooth","weight":1},"colors":{"value":["green"],"weight":1},"interior":{"value":"solid","weight":1},"has_thorns":{"value":"no","weight":1}},"flower":null,"fruit":{"type":{"value":"capsule","weight":1},"colors":{"value":["green"],"weight":1},"size":{"value":"small_5-15mm","weight":1},"surface":{"value":"smooth","weight":1}},"root":{"type":{"value":"aerial_root","weight":6}}},"human_readable":{"toxicity":"含草酸鈣針晶；咬食可致口腔刺痛、流涎、嘔吐；汁液接觸眼睛或傷口會劇烈刺激。居家與公園常見，兒童與寵物誤食風險需注意。","symptoms":["口腔灼熱","流涎","噁心嘔吐","皮膚刺激","眼部劇痛（汁液誤入）"],"first_aid":"清水漱口；眼睛以生理食鹽水或清水沖洗至少 15 分鐘並就醫。皮膚沖洗乾淨。多數症狀可漸緩，但腫脹影響呼吸需急診。","affected_parts":["葉","莖","汁液"],"diagnostic_features":["常綠攀緣藤本，幼株蔓生或垂吊，成株節上可見發達氣生根附著樹幹或牆面","幼葉心形全緣，成株葉可變大型裂葉；一般觀賞型以心形葉為主，長 5-15cm 視齡期變化","革質光澤，深綠底色，表面有黃色至乳黃色縱向斑紋或塊斑（園藝品系眾多，斑紋密度不一）","兩基出脈自心形基部伸向葉尖兩側，側脈弧曲","莖節間明顯，蔓莖圓柱，常綠，營養充足時節間縮短呈密葉"]},"distribution":{"altitude_range":[0,500],"climate_zones":["subtropical","tropical"]},"total_weight":61,"photo_observable_weight":51},{"id":"digitalis_purpurea","scientific_name":"Digitalis purpurea","common_names":{"zh-TW":"毛地黃","en":"Common Foxglove"},"category":"dangerous","danger_level":"critical","features":{"overall":{"growth_form":{"value":"herb","weight":1},"height_estimate":{"value":"1-2m","weight":2},"latex":{"value":"none","weight":1},"smell":{"value":"none","weight":1},"habitat":{"value":"forest_floor","weight":2},"water_droplet_test":{"value":"flat","weight":1}},"leaf":{"leaf_type":{"value":"simple","weight":1},"shape":{"value":"oval","weight":1},"edge":{"value":"crenate","weight":5},"tip":{"value":"acuminate","weight":1},"base":{"value":"cuneate","weight":1},"colors":{"value":["green"],"weight":1},"color_pattern":{"value":"solid","weight":1},"surface_top":{"value":"hairy","weight":3},"surface_bottom":{"value":"hairy","weight":4},"arrangement":{"value":"basal_rosette","weight":5},"size":{"value":"large_15-50cm","weight":2},"venation":{"value":"pinnate","weight":1},"texture":{"value":"papery","weight":1},"petiole_attach":{"value":"normal","weight":1}},"stem":{"type":{"value":"erect","weight":1},"cross_section":{"value":"round","weight":1},"surface":{"value":"hairy","weight":4},"colors":{"value":["green"],"weight":1},"interior":{"value":"solid","weight":1},"has_thorns":{"value":"no","weight":1}},"flower":{"colors":{"value":["red","purple"],"weight":3},"color_pattern":{"value":"solid","weight":1},"petal_count":{"value":"many","weight":4},"symmetry":{"value":"radial","weight":1},"size":{"value":"medium_15-30mm","weight":1},"arrangement":{"value":"raceme","weight":4},"position":{"value":"terminal","weight":1},"orientation":{"value":"upright","weight":1},"special_shape":{"value":"bell_tubular","weight":5},"fragrant":{"value":"no","weight":1}},"fruit":{"type":{"value":"capsule","weight":1},"colors":{"value":["green"],"weight":1},"size":{"value":"small_5-15mm","weight":1},"surface":{"value":"smooth","weight":1}},"root":{"type":{"value":"fibrous","weight":1}}},"human_readable":{"toxicity":"全株劇毒，富含強心苷（cardiac glycosides）。誤食可致噁心嘔吐、視覺異常（黃視、綠視）、心律不整，嚴重心室頻脈或緩脈皆可致命；與體質及劑量相關。","symptoms":["噁心嘔吐","腹痛","腹瀉","頭暈","視力異常（色覺改變）","心率不整","暈厥"],"first_aid":"立即送醫監測心電圖；勿自行服用解藥。保留植株。若意識清醒可依指示處理，但強心苷中毒以醫院處置為主。避免咖啡與刺激物。","affected_parts":["全株","花","葉","種子"],"diagnostic_features":["二年生草本，第一年常只生蓮座葉，第二年抽薹，高 60-150cm","基生葉叢生，長橢圓至卵形，葉基漸狹成柄，葉面密被短絨毛，觸感粗糙灰綠色","莖生葉互生，向上漸小，葉柄短或無柄，葉片仍密毛","頂生總狀花序，花朵數量多，由下方依序向上開展","花冠鐘形或筒狀鐘形，略下垂；花色常紫紅色，冠筒內側有明顯深色斑點或網紋（關鍵園藝與野生辨識特徵）；花萼裂片短"]},"distribution":{"altitude_range":[500,2500],"climate_zones":["subtropical","temperate"]},"total_weight":72,"photo_observable_weight":66},{"id":"diplazium_esculentum","scientific_name":"Diplazium esculentum","common_names":{"zh-TW":"過貓（過溝菜蕨）","en":"Vegetable Fern"},"category":"edible","features":{"overall":{"growth_form":{"value":"fern","weight":3},"height_estimate":{"value":"1-2m","weight":2},"latex":{"value":"none","weight":1},"smell":{"value":"none","weight":1},"habitat":{"value":"streamside","weight":3},"water_droplet_test":{"value":"flat","weight":1}},"leaf":{"leaf_type":{"value":"bipinnate_compound","weight":3},"shape":{"value":"lanceolate","weight":2},"edge":{"value":"entire","weight":1},"tip":{"value":"acuminate","weight":1},"base":{"value":"cuneate","weight":1},"colors":{"value":["green","purple"],"weight":4},"color_pattern":{"value":"bicolor","weight":3},"surface_top":{"value":"hairy","weight":3},"surface_bottom":{"value":"matte","weight":1},"arrangement":{"value":"clustered","weight":2},"size":{"value":"medium_5-15cm","weight":1},"venation":{"value":"pinnate","weight":1},"texture":{"value":"papery","weight":1},"petiole_attach":{"value":"normal","weight":1}},"stem":{"type":{"value":"erect","weight":1},"cross_section":{"value":"round","weight":1},"surface":{"value":"scaly","weight":3},"colors":{"value":["brown","green"],"weight":2},"interior":{"value":"solid","weight":1},"has_thorns":{"value":"no","weight":1}},"flower":null,"fruit":{"type":{"value":"spore","weight":2},"colors":{"value":["brown"],"weight":2},"size":{"value":"tiny_<5mm","weight":2},"surface":{"value":"smooth","weight":1}},"root":{"type":{"value":"fibrous","weight":1}}},"usage":{"type":["edible_leaf","edible_root"],"edible_parts":["嫩莖","幼葉（捲曲的拳頭狀嫩芽）"],"preparation":"川燙或快炒，務必煮熟","season":"全年可採，春夏最嫩","warnings":"避免採食不認識的蕨類。部分蕨類含致癌物質，過貓需煮熟食用。"},"human_readable":{"description_zh":"避免採食不認識的蕨類。部分蕨類含致癌物質，過貓需煮熟食用。","diagnostic_features":["二回羽狀複葉，成熟株高可達 1-2m","頂端捲曲呈拳頭狀（蕨類典型特徵）——可食部位","嫩葉綠色帶微紅，有細毛","莖部黑褐色，有鱗片"]},"distribution":{"altitude_range":[0,1000],"climate_zones":["subtropical","tropical"]},"total_weight":53,"photo_observable_weight":48},{"id":"hedychium_coronarium","scientific_name":"Hedychium coronarium","common_names":{"zh-TW":"野薑花","en":"White Ginger Lily"},"category":"medicinal","features":{"overall":{"growth_form":{"value":"herb","weight":1},"height_estimate":{"value":"1-2m","weight":2},"latex":{"value":"none","weight":1},"smell":{"value":"aromatic","weight":2},"habitat":{"value":"streamside","weight":3},"water_droplet_test":{"value":"flat","weight":1}},"leaf":{"leaf_type":{"value":"simple","weight":1},"shape":{"value":"elliptic","weight":4},"edge":{"value":"entire","weight":1},"tip":{"value":"acuminate","weight":1},"base":{"value":"cuneate","weight":1},"colors":{"value":["green"],"weight":1},"color_pattern":{"value":"solid","weight":1},"surface_top":{"value":"glossy","weight":2},"surface_bottom":{"value":"hairy","weight":4},"arrangement":{"value":"alternate","weight":1},"size":{"value":"large_15-50cm","weight":2},"venation":{"value":"pinnate","weight":1},"texture":{"value":"leathery","weight":2},"petiole_attach":{"value":"normal","weight":1}},"stem":{"type":{"value":"erect","weight":1},"cross_section":{"value":"round","weight":1},"surface":{"value":"smooth","weight":1},"colors":{"value":["green"],"weight":1},"interior":{"value":"solid","weight":1},"has_thorns":{"value":"no","weight":1}},"flower":{"colors":{"value":["white"],"weight":2},"color_pattern":{"value":"solid","weight":1},"petal_count":{"value":"5","weight":1},"symmetry":{"value":"radial","weight":1},"size":{"value":"medium_15-30mm","weight":1},"arrangement":{"value":"solitary","weight":3},"position":{"value":"terminal","weight":1},"orientation":{"value":"upright","weight":1},"special_shape":{"value":"butterfly_shape","weight":5},"fragrant":{"value":"yes","weight":3}},"fruit":{"type":{"value":"capsule","weight":1},"colors":{"value":["green"],"weight":1},"size":{"value":"small_5-15mm","weight":1},"surface":{"value":"smooth","weight":1}},"root":{"type":{"value":"rhizome","weight":3}}},"usage":{"type":["edible_root"],"edible_parts":["花瓣","嫩莖","根莖（調味用）"],"preparation":"花瓣可直接食用或入菜。嫩莖可炒食。根莖可當薑使用。","season":"花期 6-10 月","warnings":"確認花朵為純白蝴蝶形且有濃郁香氣。勿與其他薑科植物混淆。","medicinal_effects":["根莖：去寒、消腫"]},"human_readable":{"description_zh":"確認花朵為純白蝴蝶形且有濃郁香氣。勿與其他薑科植物混淆。","diagnostic_features":["長橢圓形披針狀，長 20-40cm，互生排列","綠色，光滑，葉背微被毛","白色蝴蝶形花朵，香氣濃郁（關鍵辨識特徵）","直立草本，高 100-200cm","根莖塊狀，有薑味"]},"distribution":{"altitude_range":[0,800],"climate_zones":["subtropical","tropical"]},"total_weight":65,"photo_observable_weight":54},{"id":"solanum_nigrum","scientific_name":"Solanum nigrum","common_names":{"zh-TW":"龍葵（黑籽仔菜）","en":"Black Nightshade"},"category":"edible","features":{"overall":{"growth_form":{"value":"herb","weight":1},"height_estimate":{"value":"30-100cm","weight":1},"latex":{"value":"none","weight":1},"smell":{"value":"none","weight":1},"habitat":{"value":"roadside","weight":2},"water_droplet_test":{"value":"flat","weight":1}},"leaf":{"leaf_type":{"value":"simple","weight":1},"shape":{"value":"oval","weight":1},"edge":{"value":"entire","weight":1},"tip":{"value":"acuminate","weight":1},"base":{"value":"cuneate","weight":1},"colors":{"value":["green"],"weight":1},"color_pattern":{"value":"solid","weight":1},"surface_top":{"value":"matte","weight":1},"surface_bottom":{"value":"matte","weight":1},"arrangement":{"value":"alternate","weight":1},"size":{"value":"medium_5-15cm","weight":1},"venation":{"value":"pinnate","weight":1},"texture":{"value":"papery","weight":1},"petiole_attach":{"value":"normal","weight":1}},"stem":{"type":{"value":"erect","weight":1},"cross_section":{"value":"round","weight":1},"surface":{"value":"smooth","weight":1},"colors":{"value":["green"],"weight":1},"interior":{"value":"solid","weight":1},"has_thorns":{"value":"no","weight":1}},"flower":{"colors":{"value":["white","yellow"],"weight":2},"color_pattern":{"value":"solid","weight":1},"petal_count":{"value":"5","weight":1},"symmetry":{"value":"radial","weight":1},"size":{"value":"medium_15-30mm","weight":1},"arrangement":{"value":"cyme","weight":4},"position":{"value":"terminal","weight":1},"orientation":{"value":"upright","weight":1},"special_shape":{"value":"none","weight":1},"fragrant":{"value":"no","weight":1}},"fruit":{"type":{"value":"berry","weight":3},"colors":{"value":["green","purple"],"weight":3},"size":{"value":"small_5-15mm","weight":1},"surface":{"value":"smooth","weight":1}},"root":{"type":{"value":"fibrous","weight":1}}},"usage":{"type":["edible_leaf","edible_fruit","edible_root"],"edible_parts":["嫩莖葉","成熟果實（全黑時）"],"preparation":"嫩莖葉川燙或炒食。成熟黑果可直接食用。","season":"全年可採","warnings":"⚠️ 未成熟的綠色果實含龍葵鹼（solanine），有毒！只能食用完全成熟的黑色果實。嫩莖葉需煮熟。"},"human_readable":{"description_zh":"⚠️ 未成熟的綠色果實含龍葵鹼（solanine），有毒！只能食用完全成熟的黑色果實。嫩莖葉需煮熟。","diagnostic_features":["卵形至橢圓形，長 3-10cm，葉緣全緣或微波狀","綠色，薄紙質","白色小花，5 瓣，花心黃色，2-8 朵成聚繖花序","球形漿果，直徑 6-8mm，未熟時綠色，成熟後全黑（關鍵）","草本，高 30-100cm，莖多分枝"]},"distribution":{"altitude_range":[0,1500],"climate_zones":["subtropical","temperate","tropical"]},"total_weight":50,"photo_observable_weight":44},{"id":"crassocephalum_crepidioides","scientific_name":"Crassocephalum crepidioides","common_names":{"zh-TW":"昭和草（飛機草）","en":"Redflower Ragleaf"},"category":"edible","features":{"overall":{"growth_form":{"value":"herb","weight":1},"height_estimate":{"value":"30-100cm","weight":1},"latex":{"value":"yes_white","weight":2},"smell":{"value":"none","weight":1},"habitat":{"value":"roadside","weight":2},"water_droplet_test":{"value":"flat","weight":1}},"leaf":{"leaf_type":{"value":"simple","weight":1},"shape":{"value":"elliptic","weight":4},"edge":{"value":"serrated","weight":2},"tip":{"value":"acuminate","weight":1},"base":{"value":"cuneate","weight":1},"colors":{"value":["green","purple"],"weight":4},"color_pattern":{"value":"bicolor","weight":3},"surface_top":{"value":"matte","weight":1},"surface_bottom":{"value":"matte","weight":1},"arrangement":{"value":"alternate","weight":1},"size":{"value":"medium_5-15cm","weight":1},"venation":{"value":"pinnate","weight":1},"texture":{"value":"papery","weight":1},"petiole_attach":{"value":"normal","weight":1}},"stem":{"type":{"value":"erect","weight":1},"cross_section":{"value":"round","weight":1},"surface":{"value":"smooth","weight":1},"colors":{"value":["green"],"weight":1},"interior":{"value":"hollow","weight":3},"has_thorns":{"value":"no","weight":1}},"flower":{"colors":{"value":["red"],"weight":3},"color_pattern":{"value":"solid","weight":1},"petal_count":{"value":"5","weight":1},"symmetry":{"value":"radial","weight":1},"size":{"value":"medium_15-30mm","weight":1},"arrangement":{"value":"head_composite","weight":3},"position":{"value":"terminal","weight":1},"orientation":{"value":"upright","weight":1},"special_shape":{"value":"none","weight":1},"fragrant":{"value":"no","weight":1}},"fruit":{"type":{"value":"achene","weight":3},"colors":{"value":["white_waxy"],"weight":5},"size":{"value":"small_5-15mm","weight":1},"surface":{"value":"smooth","weight":1}},"root":{"type":{"value":"fibrous","weight":1}}},"usage":{"type":["edible_leaf","edible_root"],"edible_parts":["嫩莖葉"],"preparation":"川燙後涼拌、炒食、煮湯皆可。","season":"全年可採，冬春最佳","warnings":"辨識重點是橘紅色下垂花序和斷莖白色乳汁。避免與其他菊科植物混淆。"},"human_readable":{"description_zh":"辨識重點是橘紅色下垂花序和斷莖白色乳汁。避免與其他菊科植物混淆。","diagnostic_features":["長橢圓形，長 5-15cm，葉緣有不規則鋸齒或裂片","綠色，略帶紫暈，柔軟","頭狀花序，下垂，花冠橘紅色（關鍵特徵）","直立草本，高 20-80cm，莖中空，斷面有白色乳汁","瘦果帶白色冠毛（似蒲公英），可隨風飄散"]},"distribution":{"altitude_range":[0,2000],"climate_zones":["subtropical","tropical"]},"total_weight":64,"photo_observable_weight":55},{"id":"auricularia_auricula_judae","scientific_name":"Auricularia auricula-judae","common_names":{"zh-TW":"木耳","en":"Wood Ear Mushroom"},"category":"edible","features":{"overall":{"growth_form":{"value":"fungus","weight":3},"height_estimate":{"value":"<30cm","weight":3},"latex":{"value":"none","weight":1},"smell":{"value":"none","weight":1},"habitat":{"value":"forest_floor","weight":2},"water_droplet_test":{"value":"flat","weight":1}},"leaf":{"leaf_type":{"value":"simple","weight":1},"shape":{"value":"oval","weight":1},"edge":{"value":"entire","weight":1},"tip":{"value":"rounded","weight":3},"base":{"value":"rounded","weight":3},"colors":{"value":["brown","dark_green"],"weight":4},"color_pattern":{"value":"solid","weight":1},"surface_top":{"value":"hairy","weight":3},"surface_bottom":{"value":"glossy","weight":6},"arrangement":{"value":"clustered","weight":2},"size":{"value":"medium_5-15cm","weight":1},"venation":{"value":"reticulate","weight":3},"texture":{"value":"fleshy","weight":3},"petiole_attach":{"value":"normal","weight":1}},"stem":{"type":{"value":"erect","weight":1},"cross_section":{"value":"round","weight":1},"surface":{"value":"hairy","weight":4},"colors":{"value":["brown"],"weight":2},"interior":{"value":"solid","weight":1},"has_thorns":{"value":"no","weight":1}},"flower":null,"fruit":{"type":{"value":"spore","weight":2},"colors":{"value":["brown"],"weight":2},"size":{"value":"tiny_<5mm","weight":2},"surface":{"value":"smooth","weight":1}},"root":{"type":{"value":"fibrous","weight":1}}},"usage":{"type":["edible_fruit"],"edible_parts":["子實體（整朵）"],"preparation":"川燙或炒食。新鮮木耳可直接料理。","season":"雨後最多，春秋最佳","warnings":"確認生長在枯木上（而非土壤）。避免採集已腐爛、發臭、或顏色異常的菇體。不確定時禁止食用。"},"human_readable":{"description_zh":"確認生長在枯木上（而非土壤）。避免採集已腐爛、發臭、或顏色異常的菇體。不確定時禁止食用。","diagnostic_features":["耳狀或碗狀，直徑 3-10cm","外面有細微絨毛，灰褐色。內面光滑，深褐色至黑色。","新鮮時膠質狀、半透明、有彈性。乾燥後硬縮，泡水恢復。","叢生於枯木或腐木表面（關鍵特徵）","深褐色至黑色"]},"distribution":{"altitude_range":[0,2000],"climate_zones":["subtropical","temperate","tropical"]},"total_weight":62,"photo_observable_weight":57},{"id":"nephrolepis_cordifolia","scientific_name":"Nephrolepis cordifolia","common_names":{"zh-TW":"腎蕨（山鳳梨）","en":"Tuberous Sword Fern"},"category":"edible","features":{"overall":{"growth_form":{"value":"fern","weight":3},"height_estimate":{"value":"30-100cm","weight":1},"latex":{"value":"none","weight":1},"smell":{"value":"none","weight":1},"habitat":{"value":"forest_floor","weight":2},"water_droplet_test":{"value":"flat","weight":1}},"leaf":{"leaf_type":{"value":"pinnate_compound","weight":3},"shape":{"value":"lanceolate","weight":2},"edge":{"value":"entire","weight":1},"tip":{"value":"acuminate","weight":1},"base":{"value":"cuneate","weight":1},"colors":{"value":["green"],"weight":1},"color_pattern":{"value":"solid","weight":1},"surface_top":{"value":"matte","weight":1},"surface_bottom":{"value":"matte","weight":1},"arrangement":{"value":"clustered","weight":2},"size":{"value":"very_large_>50cm","weight":3},"venation":{"value":"pinnate","weight":1},"texture":{"value":"papery","weight":1},"petiole_attach":{"value":"normal","weight":1}},"stem":{"type":{"value":"rhizome","weight":5},"cross_section":{"value":"round","weight":1},"surface":{"value":"scaly","weight":3},"colors":{"value":["brown"],"weight":2},"interior":{"value":"solid","weight":1},"has_thorns":{"value":"no","weight":1}},"flower":null,"fruit":{"type":{"value":"spore","weight":2},"colors":{"value":["brown"],"weight":2},"size":{"value":"tiny_<5mm","weight":2},"surface":{"value":"smooth","weight":1}},"root":{"type":{"value":"storage_tuber","weight":4}}},"usage":{"type":["edible_root"],"edible_parts":["球形儲水塊莖（地下）"],"preparation":"剝皮後可直接生食或煮食。塊莖含水分，可解渴。","season":"全年可採","warnings":"確認是腎蕨（一回羽狀複葉 + 地下球形塊莖）。其他蕨類無此塊莖。"},"human_readable":{"description_zh":"確認是腎蕨（一回羽狀複葉 + 地下球形塊莖）。其他蕨類無此塊莖。","diagnostic_features":["一回羽狀複葉，長 30-80cm，小葉互生，長 2-4cm","叢生，向外弧形展開","綠色，薄草質","根部有球形儲水塊莖，直徑 1-2cm（關鍵特徵，含水分）","地生或附生，常見於路旁岩壁"]},"distribution":{"altitude_range":[0,1500],"climate_zones":["subtropical","tropical","temperate"]},"total_weight":53,"photo_observable_weight":45},{"id":"oenanthe_javanica","scientific_name":"Oenanthe javanica","common_names":{"zh-TW":"山芹菜（水芹菜）","en":"Water Dropwort"},"category":"edible","features":{"overall":{"growth_form":{"value":"aquatic","weight":6},"height_estimate":{"value":"30-100cm","weight":1},"latex":{"value":"none","weight":1},"smell":{"value":"aromatic","weight":2},"habitat":{"value":"streamside","weight":3},"water_droplet_test":{"value":"flat","weight":1}},"leaf":{"leaf_type":{"value":"bipinnate_compound","weight":3},"shape":{"value":"oval","weight":1},"edge":{"value":"serrated","weight":2},"tip":{"value":"acuminate","weight":1},"base":{"value":"cuneate","weight":1},"colors":{"value":["green"],"weight":1},"color_pattern":{"value":"solid","weight":1},"surface_top":{"value":"glossy","weight":2},"surface_bottom":{"value":"matte","weight":1},"arrangement":{"value":"alternate","weight":1},"size":{"value":"small_2-5cm","weight":4},"venation":{"value":"pinnate","weight":1},"texture":{"value":"papery","weight":1},"petiole_attach":{"value":"normal","weight":1}},"stem":{"type":{"value":"creeping","weight":4},"cross_section":{"value":"round","weight":1},"surface":{"value":"smooth","weight":1},"colors":{"value":["green"],"weight":1},"interior":{"value":"hollow","weight":3},"has_thorns":{"value":"no","weight":1}},"flower":{"colors":{"value":["white"],"weight":2},"color_pattern":{"value":"solid","weight":1},"petal_count":{"value":"5","weight":1},"symmetry":{"value":"radial","weight":1},"size":{"value":"medium_15-30mm","weight":1},"arrangement":{"value":"umbel","weight":4},"position":{"value":"terminal","weight":1},"orientation":{"value":"upright","weight":1},"special_shape":{"value":"none","weight":1},"fragrant":{"value":"no","weight":1}},"fruit":{"type":{"value":"capsule","weight":1},"colors":{"value":["green"],"weight":1},"size":{"value":"small_5-15mm","weight":1},"surface":{"value":"smooth","weight":1}},"root":{"type":{"value":"fibrous","weight":1}}},"usage":{"type":["edible_leaf","edible_root"],"edible_parts":["嫩莖葉"],"preparation":"川燙或快炒。香味濃郁，適合炒肉絲或煮湯。","season":"全年可採，春季最嫩","warnings":"⚠️ 注意！同屬的毒水芹（Oenanthe crocata）劇毒。確認有芹菜香味，且生長在乾淨水源旁。水質不佳的地方不建議採集（寄生蟲風險）。"},"human_readable":{"description_zh":"⚠️ 注意！同屬的毒水芹（Oenanthe crocata）劇毒。確認有芹菜香味，且生長在乾淨水源旁。水質不佳的地方不建議採集（寄生蟲風險）。","diagnostic_features":["二至三回羽狀複葉，小葉卵形至菱形，長 1-3cm，葉緣鋸齒狀","複繖形花序，白色小花","中空，光滑，有節，匍匐莖可水生","有獨特的芹菜香味（關鍵辨識特徵）"]},"distribution":{"altitude_range":[0,1500],"climate_zones":["subtropical","temperate"]},"total_weight":65,"photo_observable_weight":56},{"id":"bidens_pilosa_var_radiata","scientific_name":"Bidens pilosa var. radiata","common_names":{"zh-TW":"大花咸豐草（鬼針草）","en":"Spanish Needle (Radiate Variety)"},"category":"edible","features":{"overall":{"growth_form":{"value":"herb","weight":1},"height_estimate":{"value":"30-100cm","weight":1},"latex":{"value":"none","weight":1},"smell":{"value":"none","weight":1},"habitat":{"value":"roadside","weight":2},"water_droplet_test":{"value":"flat","weight":1}},"leaf":{"leaf_type":{"value":"trifoliate","weight":6},"shape":{"value":"oval","weight":1},"edge":{"value":"serrated","weight":2},"tip":{"value":"acuminate","weight":1},"base":{"value":"cuneate","weight":1},"colors":{"value":["green"],"weight":1},"color_pattern":{"value":"solid","weight":1},"surface_top":{"value":"rough","weight":4},"surface_bottom":{"value":"matte","weight":1},"arrangement":{"value":"opposite","weight":4},"size":{"value":"medium_5-15cm","weight":1},"venation":{"value":"pinnate","weight":1},"texture":{"value":"papery","weight":1},"petiole_attach":{"value":"normal","weight":1}},"stem":{"type":{"value":"erect","weight":1},"cross_section":{"value":"square","weight":4},"surface":{"value":"smooth","weight":1},"colors":{"value":["green"],"weight":1},"interior":{"value":"solid","weight":1},"has_thorns":{"value":"no","weight":1}},"flower":{"colors":{"value":["white","yellow"],"weight":2},"color_pattern":{"value":"solid","weight":1},"petal_count":{"value":"5","weight":1},"symmetry":{"value":"radial","weight":1},"size":{"value":"medium_15-30mm","weight":1},"arrangement":{"value":"head_composite","weight":3},"position":{"value":"terminal","weight":1},"orientation":{"value":"upright","weight":1},"special_shape":{"value":"none","weight":1},"fragrant":{"value":"no","weight":1}},"fruit":{"type":{"value":"achene","weight":3},"colors":{"value":["green"],"weight":1},"size":{"value":"small_5-15mm","weight":1},"surface":{"value":"smooth","weight":1}},"root":{"type":{"value":"fibrous","weight":1}}},"usage":{"type":["edible_leaf","edible_root"],"edible_parts":["嫩莖葉","花（泡茶）"],"preparation":"嫩莖葉川燙後涼拌、快炒或煮湯；乾燥花朵可沖泡花草茶","season":"全年可採，春夏秋嫩葉品質最佳","warnings":"辨識舌狀花與鉤刺瘦果即可與多數菊科野菜區分。腸胃敏感者不宜大量生食。"},"human_readable":{"description_zh":"辨識舌狀花與鉤刺瘦果即可與多數菊科野菜區分。腸胃敏感者不宜大量生食。","diagnostic_features":["一年生草本，株高 30-100cm，分枝多而蔓延或直立","莖具四稜或近似方形、有明顯稜線（便於手觸辨識）","葉對生，三出複葉或羽狀複葉，小葉卵形、葉緣具鋸齒，質地薄而稍粗糙","頭狀花序頂生或腋生；中央為黃色管狀花，周圍具 5-8 枚白色舌狀花（關鍵特徵，與無舌狀花變種區別）","瘦果具 2-3 枚倒鉤刺狀芒，極易黏附衣物與動物毛髮（關鍵特徵）"]},"distribution":{"altitude_range":[0,2500],"climate_zones":["subtropical","tropical"]},"total_weight":62,"photo_observable_weight":56},{"id":"alpinia_zerumbet","scientific_name":"Alpinia zerumbet","common_names":{"zh-TW":"月桃","en":"Shell Ginger"},"category":"edible","features":{"overall":{"growth_form":{"value":"herb","weight":1},"height_estimate":{"value":"1-2m","weight":2},"latex":{"value":"none","weight":1},"smell":{"value":"aromatic","weight":2},"habitat":{"value":"streamside","weight":3},"water_droplet_test":{"value":"flat","weight":1}},"leaf":{"leaf_type":{"value":"simple","weight":1},"shape":{"value":"elliptic","weight":4},"edge":{"value":"entire","weight":1},"tip":{"value":"acuminate","weight":1},"base":{"value":"cuneate","weight":1},"colors":{"value":["green"],"weight":1},"color_pattern":{"value":"solid","weight":1},"surface_top":{"value":"glossy","weight":2},"surface_bottom":{"value":"matte","weight":1},"arrangement":{"value":"alternate","weight":1},"size":{"value":"very_large_>50cm","weight":3},"venation":{"value":"parallel","weight":4},"texture":{"value":"leathery","weight":2},"petiole_attach":{"value":"normal","weight":1}},"stem":{"type":{"value":"erect","weight":1},"cross_section":{"value":"round","weight":1},"surface":{"value":"smooth","weight":1},"colors":{"value":["green"],"weight":1},"interior":{"value":"solid","weight":1},"has_thorns":{"value":"no","weight":1}},"flower":{"colors":{"value":["white","yellow","red","pink","purple"],"weight":3},"color_pattern":{"value":"solid","weight":1},"petal_count":{"value":"5","weight":1},"symmetry":{"value":"radial","weight":1},"size":{"value":"medium_15-30mm","weight":1},"arrangement":{"value":"panicle","weight":4},"position":{"value":"terminal","weight":1},"orientation":{"value":"upright","weight":1},"special_shape":{"value":"lip_labellum","weight":6},"fragrant":{"value":"yes","weight":3}},"fruit":{"type":{"value":"capsule","weight":1},"colors":{"value":["green"],"weight":1},"size":{"value":"small_5-15mm","weight":1},"surface":{"value":"smooth","weight":1}},"root":{"type":{"value":"fibrous","weight":1}}},"usage":{"type":["edible_leaf"],"edible_parts":["嫩葉（包粽、裹食）","花","種子（傳統加工原料）"],"preparation":"嫩葉煮軟後可作「月桃粽」外皮；花可汆燙入菜或甜湯；種子僅建議使用市售合法加工品","season":"葉全年可用，盛花期多集中在春夏，果熟期視海拔略異","warnings":"與觀賞朱蕉等不同科屬植物葉形偶似，務必以搓葉香氣與花序結構複核。"},"human_readable":{"description_zh":"與觀賞朱蕉等不同科屬植物葉形偶似，務必以搓葉香氣與花序結構複核。","diagnostic_features":["多年生大型草本，株高 1-3m，具地下匍匐莖與明顯地上假莖","葉互生，長橢圓狀披針形，長 40-70cm、寬 8-15cm，具平行脈、葉質厚而亮綠","葉片搓揉或撕裂後散發濃郁芳香（關鍵辨識特徵），近似薑花與肉桂混合調性","圓錐花序自葉腋抽出並下垂，披覆蠟質苞片；花白色至淡粉紅，唇瓣近黃底色並常見紅色或紫紅條紋","球形蒴果，具縱稜，成熟時果皮纖維質；內含種子可供傳統「仁丹」類製品原料"]},"distribution":{"altitude_range":[0,800],"climate_zones":["subtropical","tropical"]},"total_weight":67,"photo_observable_weight":58},{"id":"broussonetia_papyrifera","scientific_name":"Broussonetia papyrifera","common_names":{"zh-TW":"構樹（構桃）","en":"Paper Mulberry"},"category":"edible","features":{"overall":{"growth_form":{"value":"tree","weight":3},"height_estimate":{"value":">5m","weight":3},"latex":{"value":"yes_white","weight":2},"smell":{"value":"none","weight":1},"habitat":{"value":"roadside","weight":2},"water_droplet_test":{"value":"flat","weight":1}},"leaf":{"leaf_type":{"value":"simple","weight":1},"shape":{"value":"heart","weight":3},"edge":{"value":"lobed","weight":3},"tip":{"value":"acuminate","weight":1},"base":{"value":"cuneate","weight":1},"colors":{"value":["green"],"weight":1},"color_pattern":{"value":"solid","weight":1},"surface_top":{"value":"rough","weight":4},"surface_bottom":{"value":"matte","weight":1},"arrangement":{"value":"alternate","weight":1},"size":{"value":"medium_5-15cm","weight":1},"venation":{"value":"pinnate","weight":1},"texture":{"value":"papery","weight":1},"petiole_attach":{"value":"normal","weight":1}},"stem":{"type":{"value":"erect","weight":1},"cross_section":{"value":"round","weight":1},"surface":{"value":"smooth","weight":1},"colors":{"value":["brown"],"weight":2},"interior":{"value":"solid","weight":1},"has_thorns":{"value":"no","weight":1}},"flower":null,"fruit":{"type":{"value":"aggregate","weight":5},"colors":{"value":["orange","red"],"weight":4},"size":{"value":"small_5-15mm","weight":1},"surface":{"value":"smooth","weight":1}},"root":{"type":{"value":"fibrous","weight":1}}},"usage":{"type":["edible_leaf","edible_fruit"],"edible_parts":["成熟聚合假果（俗稱構樹果）","嫩葉與嫩芽（需煮熟）"],"preparation":"果實可生食或打成果汁；嫩葉務必川燙或久煮去草酸與單寧澀味","season":"果熟多夏秋，嫩葉以春季最佳","warnings":"務必採完全著色之熟果；嫩皮層可能刺激口腔，宜煮熟。勿與桑科未確認種類之乳汁植物混淆。"},"human_readable":{"description_zh":"務必採完全著色之熟果；嫩皮層可能刺激口腔，宜煮熟。勿與桑科未確認種類之乳汁植物混淆。","diagnostic_features":["落葉喬木，高 10–20m，樹皮灰褐，樹冠開展，為先驅陽性樹種","葉互生，卵形至心形，常見 3–5 裂之掌狀裂葉，長 6–18cm；幼葉密被毛","葉面極粗糙、如砂紙觸感（關鍵特徵），葉背密被柔毛呈淡灰綠","雌雄異株；雄花序柔荑狀，雌株結球形聚合果","雌株果序圓球形，由多數小核果集生；成熟時橙紅色，表面具肉質小突起（關鍵特徵）"]},"distribution":{"altitude_range":[0,1500],"climate_zones":["subtropical","tropical","temperate"]},"total_weight":52,"photo_observable_weight":46},{"id":"morus_australis","scientific_name":"Morus australis","common_names":{"zh-TW":"小葉桑（野生桑）","en":"Chinese Mulberry / Small-leaf Mulberry"},"category":"edible","features":{"overall":{"growth_form":{"value":"tree","weight":3},"height_estimate":{"value":"2-5m","weight":3},"latex":{"value":"yes_white","weight":2},"smell":{"value":"none","weight":1},"habitat":{"value":"forest_floor","weight":2},"water_droplet_test":{"value":"flat","weight":1}},"leaf":{"leaf_type":{"value":"simple","weight":1},"shape":{"value":"oval","weight":1},"edge":{"value":"serrated","weight":2},"tip":{"value":"acuminate","weight":1},"base":{"value":"cuneate","weight":1},"colors":{"value":["dark_green"],"weight":2},"color_pattern":{"value":"solid","weight":1},"surface_top":{"value":"matte","weight":1},"surface_bottom":{"value":"matte","weight":1},"arrangement":{"value":"alternate","weight":1},"size":{"value":"medium_5-15cm","weight":1},"venation":{"value":"pinnate","weight":1},"texture":{"value":"papery","weight":1},"petiole_attach":{"value":"normal","weight":1}},"stem":{"type":{"value":"erect","weight":1},"cross_section":{"value":"round","weight":1},"surface":{"value":"smooth","weight":1},"colors":{"value":["green"],"weight":1},"interior":{"value":"solid","weight":1},"has_thorns":{"value":"no","weight":1}},"flower":null,"fruit":{"type":{"value":"aggregate","weight":5},"colors":{"value":["red","purple"],"weight":3},"size":{"value":"small_5-15mm","weight":1},"surface":{"value":"smooth","weight":1}},"root":{"type":{"value":"fibrous","weight":1}}},"usage":{"type":["edible_leaf","edible_fruit"],"edible_parts":["成熟桑椹（聚合果）","嫩葉（煮湯）"],"preparation":"桑椹可鮮食、做果醬或煮甜湯；嫩葉川燙煮蛋花湯或肉湯","season":"桑椹多春夏至初秋，嫩葉春季最佳","warnings":"與其他聚合漿果混淆時以樹形、葉形與乳汁確認。胃酸過多者少食大量酸椹。"},"human_readable":{"description_zh":"與其他聚合漿果混淆時以樹形、葉形與乳汁確認。胃酸過多者少食大量酸椹。","diagnostic_features":["落葉灌木至小喬木，高 3-10m，枝條柔韌，皮孔明顯","葉互生，卵形至廣卵形，或不規則缺刻與淺裂，長寬變異大，葉緣具鈍鋸齒","表面深綠、質地紙質至薄革質；嫩葉有毛，老葉較光滑無光澤","雌花結長圓筒形至略柱狀之聚合果（俗稱桑椹）；未熟紅色，完全成熟深紫至近黑（關鍵特徵）","幼枝斷裂或葉柄撕裂可見白色乳汁（桑科常見，輔助辨識）"]},"distribution":{"altitude_range":[0,1500],"climate_zones":["subtropical","temperate","tropical"]},"total_weight":45,"photo_observable_weight":39},{"id":"amaranthus_viridis","scientific_name":"Amaranthus viridis","common_names":{"zh-TW":"野莧菜（綠莧）","en":"Green Amaranth"},"category":"edible","features":{"overall":{"growth_form":{"value":"herb","weight":1},"height_estimate":{"value":"30-100cm","weight":1},"latex":{"value":"none","weight":1},"smell":{"value":"none","weight":1},"habitat":{"value":"roadside","weight":2},"water_droplet_test":{"value":"flat","weight":1}},"leaf":{"leaf_type":{"value":"simple","weight":1},"shape":{"value":"oval","weight":1},"edge":{"value":"entire","weight":1},"tip":{"value":"acuminate","weight":1},"base":{"value":"cuneate","weight":1},"colors":{"value":["green"],"weight":1},"color_pattern":{"value":"solid","weight":1},"surface_top":{"value":"matte","weight":1},"surface_bottom":{"value":"matte","weight":1},"arrangement":{"value":"alternate","weight":1},"size":{"value":"medium_5-15cm","weight":1},"venation":{"value":"pinnate","weight":1},"texture":{"value":"papery","weight":1},"petiole_attach":{"value":"normal","weight":1}},"stem":{"type":{"value":"erect","weight":1},"cross_section":{"value":"round","weight":1},"surface":{"value":"smooth","weight":1},"colors":{"value":["green","red"],"weight":5},"interior":{"value":"solid","weight":1},"has_thorns":{"value":"no","weight":1}},"flower":{"colors":{"value":["green"],"weight":2},"color_pattern":{"value":"solid","weight":1},"petal_count":{"value":"5","weight":1},"symmetry":{"value":"radial","weight":1},"size":{"value":"medium_15-30mm","weight":1},"arrangement":{"value":"spike","weight":3},"position":{"value":"terminal","weight":1},"orientation":{"value":"upright","weight":1},"special_shape":{"value":"none","weight":1},"fragrant":{"value":"no","weight":1}},"fruit":{"type":{"value":"capsule","weight":1},"colors":{"value":["green"],"weight":1},"size":{"value":"small_5-15mm","weight":1},"surface":{"value":"smooth","weight":1}},"root":{"type":{"value":"fibrous","weight":1}}},"usage":{"type":["edible_leaf","edible_root"],"edible_parts":["嫩莖葉"],"preparation":"嫩莖葉川燙後炒食、煮粥或煮羹；亦可切碎煎蛋","season":"全年可採，冬春溫暖時仍見幼株","warnings":"⚠️ 幼株易與美洲商陸（Phytolacca americana）等有毒植物混淆！商陸常具紫紅莖、互生大葉與下垂總狀紫黑漿果；無把握切勿採食。"},"human_readable":{"description_zh":"⚠️ 幼株易與美洲商陸（Phytolacca americana）等有毒植物混淆！商陸常具紫紅莖、互生大葉與下垂總狀紫黑漿果；無把握切勿採食。","diagnostic_features":["一年生草本，高 20-80cm，具多數分枝，全株鮮綠或莖帶淡紅","莖直立或斜上，表面溝稜明顯；節間長度中等","葉互生，卵形至菱形，長 3-8cm，先端漸尖，葉緣全緣，葉柄長","穗狀花序頂生與腋生，小花密集呈細長或圓錐狀，花被淡綠不顯眼","胞果開裂釋出大量黑色、微小而光亮種子（關鍵於成熟植株辨識）"]},"distribution":{"altitude_range":[0,2000],"climate_zones":["subtropical","tropical","temperate"]},"total_weight":49,"photo_observable_weight":43},{"id":"miscanthus_floridulus","scientific_name":"Miscanthus floridulus","common_names":{"zh-TW":"五節芒（嫩筍）","en":"Giant Chinese Silver Grass"},"category":"edible","features":{"overall":{"growth_form":{"value":"grass","weight":6},"height_estimate":{"value":"2-5m","weight":3},"latex":{"value":"none","weight":1},"smell":{"value":"none","weight":1},"habitat":{"value":"open_field","weight":3},"water_droplet_test":{"value":"flat","weight":1}},"leaf":{"leaf_type":{"value":"simple","weight":1},"shape":{"value":"linear","weight":5},"edge":{"value":"entire","weight":1},"tip":{"value":"acuminate","weight":1},"base":{"value":"cuneate","weight":1},"colors":{"value":["green"],"weight":1},"color_pattern":{"value":"solid","weight":1},"surface_top":{"value":"matte","weight":1},"surface_bottom":{"value":"matte","weight":1},"arrangement":{"value":"alternate","weight":1},"size":{"value":"very_large_>50cm","weight":3},"venation":{"value":"parallel","weight":4},"texture":{"value":"leathery","weight":2},"petiole_attach":{"value":"normal","weight":1}},"stem":{"type":{"value":"erect","weight":1},"cross_section":{"value":"round","weight":1},"surface":{"value":"smooth","weight":1},"colors":{"value":["green"],"weight":1},"interior":{"value":"solid","weight":1},"has_thorns":{"value":"no","weight":1}},"flower":{"colors":{"value":["white","purple"],"weight":3},"color_pattern":{"value":"solid","weight":1},"petal_count":{"value":"5","weight":1},"symmetry":{"value":"radial","weight":1},"size":{"value":"medium_15-30mm","weight":1},"arrangement":{"value":"panicle","weight":4},"position":{"value":"terminal","weight":1},"orientation":{"value":"upright","weight":1},"special_shape":{"value":"none","weight":1},"fragrant":{"value":"no","weight":1}},"fruit":{"type":{"value":"achene","weight":3},"colors":{"value":["green"],"weight":1},"size":{"value":"small_5-15mm","weight":1},"surface":{"value":"smooth","weight":1}},"root":{"type":{"value":"rhizome","weight":3}}},"usage":{"type":["edible_leaf"],"edible_parts":["春季幼嫩筍狀萌蘗（剝除外葉鞘後之嫩芽）"],"preparation":"如竹筍處理：冷水或米泔水煮去苦澀，再涼拌、滷煮或炒肉絲","season":"春季為主，冷暖年節氣可略前後移","warnings":"葉緣割手；須確認為芒屬大型叢生型態與春季筍期，勿與幼竹或有毒草本芽混淆。"},"human_readable":{"description_zh":"葉緣割手；須確認為芒屬大型叢生型態與春季筍期，勿與幼竹或有毒草本芽混淆。","diagnostic_features":["多年生大型禾草，成株高 2-4m，常形成廣闊叢聚","稈粗直，節部常被白粉，質硬；基部宿存葉鞘包被","葉線形，長 40-80cm，葉身與葉舌邊緣鋒利，易割傷皮膚（操作須戴手套）","秋季抽出大型羽狀銀白色至淡紫色髮狀花序，開展如羽扇","春季自地面或舊稈腋抽出，外形似小竹筍，外被層層革質葉鞘，內層色淡可食（關鍵可食期指標）"]},"distribution":{"altitude_range":[0,1500],"climate_zones":["subtropical","tropical","temperate"]},"total_weight":69,"photo_observable_weight":61},{"id":"zanthoxylum_ailanthoides","scientific_name":"Zanthoxylum ailanthoides","common_names":{"zh-TW":"食茱萸（紅刺椒）","en":"Ailanthus-leaf Prickly Ash"},"category":"edible","features":{"overall":{"growth_form":{"value":"tree","weight":3},"height_estimate":{"value":">5m","weight":3},"latex":{"value":"none","weight":1},"smell":{"value":"aromatic","weight":2},"habitat":{"value":"forest_floor","weight":2},"water_droplet_test":{"value":"flat","weight":1}},"leaf":{"leaf_type":{"value":"pinnate_compound","weight":3},"shape":{"value":"lanceolate","weight":2},"edge":{"value":"serrated","weight":2},"tip":{"value":"acuminate","weight":1},"base":{"value":"cuneate","weight":1},"colors":{"value":["green"],"weight":1},"color_pattern":{"value":"solid","weight":1},"surface_top":{"value":"matte","weight":1},"surface_bottom":{"value":"matte","weight":1},"arrangement":{"value":"alternate","weight":1},"size":{"value":"medium_5-15cm","weight":1},"venation":{"value":"pinnate","weight":1},"texture":{"value":"papery","weight":1},"petiole_attach":{"value":"normal","weight":1}},"stem":{"type":{"value":"erect","weight":1},"cross_section":{"value":"round","weight":1},"surface":{"value":"bark_rough","weight":5},"colors":{"value":["brown","gray"],"weight":4},"interior":{"value":"solid","weight":1},"has_thorns":{"value":"yes","weight":4}},"flower":null,"fruit":{"type":{"value":"capsule","weight":1},"colors":{"value":["red"],"weight":3},"size":{"value":"small_5-15mm","weight":1},"surface":{"value":"smooth","weight":1}},"root":{"type":{"value":"fibrous","weight":1}}},"usage":{"type":["edible_leaf"],"edible_parts":["春季嫩芽與幼嫩羽狀複葉（調味、拌食少量）"],"preparation":"嫩芽汆燙數秒去青後涼拌、煎蛋或切碎配肉；亦乾燥研磨作辛香料","season":"春季 3-5 月為主，視海拔提前或延後","warnings":"花椒科有刺；過量生食刺激腸胃；對辛香料過敏者慎用。與苦楝、臭椿等區分之關鍵在於搓葉氣味與麻感。"},"human_readable":{"description_zh":"花椒科有刺；過量生食刺激腸胃；對辛香料過敏者慎用。與苦楝、臭椿等區分之關鍵在於搓葉氣味與麻感。","diagnostic_features":["落葉喬木，高 5-15m，樹冠廣卵形，樹皮粗糙縱裂至片狀剝落","樹幹常具瘤狀突起與木質化瘤刺，枝條具直刺或彎刺（警戒：避免徒手攀折）","奇數羽狀複葉互生，小葉 7-15 枚，卵狀披針形，長 6-12cm，葉緣細鋸齒","小葉搗碎或搓揉釋放強烈花椒、檸檬烯類辛香（關鍵特徵），回味麻舌","蓇葖果小，成熟時紅色，果皮具腺點與油胞，散發香辛料氣息"]},"distribution":{"altitude_range":[300,1800],"climate_zones":["subtropical","temperate"]},"total_weight":53,"photo_observable_weight":47},{"id":"trema_orientalis","scientific_name":"Trema orientalis","common_names":{"zh-TW":"山黃麻","en":"Indian Charcoal Tree"},"category":"edible","features":{"overall":{"growth_form":{"value":"tree","weight":3},"height_estimate":{"value":">5m","weight":3},"latex":{"value":"none","weight":1},"smell":{"value":"none","weight":1},"habitat":{"value":"roadside","weight":2},"water_droplet_test":{"value":"flat","weight":1}},"leaf":{"leaf_type":{"value":"simple","weight":1},"shape":{"value":"oval","weight":1},"edge":{"value":"serrated","weight":2},"tip":{"value":"acuminate","weight":1},"base":{"value":"cuneate","weight":1},"colors":{"value":["green"],"weight":1},"color_pattern":{"value":"solid","weight":1},"surface_top":{"value":"rough","weight":4},"surface_bottom":{"value":"matte","weight":1},"arrangement":{"value":"alternate","weight":1},"size":{"value":"medium_5-15cm","weight":1},"venation":{"value":"pinnate","weight":1},"texture":{"value":"papery","weight":1},"petiole_attach":{"value":"normal","weight":1}},"stem":{"type":{"value":"erect","weight":1},"cross_section":{"value":"round","weight":1},"surface":{"value":"smooth","weight":1},"colors":{"value":["brown"],"weight":2},"interior":{"value":"solid","weight":1},"has_thorns":{"value":"no","weight":1}},"flower":null,"fruit":{"type":{"value":"drupe","weight":3},"colors":{"value":["orange","red","purple"],"weight":4},"size":{"value":"tiny_<5mm","weight":2},"surface":{"value":"smooth","weight":1}},"root":{"type":{"value":"fibrous","weight":1}}},"usage":{"type":["edible_leaf","edible_fruit"],"edible_parts":["嫩葉（煮熟）","成熟小型漿果狀核果"],"preparation":"嫩葉川燙後炒食或煮湯；熟果少量鮮食或煮甜湯","season":"嫩葉春夏；果熟多夏秋，因地域略異","warnings":"果實極小，誤採尚未由紅轉黑者澀味與單寧較高。污染環境植株勿採。"},"human_readable":{"description_zh":"果實極小，誤採尚未由紅轉黑者澀味與單寧較高。污染環境植株勿採。","diagnostic_features":["常綠小喬木至中喬木，高 5-15m，枝條細長，樹皮灰褐而稍光滑","葉互生，兩列近於水平伸展，長卵形至卵狀披針形，長 6-14cm，先端漸尖，葉緣細鋸齒","葉面粗糙似砂布但較構樹細緻，背面色淡網脈明顯；脫落前可轉黃","腋生小型核果，直徑約 3-4mm，熟時由橙紅變深紫至近黑（關鍵特徵），多肉質薄層","先驅陽性樹種，都市荒廢地、坡坎、防火巷與河岸裸地最常見之一"]},"distribution":{"altitude_range":[0,1500],"climate_zones":["subtropical","tropical"]},"total_weight":47,"photo_observable_weight":42},{"id":"pterocypsela_indica","scientific_name":"Pterocypsela indica","common_names":{"zh-TW":"鵝仔草（鵝菜、山萵苣）","en":"Indian Lettuce"},"category":"edible","features":{"overall":{"growth_form":{"value":"herb","weight":1},"height_estimate":{"value":"30-100cm","weight":1},"latex":{"value":"yes_white","weight":2},"smell":{"value":"none","weight":1},"habitat":{"value":"open_field","weight":3},"water_droplet_test":{"value":"flat","weight":1}},"leaf":{"leaf_type":{"value":"simple","weight":1},"shape":{"value":"lanceolate","weight":2},"edge":{"value":"lobed","weight":3},"tip":{"value":"acuminate","weight":1},"base":{"value":"cuneate","weight":1},"colors":{"value":["green"],"weight":1},"color_pattern":{"value":"solid","weight":1},"surface_top":{"value":"matte","weight":1},"surface_bottom":{"value":"matte","weight":1},"arrangement":{"value":"alternate","weight":1},"size":{"value":"medium_5-15cm","weight":1},"venation":{"value":"pinnate","weight":1},"texture":{"value":"papery","weight":1},"petiole_attach":{"value":"normal","weight":1}},"stem":{"type":{"value":"erect","weight":1},"cross_section":{"value":"round","weight":1},"surface":{"value":"smooth","weight":1},"colors":{"value":["green"],"weight":1},"interior":{"value":"hollow","weight":3},"has_thorns":{"value":"no","weight":1}},"flower":{"colors":{"value":["yellow"],"weight":2},"color_pattern":{"value":"solid","weight":1},"petal_count":{"value":"many","weight":4},"symmetry":{"value":"radial","weight":1},"size":{"value":"medium_15-30mm","weight":1},"arrangement":{"value":"head_composite","weight":3},"position":{"value":"terminal","weight":1},"orientation":{"value":"upright","weight":1},"special_shape":{"value":"none","weight":1},"fragrant":{"value":"no","weight":1}},"fruit":{"type":{"value":"achene","weight":3},"colors":{"value":["brown"],"weight":2},"size":{"value":"tiny_<5mm","weight":2},"surface":{"value":"smooth","weight":1}},"root":{"type":{"value":"fibrous","weight":1}}},"usage":{"type":["edible_leaf","edible_root"],"edible_parts":["嫩莖葉"],"preparation":"川燙去苦味後涼拌（麻油、蒜泥）、快炒或煮粥","season":"全年可採，冬春基生葉肥厚","warnings":"乳汁過敏者戴手套；公路邊植株注意重金屬與灰塵。與有毒乳草類勿僅靠葉形判斷。"},"human_readable":{"description_zh":"乳汁過敏者戴手套；公路邊植株注意重金屬與灰塵。與有毒乳草類勿僅靠葉形判斷。","diagnostic_features":["一年生草本，高 30-100cm，全株含乳汁，斷面明顯中空（關鍵特徵）","莖直立具縱稜，表面光滑或略被粉；節間長度中至長","基生葉大型，琴形羽裂至深羽狀分裂，長 10-25cm，裂片不等大","莖生葉漸小，基部常抱莖或半抱莖，先端裂片變化大","頭狀花序小型，多花排列成鬆散繖房或圓錐狀；花淡黃色舌狀花，無明顯大型舌瓣展示"]},"distribution":{"altitude_range":[0,2500],"climate_zones":["subtropical","temperate"]},"total_weight":59,"photo_observable_weight":50},{"id":"melia_azedarach","scientific_name":"Melia azedarach","common_names":{"zh-TW":"苦楝","en":"Chinaberry"},"category":"dangerous","danger_level":"high","features":{"overall":{"growth_form":{"value":"tree","weight":3},"height_estimate":{"value":">5m","weight":3},"latex":{"value":"none","weight":1},"smell":{"value":"aromatic","weight":2},"habitat":{"value":"urban","weight":3},"water_droplet_test":{"value":"flat","weight":1}},"leaf":{"leaf_type":{"value":"bipinnate_compound","weight":3},"shape":{"value":"oval","weight":1},"edge":{"value":"serrated","weight":2},"tip":{"value":"acuminate","weight":1},"base":{"value":"cuneate","weight":1},"colors":{"value":["green"],"weight":1},"color_pattern":{"value":"solid","weight":1},"surface_top":{"value":"matte","weight":1},"surface_bottom":{"value":"matte","weight":1},"arrangement":{"value":"alternate","weight":1},"size":{"value":"small_2-5cm","weight":4},"venation":{"value":"pinnate","weight":1},"texture":{"value":"papery","weight":1},"petiole_attach":{"value":"normal","weight":1}},"stem":{"type":{"value":"erect","weight":1},"cross_section":{"value":"round","weight":1},"surface":{"value":"smooth","weight":1},"colors":{"value":["brown"],"weight":2},"interior":{"value":"solid","weight":1},"has_thorns":{"value":"no","weight":1}},"flower":{"colors":{"value":["purple"],"weight":3},"color_pattern":{"value":"solid","weight":1},"petal_count":{"value":"5","weight":1},"symmetry":{"value":"radial","weight":1},"size":{"value":"medium_15-30mm","weight":1},"arrangement":{"value":"panicle","weight":4},"position":{"value":"terminal","weight":1},"orientation":{"value":"upright","weight":1},"special_shape":{"value":"none","weight":1},"fragrant":{"value":"yes","weight":3}},"fruit":{"type":{"value":"drupe","weight":3},"colors":{"value":["yellow","brown"],"weight":4},"size":{"value":"small_5-15mm","weight":1},"surface":{"value":"smooth","weight":1}},"root":{"type":{"value":"fibrous","weight":1}}},"human_readable":{"toxicity":"果實有毒，含苦楝素，誤食致嘔吐、腹瀉、呼吸困難","symptoms":["嘔吐","腹瀉","呼吸困難"],"first_aid":"催吐後送醫","affected_parts":[],"diagnostic_features":["二至三回羽狀複葉，小葉卵形至橢圓形，長3-5cm，鋸齒緣","淡紫色小花，圓錐花序，有香味","黃褐色核果，球形，直徑1-2cm","落葉喬木，高可達15-20m，樹皮灰褐色縱裂"]},"distribution":{"altitude_range":[0,800],"climate_zones":["subtropical"]},"total_weight":67,"photo_observable_weight":58},{"id":"pteridium_aquilinum","scientific_name":"Pteridium aquilinum","common_names":{"zh-TW":"蕨","en":"Bracken Fern"},"category":"dangerous","danger_level":"low","features":{"overall":{"growth_form":{"value":"fern","weight":3},"height_estimate":{"value":"30-100cm","weight":1},"latex":{"value":"none","weight":1},"smell":{"value":"none","weight":1},"habitat":{"value":"forest_floor","weight":2},"water_droplet_test":{"value":"flat","weight":1}},"leaf":{"leaf_type":{"value":"bipinnate_compound","weight":3},"shape":{"value":"oval","weight":1},"edge":{"value":"entire","weight":1},"tip":{"value":"acuminate","weight":1},"base":{"value":"cuneate","weight":1},"colors":{"value":["green"],"weight":1},"color_pattern":{"value":"solid","weight":1},"surface_top":{"value":"matte","weight":1},"surface_bottom":{"value":"matte","weight":1},"arrangement":{"value":"alternate","weight":1},"size":{"value":"very_large_>50cm","weight":3},"venation":{"value":"pinnate","weight":1},"texture":{"value":"papery","weight":1},"petiole_attach":{"value":"normal","weight":1}},"stem":{"type":{"value":"erect","weight":1},"cross_section":{"value":"round","weight":1},"surface":{"value":"smooth","weight":1},"colors":{"value":["green"],"weight":1},"interior":{"value":"solid","weight":1},"has_thorns":{"value":"no","weight":1}},"flower":null,"fruit":{"type":{"value":"spore","weight":2},"colors":{"value":["green"],"weight":1},"size":{"value":"tiny_<5mm","weight":2},"surface":{"value":"smooth","weight":1}},"root":{"type":{"value":"rhizome","weight":3}}},"human_readable":{"toxicity":"含致癌物 ptaquiloside，生食有害","symptoms":["長期食用可能致癌"],"first_aid":"少量誤食無立即危險，但不建議食用","affected_parts":[],"diagnostic_features":["大型三角形二至三回羽狀複葉，長50-100cm","地下根莖粗壯，地上無明顯莖","嫩芽捲曲呈拳頭狀"]},"distribution":{"altitude_range":[0,2500],"climate_zones":["subtropical","temperate"]},"total_weight":42,"photo_observable_weight":35},{"id":"melastoma_candidum","scientific_name":"Melastoma candidum","common_names":{"zh-TW":"野牡丹","en":"Malabar Melastome"},"category":"edible","features":{"overall":{"growth_form":{"value":"shrub","weight":3},"height_estimate":{"value":"30-100cm","weight":1},"latex":{"value":"none","weight":1},"smell":{"value":"none","weight":1},"habitat":{"value":"roadside","weight":2},"water_droplet_test":{"value":"flat","weight":1}},"leaf":{"leaf_type":{"value":"simple","weight":1},"shape":{"value":"oval","weight":1},"edge":{"value":"entire","weight":1},"tip":{"value":"acuminate","weight":1},"base":{"value":"cuneate","weight":1},"colors":{"value":["green"],"weight":1},"color_pattern":{"value":"solid","weight":1},"surface_top":{"value":"matte","weight":1},"surface_bottom":{"value":"matte","weight":1},"arrangement":{"value":"opposite","weight":4},"size":{"value":"medium_5-15cm","weight":1},"venation":{"value":"palmate","weight":3},"texture":{"value":"papery","weight":1},"petiole_attach":{"value":"normal","weight":1}},"stem":{"type":{"value":"erect","weight":1},"cross_section":{"value":"round","weight":1},"surface":{"value":"smooth","weight":1},"colors":{"value":["green"],"weight":1},"interior":{"value":"solid","weight":1},"has_thorns":{"value":"no","weight":1}},"flower":{"colors":{"value":["red","pink","purple"],"weight":3},"color_pattern":{"value":"solid","weight":1},"petal_count":{"value":"5","weight":1},"symmetry":{"value":"radial","weight":1},"size":{"value":"medium_15-30mm","weight":1},"arrangement":{"value":"solitary","weight":3},"position":{"value":"terminal","weight":1},"orientation":{"value":"upright","weight":1},"special_shape":{"value":"none","weight":1},"fragrant":{"value":"no","weight":1}},"fruit":{"type":{"value":"berry","weight":3},"colors":{"value":["purple"],"weight":3},"size":{"value":"small_5-15mm","weight":1},"surface":{"value":"smooth","weight":1}},"root":{"type":{"value":"fibrous","weight":1}}},"usage":{"type":["edible_fruit"],"edible_parts":["果實（紫黑色漿果）"],"preparation":"","season":"","warnings":""},"human_readable":{"description_zh":"","diagnostic_features":["橢圓形至卵形，長5-10cm，3-5條主脈明顯","粉紅至紫色大花，直徑5-7cm，五瓣","壺形漿果，成熟紫黑色","灌木，高50-150cm"]},"distribution":{"altitude_range":[0,1200],"climate_zones":["subtropical"]},"total_weight":57,"photo_observable_weight":51},{"id":"emilia_sonchifolia","scientific_name":"Emilia sonchifolia","common_names":{"zh-TW":"紫背草（一點紅）","en":"Lilac Tasselflower"},"category":"edible","features":{"overall":{"growth_form":{"value":"herb","weight":1},"height_estimate":{"value":"<30cm","weight":3},"latex":{"value":"none","weight":1},"smell":{"value":"none","weight":1},"habitat":{"value":"roadside","weight":2},"water_droplet_test":{"value":"flat","weight":1}},"leaf":{"leaf_type":{"value":"simple","weight":1},"shape":{"value":"oval","weight":1},"edge":{"value":"lobed","weight":3},"tip":{"value":"acuminate","weight":1},"base":{"value":"cuneate","weight":1},"colors":{"value":["green"],"weight":1},"color_pattern":{"value":"solid","weight":1},"surface_top":{"value":"matte","weight":1},"surface_bottom":{"value":"matte","weight":1},"arrangement":{"value":"alternate","weight":1},"size":{"value":"medium_5-15cm","weight":1},"venation":{"value":"pinnate","weight":1},"texture":{"value":"papery","weight":1},"petiole_attach":{"value":"normal","weight":1}},"stem":{"type":{"value":"erect","weight":1},"cross_section":{"value":"round","weight":1},"surface":{"value":"smooth","weight":1},"colors":{"value":["green"],"weight":1},"interior":{"value":"solid","weight":1},"has_thorns":{"value":"no","weight":1}},"flower":{"colors":{"value":["orange","red"],"weight":5},"color_pattern":{"value":"solid","weight":1},"petal_count":{"value":"5","weight":1},"symmetry":{"value":"radial","weight":1},"size":{"value":"medium_15-30mm","weight":1},"arrangement":{"value":"head_composite","weight":3},"position":{"value":"terminal","weight":1},"orientation":{"value":"upright","weight":1},"special_shape":{"value":"bell_tubular","weight":5},"fragrant":{"value":"no","weight":1}},"fruit":{"type":{"value":"capsule","weight":1},"colors":{"value":["green"],"weight":1},"size":{"value":"small_5-15mm","weight":1},"surface":{"value":"smooth","weight":1}},"root":{"type":{"value":"fibrous","weight":1}}},"usage":{"type":["edible_leaf","edible_root"],"edible_parts":["嫩莖葉"],"preparation":"","season":"","warnings":""},"human_readable":{"description_zh":"","diagnostic_features":["琴形或卵形，長3-8cm，邊緣不規則齒裂","紫紅色頭狀花序，管狀小花","草本，高20-50cm"]},"distribution":{"altitude_range":[0,1000],"climate_zones":["subtropical"]},"total_weight":56,"photo_observable_weight":50},{"id":"pseudognaphalium_affine","scientific_name":"Pseudognaphalium affine","common_names":{"zh-TW":"鼠麴草","en":"Cudweed"},"category":"edible","features":{"overall":{"growth_form":{"value":"herb","weight":1},"height_estimate":{"value":"<30cm","weight":3},"latex":{"value":"none","weight":1},"smell":{"value":"none","weight":1},"habitat":{"value":"roadside","weight":2},"water_droplet_test":{"value":"flat","weight":1}},"leaf":{"leaf_type":{"value":"simple","weight":1},"shape":{"value":"lanceolate","weight":2},"edge":{"value":"entire","weight":1},"tip":{"value":"acuminate","weight":1},"base":{"value":"cuneate","weight":1},"colors":{"value":["green"],"weight":1},"color_pattern":{"value":"solid","weight":1},"surface_top":{"value":"hairy","weight":3},"surface_bottom":{"value":"matte","weight":1},"arrangement":{"value":"alternate","weight":1},"size":{"value":"small_2-5cm","weight":4},"venation":{"value":"pinnate","weight":1},"texture":{"value":"papery","weight":1},"petiole_attach":{"value":"normal","weight":1}},"stem":{"type":{"value":"erect","weight":1},"cross_section":{"value":"round","weight":1},"surface":{"value":"hairy","weight":4},"colors":{"value":["green"],"weight":1},"interior":{"value":"solid","weight":1},"has_thorns":{"value":"no","weight":1}},"flower":{"colors":{"value":["yellow"],"weight":2},"color_pattern":{"value":"solid","weight":1},"petal_count":{"value":"5","weight":1},"symmetry":{"value":"radial","weight":1},"size":{"value":"medium_15-30mm","weight":1},"arrangement":{"value":"head_composite","weight":3},"position":{"value":"terminal","weight":1},"orientation":{"value":"upright","weight":1},"special_shape":{"value":"none","weight":1},"fragrant":{"value":"no","weight":1}},"fruit":{"type":{"value":"capsule","weight":1},"colors":{"value":["green"],"weight":1},"size":{"value":"small_5-15mm","weight":1},"surface":{"value":"smooth","weight":1}},"root":{"type":{"value":"fibrous","weight":1}}},"usage":{"type":["edible_leaf","edible_root"],"edible_parts":["嫩莖葉（做草仔粿）"],"preparation":"","season":"","warnings":""},"human_readable":{"description_zh":"","diagnostic_features":["倒披針形至匙形，長3-6cm，全株白色絨毛","黃色小頭狀花序，密集排列","草本，高15-40cm，全株被白色絨毛"]},"distribution":{"altitude_range":[0,1500],"climate_zones":["subtropical"]},"total_weight":56,"photo_observable_weight":50},{"id":"centella_asiatica","scientific_name":"Centella asiatica","common_names":{"zh-TW":"雷公根","en":"Gotu Kola"},"category":"medicinal","features":{"overall":{"growth_form":{"value":"herb","weight":1},"height_estimate":{"value":"30-100cm","weight":1},"latex":{"value":"none","weight":1},"smell":{"value":"none","weight":1},"habitat":{"value":"open_field","weight":3},"water_droplet_test":{"value":"flat","weight":1}},"leaf":{"leaf_type":{"value":"simple","weight":1},"shape":{"value":"round","weight":4},"edge":{"value":"crenate","weight":5},"tip":{"value":"rounded","weight":3},"base":{"value":"cuneate","weight":1},"colors":{"value":["green"],"weight":1},"color_pattern":{"value":"solid","weight":1},"surface_top":{"value":"matte","weight":1},"surface_bottom":{"value":"matte","weight":1},"arrangement":{"value":"clustered","weight":2},"size":{"value":"small_2-5cm","weight":4},"venation":{"value":"palmate","weight":3},"texture":{"value":"papery","weight":1},"petiole_attach":{"value":"normal","weight":1}},"stem":{"type":{"value":"creeping","weight":4},"cross_section":{"value":"round","weight":1},"surface":{"value":"smooth","weight":1},"colors":{"value":["green"],"weight":1},"interior":{"value":"solid","weight":1},"has_thorns":{"value":"no","weight":1}},"flower":{"colors":{"value":["pink","white"],"weight":3},"color_pattern":{"value":"solid","weight":1},"petal_count":{"value":"5","weight":1},"symmetry":{"value":"radial","weight":1},"size":{"value":"tiny_<5mm","weight":4},"arrangement":{"value":"umbel","weight":4},"position":{"value":"terminal","weight":1},"orientation":{"value":"upright","weight":1},"special_shape":{"value":"none","weight":1},"fragrant":{"value":"no","weight":1}},"fruit":{"type":{"value":"capsule","weight":1},"colors":{"value":["green"],"weight":1},"size":{"value":"small_5-15mm","weight":1},"surface":{"value":"smooth","weight":1}},"root":{"type":{"value":"fibrous","weight":1}}},"usage":{"type":["wound_care","anti_inflammatory"],"edible_parts":["全草"],"preparation":"","season":"","warnings":"","medicinal_effects":["消炎","促進傷口癒合","清熱解毒"]},"human_readable":{"description_zh":"","diagnostic_features":["腎形至圓形，長2-5cm，邊緣鈍齒","繖形花序，小花紫紅色，極小","匍匐草本，節上生根"]},"distribution":{"altitude_range":[0,1500],"climate_zones":["subtropical"]},"total_weight":69,"photo_observable_weight":63},{"id":"momordica_charantia_var","scientific_name":"Momordica charantia var. abbreviata","common_names":{"zh-TW":"山苦瓜","en":"Wild Bitter Gourd"},"category":"edible","features":{"overall":{"growth_form":{"value":"vine","weight":4},"height_estimate":{"value":"30-100cm","weight":1},"latex":{"value":"none","weight":1},"smell":{"value":"none","weight":1},"habitat":{"value":"forest_floor","weight":2},"water_droplet_test":{"value":"flat","weight":1}},"leaf":{"leaf_type":{"value":"simple","weight":1},"shape":{"value":"fan","weight":6},"edge":{"value":"lobed","weight":3},"tip":{"value":"acuminate","weight":1},"base":{"value":"cuneate","weight":1},"colors":{"value":["green"],"weight":1},"color_pattern":{"value":"solid","weight":1},"surface_top":{"value":"matte","weight":1},"surface_bottom":{"value":"matte","weight":1},"arrangement":{"value":"alternate","weight":1},"size":{"value":"medium_5-15cm","weight":1},"venation":{"value":"pinnate","weight":1},"texture":{"value":"papery","weight":1},"petiole_attach":{"value":"normal","weight":1}},"stem":{"type":{"value":"creeping","weight":4},"cross_section":{"value":"round","weight":1},"surface":{"value":"smooth","weight":1},"colors":{"value":["green"],"weight":1},"interior":{"value":"solid","weight":1},"has_thorns":{"value":"no","weight":1}},"flower":{"colors":{"value":["yellow"],"weight":2},"color_pattern":{"value":"solid","weight":1},"petal_count":{"value":"5","weight":1},"symmetry":{"value":"radial","weight":1},"size":{"value":"medium_15-30mm","weight":1},"arrangement":{"value":"solitary","weight":3},"position":{"value":"terminal","weight":1},"orientation":{"value":"upright","weight":1},"special_shape":{"value":"none","weight":1},"fragrant":{"value":"no","weight":1}},"fruit":{"type":{"value":"berry","weight":3},"colors":{"value":["yellow","orange"],"weight":4},"size":{"value":"large_>30mm","weight":4},"surface":{"value":"smooth","weight":1}},"root":{"type":{"value":"fibrous","weight":1}}},"usage":{"type":["edible_leaf","edible_fruit"],"edible_parts":["果實","嫩葉"],"preparation":"","season":"","warnings":""},"human_readable":{"description_zh":"","diagnostic_features":["掌狀深裂，5-7裂片","紡錘形，表面有瘤狀突起，成熟橙黃色","攀緣藤本，有卷鬚"]},"distribution":{"altitude_range":[0,800],"climate_zones":["subtropical"]},"total_weight":66,"photo_observable_weight":60},{"id":"cyathea_lepifera","scientific_name":"Cyathea lepifera","common_names":{"zh-TW":"筆筒樹","en":"Flying Spider-monkey Tree Fern"},"category":"edible","features":{"overall":{"growth_form":{"value":"fern","weight":3},"height_estimate":{"value":">5m","weight":3},"latex":{"value":"none","weight":1},"smell":{"value":"none","weight":1},"habitat":{"value":"forest_floor","weight":2},"water_droplet_test":{"value":"flat","weight":1}},"leaf":{"leaf_type":{"value":"bipinnate_compound","weight":3},"shape":{"value":"lanceolate","weight":2},"edge":{"value":"entire","weight":1},"tip":{"value":"acuminate","weight":1},"base":{"value":"cuneate","weight":1},"colors":{"value":["green"],"weight":1},"color_pattern":{"value":"solid","weight":1},"surface_top":{"value":"scaly","weight":4},"surface_bottom":{"value":"matte","weight":1},"arrangement":{"value":"clustered","weight":2},"size":{"value":"very_large_>50cm","weight":3},"venation":{"value":"pinnate","weight":1},"texture":{"value":"papery","weight":1},"petiole_attach":{"value":"normal","weight":1}},"stem":{"type":{"value":"erect","weight":1},"cross_section":{"value":"round","weight":1},"surface":{"value":"scaly","weight":3},"colors":{"value":["brown"],"weight":2},"interior":{"value":"solid","weight":1},"has_thorns":{"value":"no","weight":1}},"flower":null,"fruit":{"type":{"value":"spore","weight":2},"colors":{"value":["brown"],"weight":2},"size":{"value":"tiny_<5mm","weight":2},"surface":{"value":"smooth","weight":1}},"root":{"type":{"value":"fibrous","weight":1}}},"usage":{"type":["edible_leaf"],"edible_parts":["嫩心（樹頂生長點）"],"preparation":"","season":"","warnings":""},"human_readable":{"description_zh":"","diagnostic_features":["大型二回至三回羽狀複葉，長2-3m","直立樹幹狀，高可達10-15m，幹面有橢圓形葉痕","嫩芽有金褐色鱗片"]},"distribution":{"altitude_range":[200,1500],"climate_zones":["subtropical"]},"total_weight":51,"photo_observable_weight":46},{"id":"macrolepiota_procera","scientific_name":"Macrolepiota procera","common_names":{"zh-TW":"高大環柄菇","en":"Parasol Mushroom"},"category":"edible","features":{"overall":{"growth_form":{"value":"fungus","weight":3},"height_estimate":{"value":"30-100cm","weight":1},"latex":{"value":"none","weight":1},"smell":{"value":"none","weight":1},"habitat":{"value":"open_field","weight":3},"water_droplet_test":{"value":"flat","weight":1}},"leaf":{"leaf_type":{"value":"simple","weight":1},"shape":{"value":"round","weight":4},"edge":{"value":"entire","weight":1},"tip":{"value":"rounded","weight":3},"base":{"value":"rounded","weight":3},"colors":{"value":["brown"],"weight":4},"color_pattern":{"value":"spotted","weight":6},"surface_top":{"value":"scaly","weight":4},"surface_bottom":{"value":"matte","weight":1},"arrangement":{"value":"clustered","weight":2},"size":{"value":"large_15-50cm","weight":2},"venation":{"value":"reticulate","weight":3},"texture":{"value":"fleshy","weight":3},"petiole_attach":{"value":"normal","weight":1}},"stem":{"type":{"value":"erect","weight":1},"cross_section":{"value":"round","weight":1},"surface":{"value":"scaly","weight":3},"colors":{"value":["brown"],"weight":2},"interior":{"value":"hollow","weight":3},"has_thorns":{"value":"no","weight":1}},"flower":null,"fruit":{"type":{"value":"spore","weight":2},"colors":{"value":["brown"],"weight":2},"size":{"value":"tiny_<5mm","weight":2},"surface":{"value":"smooth","weight":1}},"root":{"type":{"value":"fibrous","weight":1}}},"usage":{"type":["edible_leaf"],"edible_parts":["菌傘（煎烤最佳）"],"preparation":"去除菌柄後煎烤或油炸。味道鮮美。","season":"夏秋雨後","warnings":"極易與有毒綠褶菇混淆！必須確認孢子印為白色。"},"human_readable":{"description_zh":"大型傘菌，菌傘直徑15-30cm，表面有褐色鱗片。菌柄細長，有可滑動的環。","diagnostic_features":["菌傘大型（15-30cm），表面褐色鱗片呈同心圓排列","菌柄細長，有可上下滑動的雙層環","菌褶白色（成熟後也不變色，關鍵！）","孢子印白色（關鍵區分：綠褶菇為灰綠色）","常生長於草地、牧場"]},"distribution":{"altitude_range":[0,1500],"climate_zones":["subtropical","temperate"]},"total_weight":67,"photo_observable_weight":60},{"id":"termitomyces_sp","scientific_name":"Termitomyces sp.","common_names":{"zh-TW":"雞肉絲菇","en":"Termite Mushroom"},"category":"edible","features":{"overall":{"growth_form":{"value":"fungus","weight":3},"height_estimate":{"value":"<30cm","weight":3},"latex":{"value":"none","weight":1},"smell":{"value":"aromatic","weight":2},"habitat":{"value":"forest_floor","weight":2},"water_droplet_test":{"value":"flat","weight":1}},"leaf":{"leaf_type":{"value":"simple","weight":1},"shape":{"value":"round","weight":4},"edge":{"value":"entire","weight":1},"tip":{"value":"acute","weight":3},"base":{"value":"rounded","weight":3},"colors":{"value":["brown"],"weight":4},"color_pattern":{"value":"solid","weight":1},"surface_top":{"value":"matte","weight":1},"surface_bottom":{"value":"matte","weight":1},"arrangement":{"value":"clustered","weight":2},"size":{"value":"medium_5-15cm","weight":1},"venation":{"value":"reticulate","weight":3},"texture":{"value":"fleshy","weight":3},"petiole_attach":{"value":"normal","weight":1}},"stem":{"type":{"value":"erect","weight":1},"cross_section":{"value":"round","weight":1},"surface":{"value":"smooth","weight":1},"colors":{"value":["brown"],"weight":2},"interior":{"value":"solid","weight":1},"has_thorns":{"value":"no","weight":1}},"flower":null,"fruit":{"type":{"value":"spore","weight":2},"colors":{"value":["brown"],"weight":2},"size":{"value":"tiny_<5mm","weight":2},"surface":{"value":"smooth","weight":1}},"root":{"type":{"value":"fibrous","weight":1}}},"usage":{"type":["edible_leaf"],"edible_parts":["全菇"],"preparation":"炒食或煮湯。味鮮似雞肉。","season":"夏秋雨季","warnings":"極易與白色毒鵝膏混淆！確認生長在白蟻巢上。菌柄基部無杯狀菌托。"},"human_readable":{"description_zh":"中型菇類，菌傘灰褐色至深褐色，中央常有小突起。味道鮮美似雞肉。","diagnostic_features":["生長在白蟻巢附近（關鍵特徵）","菌傘灰褐色，中央有尖突起","菌柄基部無杯狀菌托（毒鵝膏有！）","菌褶白色至淡粉紅色","味道鮮美似雞肉"]},"distribution":{"altitude_range":[0,1000],"climate_zones":["subtropical","tropical"]},"total_weight":56,"photo_observable_weight":50}]''')

print(f"✅ Loaded {len(PLANTS_DB)} species")
cats = {}
for sp in PLANTS_DB:
    c = sp.get('category', 'unknown')
    cats[c] = cats.get(c, 0) + 1
print(f"   Categories: {cats}")

## 🌿 叢林生存指南資料6 大分類（食/水/醫/住/工具/行），build 時預先渲染為 HTML。

In [ ]:
# Cell 10: Survival Guide

GUIDE_RENDERED = json.loads(r'''{"food":{"title":"食","icon":"🍃","html":"<div style=\"border:1px solid #ddd;border-radius:8px;padding:16px;margin:10px 0;background:#FAFAFA;\"><h3 style=\"margin:0 0 8px 0;\">植物處理方式</h3><p style=\"color:#666;font-size:13px;\"><b>適用情境：</b>辨識出可食用植物後，需要正確處理才能安全食用</p><p style=\"color:#666;font-size:13px;\"><b>所需材料：</b>火源/容器/乾淨石頭/水</p><div style=\"margin:8px 0;padding:10px;background:#F0F8FF;border-radius:6px;border-left:4px solid #4488CC;\"><b>生食法</b><br>確認為安全可生食的植物後，用清水沖洗，直接食用。<br><span style=\"color:#CC6600;font-size:13px;\">⚠️ 只有確定無毒且無寄生蟲風險的才可生食</span></div><div style=\"margin:8px 0;padding:10px;background:#F0F8FF;border-radius:6px;border-left:4px solid #4488CC;\"><b>川燙法</b><br>將水煮沸後，放入植物部位燙 30 秒至 2 分鐘，撈起瀝乾。<br><span style=\"color:#CC6600;font-size:13px;\">⚠️ 可去除部分苦味和輕微毒素</span></div><div style=\"margin:8px 0;padding:10px;background:#F0F8FF;border-radius:6px;border-left:4px solid #4488CC;\"><b>煮沸法</b><br>將植物放入水中煮沸 10-20 分鐘。如植物有輕微毒素，換水煮沸 2-3 次。<br><span style=\"color:#CC6600;font-size:13px;\">⚠️ 換水法可去除水溶性毒素</span></div><div style=\"margin:8px 0;padding:10px;background:#F0F8FF;border-radius:6px;border-left:4px solid #4488CC;\"><b>烘烤法</b><br>將植物放在石板上或用木棍串起，在火旁 15-20cm 處慢烤至熟透。<br><span style=\"color:#CC6600;font-size:13px;\">⚠️ 避免直接放入火中（會焦黑）</span></div><div style=\"margin:8px 0;padding:10px;background:#F0F8FF;border-radius:6px;border-left:4px solid #4488CC;\"><b>浸泡去毒法</b><br>將植物浸泡在流動清水中 12-24 小時，期間換水 3-5 次。<br><span style=\"color:#CC6600;font-size:13px;\">⚠️ 耗時但有效去除水溶性毒素</span></div><div style=\"margin:8px 0;padding:10px;background:#F0F8FF;border-radius:6px;border-left:4px solid #4488CC;\"><b>磨碎法</b><br>取乾淨石頭作為研磨工具，將植物放在平坦石面上反覆搗碎至泥狀。<br><span style=\"color:#CC6600;font-size:13px;\">⚠️ 確保石頭乾淨無污染</span></div><div style=\"margin:8px 0;padding:10px;background:#F0F8FF;border-radius:6px;border-left:4px solid #4488CC;\"><b>磨碎敷用法（藥用）</b><br>1. 用清水沖洗植物<br>2. 用乾淨石頭將植物搗碎成泥<br>3. 均勻敷在傷口周圍（非傷口內部）<br>4. 用大葉或布匹覆蓋固定<br>5. 每 4-6 小時更換新鮮藥泥<br><span style=\"color:#CC6600;font-size:13px;\">⚠️ 確認植物已被正確辨識為藥用。傷口如持續惡化應停止使用。</span></div></div><div style=\"border:1px solid #ddd;border-radius:8px;padding:16px;margin:10px 0;background:#FAFAFA;\"><h3 style=\"margin:0 0 8px 0;\">通用可食性測試法</h3><p style=\"color:#666;font-size:13px;\"><b>適用情境：</b>找到未知植物，AI 無法辨識或信心度不足，需要自行測試是否可食</p><p style=\"color:#666;font-size:13px;\"><b>所需材料：</b>待測植物/8 小時等待時間</p><ol style=\"line-height:2;\"><li><b>分離測試：將植物分為根、莖、葉、花、果分開測試</b><br><span style=\"color:#666;font-size:13px;\">某些植物部分部位可食、部分有毒</span></li><li><b>嗅覺測試：搓碎後聞氣味，強烈刺鼻或杏仁味→放棄</b><br><span style=\"color:#666;font-size:13px;\">杏仁味可能是氰化物</span></li><li><b>皮膚測試：將碎液塗在手腕內側，等待 15 分鐘觀察有無紅腫刺痛</b><br><span style=\"color:#666;font-size:13px;\">有反應則不可食用</span></li><li><b>嘴唇測試：塗少量在嘴唇外側，等待 5 分鐘</b><br><span style=\"color:#666;font-size:13px;\">有麻痺、灼熱感則停止</span></li><li><b>舌尖測試：放少量在舌尖，等待 15 分鐘，不要吞嚥</b><br><span style=\"color:#666;font-size:13px;\">有苦澀、灼熱、麻痺感則吐出漱口</span></li><li><b>少量食用：吞嚥少量（約一茶匙），等待 8 小時</b><br><span style=\"color:#666;font-size:13px;\">期間不吃其他食物，觀察有無腹痛、噁心、頭暈</span></li><li><b>如 8 小時無不適，可增加食用量</b><br><span style=\"color:#666;font-size:13px;\">初次食用仍建議少量</span></li></ol><div style=\"margin-top:8px;padding:8px;background:#FFF3F3;border-radius:4px;\"><b>⚠️ 注意事項：</b><ul style=\"margin:4px 0;\"><li style=\"color:#CC0000;\">⚠️ 此方法非常耗時且有風險，僅在完全無其他選擇時使用</li><li style=\"color:#CC0000;\">有 AI 辨識時優先使用 AI</li><li style=\"color:#CC0000;\">菇類絕對不適用此方法（致命毒菇可能無味無刺激）</li><li style=\"color:#CC0000;\">避免測試：乳白色汁液的植物、紅色種子/果實、三出複葉植物</li><li style=\"color:#CC0000;\">⚠️ 杏仁味 = 氰化物警告！搓碎後聞到苦杏仁味的植物可能含氰苷（hydrogen cyanide），立即放棄。含氰苷的常見植物：生竹筍（需充分煮沸）、樹薯/木薯（必須削皮+長時間煮沸）、苦杏仁、野櫻桃種子、接骨木未成熟果實、月桂櫻（Prunus laurocerasus）的葉片。氰苷加熱可分解，但生食劑量極危險。</li></ul></div></div><div style=\"border:1px solid #ddd;border-radius:8px;padding:16px;margin:10px 0;background:#FAFAFA;\"><h3 style=\"margin:0 0 8px 0;\">昆蟲食用指南</h3><p style=\"color:#666;font-size:13px;\"><b>適用情境：</b>缺乏食物來源，需要從昆蟲取得蛋白質</p><p style=\"color:#666;font-size:13px;\"><b>所需材料：</b>火源（烤熟用）</p><table style=\"width:100%;border-collapse:collapse;font-size:13px;margin:8px 0;\"><tr style=\"background:#f0f0f0;\"><th style=\"padding:6px;\">昆蟲</th><th>營養</th><th>處理</th><th>地點</th></tr><tr><td style=\"padding:6px;border:1px solid #ddd;\">蟋蟀/蚱蜢</td><td style=\"padding:6px;border:1px solid #ddd;\">高蛋白質（60-70%）、低脂</td><td style=\"padding:6px;border:1px solid #ddd;\">去頭、腿、翅 → 用樹枝串起烤熟</td><td style=\"padding:6px;border:1px solid #ddd;\">草地、田野</td></tr><tr><td style=\"padding:6px;border:1px solid #ddd;\">螞蟻</td><td style=\"padding:6px;border:1px solid #ddd;\">蛋白質+脂肪酸</td><td style=\"padding:6px;border:1px solid #ddd;\">大量收集後直接烤食或煮湯</td><td style=\"padding:6px;border:1px solid #ddd;\">蟻丘、樹幹裂縫</td></tr><tr><td style=\"padding:6px;border:1px solid #ddd;\">白蟻</td><td style=\"padding:6px;border:1px solid #ddd;\">極高蛋白質、含脂肪</td><td style=\"padding:6px;border:1px solid #ddd;\">可生食或輕烤</td><td style=\"padding:6px;border:1px solid #ddd;\">枯木內部、蟻塚</td></tr><tr><td style=\"padding:6px;border:1px solid #ddd;\">甲蟲幼蟲（雞母蟲）</td><td style=\"padding:6px;border:1px solid #ddd;\">高脂肪高蛋白</td><td style=\"padding:6px;border:1px solid #ddd;\">去腸（擠壓排出內容物）→ 烤熟</td><td style=\"padding:6px;border:1px solid #ddd;\">腐木內、土壤中</td></tr><tr><td style=\"padding:6px;border:1px solid #ddd;\">蜻蜓</td><td style=\"padding:6px;border:1px solid #ddd;\">蛋白質</td><td style=\"padding:6px;border:1px solid #ddd;\">去翅 → 烤食</td><td style=\"padding:6px;border:1px solid #ddd;\">水邊</td></tr></table><p style=\"color:#CC0000;font-size:13px;\"><b>❌ 避免：</b>色彩鮮艷的昆蟲（警告色=有毒）、有刺毛的毛毛蟲、有惡臭的甲蟲、蜜蜂/虎頭蜂（被螫風險）、蜱蟲、蜘蛛</p><div style=\"margin-top:8px;padding:8px;background:#FFF3F3;border-radius:4px;\"><b>⚠️ 注意事項：</b><ul style=\"margin:4px 0;\"><li style=\"color:#CC0000;\">所有昆蟲建議烤熟食用（殺死寄生蟲）</li><li style=\"color:#CC0000;\">第一次少量食用，觀察過敏反應</li><li style=\"color:#CC0000;\">昆蟲含幾丁質，消化能力差者不宜大量</li></ul></div></div><div style=\"border:1px solid #ddd;border-radius:8px;padding:16px;margin:10px 0;background:#FAFAFA;\"><h3 style=\"margin:0 0 8px 0;\">生火方法</h3><p style=\"color:#666;font-size:13px;\"><b>適用情境：</b>需要火源用於煮沸水、烹飪食物、保暖、驅蟲</p><p style=\"color:#666;font-size:13px;\"><b>所需材料：</b>乾燥引火物/乾燥木柴/生火工具</p><p style=\"font-size:13px;\">不論哪種方法，都需要先準備好三種燃料：<br>1. 火種（乾燥細碎物）：乾草、樹皮絲、棉絮、鳥巢材料<br>2. 引火物（細枝）：鉛筆粗的乾燥小樹枝<br>3. 燃料（粗柴）：手腕粗的乾燥木頭</p><div style=\"margin:8px 0;padding:10px;background:#F0F8FF;border-radius:6px;border-left:4px solid #4488CC;\"><b>弓鑽法</b> <span style=\"font-size:12px;color:#888;\">[中高]</span><br>1. 取一根彎曲樹枝做弓（綁上鞋帶或繩索）<br>2. 取乾燥硬木做鑽桿（約30cm，筆直）<br>3. 取一塊乾燥軟木做底板，刻V形凹槽<br>4. 用石頭或硬木做壓頂（手持壓在鑽桿頂端）<br>5. 鑽桿纏繞弓弦一圈，放入底板凹槽<br>6. 快速拉弓使鑽桿旋轉，持續30-60秒<br>7. 產生的黑色粉末開始冒煙時，將火星倒入火種中<br>8. 輕吹火種使其燃燒<br><span style=\"color:#666;font-size:12px;\">💡 最可靠的無工具生火法，但需要練習</span></div><div style=\"margin:8px 0;padding:10px;background:#F0F8FF;border-radius:6px;border-left:4px solid #4488CC;\"><b>手鑽法</b> <span style=\"font-size:12px;color:#888;\">[高]</span><br>1. 取直而乾燥的細木棍做鑽桿<br>2. 底板同弓鑽法<br>3. 雙手夾住鑽桿快速搓動<br>4. 施加向下壓力同時搓動<br>5. 持續至冒煙<br><span style=\"color:#666;font-size:12px;\">💡 比弓鑽法更累，手容易磨傷</span></div><div style=\"margin:8px 0;padding:10px;background:#F0F8FF;border-radius:6px;border-left:4px solid #4488CC;\"><b>打火石法</b> <span style=\"font-size:12px;color:#888;\">[中]</span><br>1. 找到含石英的堅硬石頭（燧石、石英岩）<br>2. 用刀背或另一塊石頭用力刮擊<br>3. 產生的火星落在乾燥火種上<br>4. 輕吹使其燃燒<br><span style=\"color:#666;font-size:12px;\">💡 需要合適的石頭，但成功率較高</span></div><div style=\"margin:8px 0;padding:10px;background:#F0F8FF;border-radius:6px;border-left:4px solid #4488CC;\"><b>電池短路法</b> <span style=\"font-size:12px;color:#888;\">[低]</span><br>1. 取出電池（手電筒或電子裝置中的）<br>2. 用細金屬線（口香糖錫箔紙中間撕細）連接正負極<br>3. 金屬線會發熱甚至燃燒<br>4. 接觸火種引燃<br><span style=\"color:#666;font-size:12px;\">💡 ⚠️ 僅緊急使用，會消耗電池且有燒傷風險</span></div><div style=\"margin:8px 0;padding:10px;background:#F0F8FF;border-radius:6px;border-left:4px solid #4488CC;\"><b>透鏡聚光法</b> <span style=\"font-size:12px;color:#888;\">[低]</span><br>1. 用眼鏡鏡片、放大鏡、甚至裝水的透明塑膠袋<br>2. 將陽光聚焦成一點<br>3. 對準乾燥深色火種<br>4. 保持穩定直到冒煙燃燒<br><span style=\"color:#666;font-size:12px;\">💡 需要晴天和充足陽光</span></div><div style=\"margin-top:8px;padding:8px;background:#FFF3F3;border-radius:4px;\"><b>⚠️ 注意事項：</b><ul style=\"margin:4px 0;\"><li style=\"color:#CC0000;\">生火前清除周圍可燃物（落葉、乾草）2公尺範圍</li><li style=\"color:#CC0000;\">風大時用石頭圍擋風</li><li style=\"color:#CC0000;\">離開前務必完全熄滅</li><li style=\"color:#CC0000;\">潮濕天氣需尋找樹皮內層等乾燥引火物</li></ul></div></div><div style=\"border:1px solid #ddd;border-radius:8px;padding:16px;margin:10px 0;background:#FAFAFA;\"><h3 style=\"margin:0 0 8px 0;\">野外烹飪方式</h3><p style=\"color:#666;font-size:13px;\"><b>適用情境：</b>有了火源和食材，需要烹飪</p><p style=\"color:#666;font-size:13px;\"><b>所需材料：</b>火源/容器（竹筒或石頭凹處）/食材</p><div style=\"margin:8px 0;padding:10px;background:#F0F8FF;border-radius:6px;border-left:4px solid #4488CC;\"><b>石板烤</b><br>找一塊扁平石頭放在火上加熱 15-20 分鐘，石板變乾後放上食物烤熟。<br><span style=\"color:#CC6600;font-size:13px;\">⚠️ ⚠️ 避免使用河邊取的石頭（含水分可能爆裂）</span></div><div style=\"margin:8px 0;padding:10px;background:#F0F8FF;border-radius:6px;border-left:4px solid #4488CC;\"><b>竹筒飯/竹筒煮水</b><br>取新鮮竹子（一端保留竹節封底），放入食材和水，開口端用葉子塞住，斜靠在火旁。<br><span style=\"color:#CC6600;font-size:13px;\">⚠️ 竹筒會慢慢碳化但水不會讓它燒穿</span></div><div style=\"margin:8px 0;padding:10px;background:#F0F8FF;border-radius:6px;border-left:4px solid #4488CC;\"><b>坑烤法</b><br>1. 挖坑放入石頭<br>2. 在石頭上燒火 1 小時（讓石頭蓄熱）<br>3. 移開火和灰，放上包覆大葉的食物<br>4. 蓋上土埋 2-3 小時<br><span style=\"color:#CC6600;font-size:13px;\">⚠️ 耗時但很均勻，適合烤肉或根莖</span></div><div style=\"margin:8px 0;padding:10px;background:#F0F8FF;border-radius:6px;border-left:4px solid #4488CC;\"><b>煮湯/煮水</b><br>用竹筒、金屬容器或天然凹穴石頭盛水，直接加熱或投入燒紅石頭。<br><span style=\"color:#CC6600;font-size:13px;\">⚠️ 投入熱石頭可在不耐火容器中煮沸水</span></div><div style=\"margin:8px 0;padding:10px;background:#F0F8FF;border-radius:6px;border-left:4px solid #4488CC;\"><b>串烤</b><br>用新鮮綠色木棍串起食材，架在火旁慢烤。定時翻面。<br><span style=\"color:#CC6600;font-size:13px;\">⚠️ 使用綠枝（有水分）避免棍子燒斷</span></div></div><div style=\"border:1px solid #ddd;border-radius:8px;padding:16px;margin:10px 0;background:#FAFAFA;\"><h3 style=\"margin:0 0 8px 0;\">簡易陷阱製作</h3><p style=\"color:#666;font-size:13px;\"><b>適用情境：</b>需要捕獲小型動物作為蛋白質來源</p><p style=\"color:#666;font-size:13px;\"><b>所需材料：</b>繩索/細枝/石頭</p><div style=\"margin:8px 0;padding:10px;background:#F0F8FF;border-radius:6px;border-left:4px solid #4488CC;\"><b>簡易落套</b><br>用細繩或金屬絲做活套，放在動物經過的小徑上，套圈直徑約拳頭大。固定另一端在彎曲的樹枝上（彈力觸發）。</div><div style=\"margin:8px 0;padding:10px;background:#F0F8FF;border-radius:6px;border-left:4px solid #4488CC;\"><b>落石陷阱（四字觸發器）</b><br>用三根木棍組成「4」字形觸發機構，上方架重石。動物觸碰觸發棍時石頭落下。</div><div style=\"margin:8px 0;padding:10px;background:#F0F8FF;border-radius:6px;border-left:4px solid #4488CC;\"><b>魚笱</b><br>用竹片或細枝編成漏斗狀入口的籠子，放入溪流中。魚進得去出不來。</div><div style=\"margin-top:8px;padding:8px;background:#FFF3F3;border-radius:4px;\"><b>⚠️ 注意事項：</b><ul style=\"margin:4px 0;\"><li style=\"color:#CC0000;\">設置多個陷阱提高成功率</li><li style=\"color:#CC0000;\">每 12 小時檢查一次</li><li style=\"color:#CC0000;\">非求生狀態下捕獵可能違法</li></ul></div></div>"},"water":{"title":"水","icon":"💧","html":"<div style=\"border:1px solid #ddd;border-radius:8px;padding:16px;margin:10px 0;background:#FAFAFA;\"><h3 style=\"margin:0 0 8px 0;\">水源尋找法</h3><p style=\"color:#666;font-size:13px;\"><b>適用情境：</b>附近沒有明顯水源，需要尋找地下水、溪流或其他水源</p><p style=\"color:#666;font-size:13px;\"><b>所需材料：</b>無特殊需求</p><ol style=\"line-height:2;\"><li><b>安靜下來，仔細聆聽周圍有無流水聲</b><br><span style=\"color:#666;font-size:13px;\">清晨和傍晚最安靜，水聲傳最遠</span></li><li><b>觀察地形：水往低處流，沿坡向下尋找</b><br><span style=\"color:#666;font-size:13px;\">山谷底部、兩山交會處最可能有水</span></li><li><b>尋找指示植物：蕨類密集處、蘆葦、水生植物附近有水</b><br><span style=\"color:#666;font-size:13px;\">植物越翠綠茂密，離水越近</span></li><li><b>觀察動物足跡和飛行路線</b><br><span style=\"color:#666;font-size:13px;\">清晨和傍晚動物會朝水源方向移動；蜜蜂活動範圍離水源通常不超過 5 公里</span></li><li><b>在乾涸河床或低窪處挖掘 30-60cm</b><br><span style=\"color:#666;font-size:13px;\">河床彎道外側最容易挖到地下水</span></li><li><b>注意岩石表面的水漬或苔蘚分布</b><br><span style=\"color:#666;font-size:13px;\">岩石縫隙常有滲水，苔蘚最潮濕處可能有泉眼</span></li></ol><div style=\"margin-top:8px;padding:8px;background:#FFF3F3;border-radius:4px;\"><b>⚠️ 注意事項：</b><ul style=\"margin:4px 0;\"><li style=\"color:#CC0000;\">找到水源後務必評估安全性（見「水源安全 Checklist」）</li><li style=\"color:#CC0000;\">死水比流水風險更高</li><li style=\"color:#CC0000;\">農田附近的水可能有農藥污染</li></ul></div></div><div style=\"border:1px solid #ddd;border-radius:8px;padding:16px;margin:10px 0;background:#FAFAFA;\"><h3 style=\"margin:0 0 8px 0;\">雨水收集法</h3><p style=\"color:#666;font-size:13px;\"><b>適用情境：</b>下雨時需要收集雨水飲用或儲存</p><p style=\"color:#666;font-size:13px;\"><b>所需材料：</b>塑膠袋/雨衣/大葉植物/容器</p><ol style=\"line-height:2;\"><li><b>塑膠袋法：將塑膠袋撐開放在地上或綁在樹枝上形成漏斗</b><br><span style=\"color:#666;font-size:13px;\">袋角剪小口導流至容器</span></li><li><b>布匹導流法：將衣物或布匹攤開在傾斜面上，底部集中導入容器</b><br><span style=\"color:#666;font-size:13px;\">T恤、雨衣、帳篷布都可用</span></li><li><b>大葉植物漏斗法：將姑婆芋等大型葉片重疊排列成漏斗形</b><br><span style=\"color:#666;font-size:13px;\">注意：僅用葉片集水，勿讓葉汁混入（姑婆芋有毒）</span></li><li><b>坑洞集水法：在地面挖淺坑，鋪上塑膠袋或大葉防滲</b><br><span style=\"color:#666;font-size:13px;\">面積越大，收集越多</span></li><li><b>初期雨水（前 5-10 分鐘）盡量不收集</b><br><span style=\"color:#666;font-size:13px;\">初期雨水會沖刷空氣和表面灰塵，較髒</span></li><li><b>收集後靜置 30 分鐘讓雜質沉底</b><br><span style=\"color:#666;font-size:13px;\">小心取上層清水飲用</span></li></ol><div style=\"margin-top:8px;padding:8px;background:#FFF3F3;border-radius:4px;\"><b>⚠️ 注意事項：</b><ul style=\"margin:4px 0;\"><li style=\"color:#CC0000;\">初期雨水避免直接飲用</li><li style=\"color:#CC0000;\">金屬表面集水可能含重金屬</li><li style=\"color:#CC0000;\">雨水相對安全但仍建議煮沸</li></ul></div></div><div style=\"border:1px solid #ddd;border-radius:8px;padding:16px;margin:10px 0;background:#FAFAFA;\"><h3 style=\"margin:0 0 8px 0;\">露水收集法</h3><p style=\"color:#666;font-size:13px;\"><b>適用情境：</b>無降雨但清晨有露水，需要少量飲用水</p><p style=\"color:#666;font-size:13px;\"><b>所需材料：</b>布匹/塑膠袋/吸水材料</p><ol style=\"line-height:2;\"><li><b>布匹擦拭法：日出前用乾淨布匹在草地上來回拖行，擰乾入容器</b><br><span style=\"color:#666;font-size:13px;\">一次可收集 200-500ml，需在太陽出來前完成</span></li><li><b>塑膠袋套植物法：前一晚將透明塑膠袋套在樹枝末端綁緊</b><br><span style=\"color:#666;font-size:13px;\">植物蒸散作用會在袋內凝結水珠，早上可收集 50-200ml/袋</span></li><li><b>在光滑表面（石頭、金屬）上鋪布吸收後擰乾</b><br><span style=\"color:#666;font-size:13px;\">光滑表面凝結露水效率最高</span></li><li><b>選擇開闊草地而非樹蔭下操作</b><br><span style=\"color:#666;font-size:13px;\">開闊處溫差大，露水更多</span></li></ol><div style=\"margin-top:8px;padding:8px;background:#FFF3F3;border-radius:4px;\"><b>⚠️ 注意事項：</b><ul style=\"margin:4px 0;\"><li style=\"color:#CC0000;\">產量有限，只能應急</li><li style=\"color:#CC0000;\">避免在有農藥噴灑的區域收集</li><li style=\"color:#CC0000;\">搭配其他方法使用</li></ul></div></div><div style=\"border:1px solid #ddd;border-radius:8px;padding:16px;margin:10px 0;background:#FAFAFA;\"><h3 style=\"margin:0 0 8px 0;\">淨水處理法</h3><p style=\"color:#666;font-size:13px;\"><b>適用情境：</b>取得水源但不確定是否安全飲用</p><p style=\"color:#666;font-size:13px;\"><b>所需材料：</b>火源/容器/布匹/沙土/木炭</p><ol style=\"line-height:2;\"><li><b>【首選】煮沸法：將水煮沸並持續滾沸至少 1 分鐘（海拔 2000m 以上滾沸 3 分鐘）</b><br><span style=\"color:#666;font-size:13px;\">最有效的淨水方式，可殺死 99.9% 病原體</span></li><li><b>【備選】太陽曝曬法（SODIS）：將透明寶特瓶裝滿水，放在陽光下曝曬 6 小時（陰天 2 天）</b><br><span style=\"color:#666;font-size:13px;\">UV 光可殺死大部分細菌和病毒</span></li><li><b>【過濾】三層過濾器製作：找容器底部穿孔，由上至下放入：細沙→木炭碎→粗沙石</b><br><span style=\"color:#666;font-size:13px;\">木炭可從營火取得，越細碎過濾效果越好。過濾後仍建議煮沸。</span></li><li><b>【應急】布匹過濾：至少折疊 4 層布匹過濾，可去除大部分懸浮物和寄生蟲卵</b><br><span style=\"color:#666;font-size:13px;\">無法去除細菌和病毒，必須搭配其他方式</span></li><li><b>【植物法】用剝皮的仙人掌肉或木瓜種子搗碎加入水中可幫助沉澱雜質</b><br><span style=\"color:#666;font-size:13px;\">純輔助手段，仍需煮沸</span></li></ol><div style=\"margin-top:8px;padding:8px;background:#FFF3F3;border-radius:4px;\"><b>⚠️ 注意事項：</b><ul style=\"margin:4px 0;\"><li style=\"color:#CC0000;\">過濾不等於淨化，只有煮沸能確保安全</li><li style=\"color:#CC0000;\">化學污染（農藥、重金屬）無法靠煮沸去除</li><li style=\"color:#CC0000;\">海水和尿液不可直接飲用（需蒸餾）</li></ul></div></div><div style=\"border:1px solid #ddd;border-radius:8px;padding:16px;margin:10px 0;background:#FAFAFA;\"><h3 style=\"margin:0 0 8px 0;\">水源安全 Checklist</h3><p style=\"color:#666;font-size:13px;\"><b>適用情境：</b>發現水源，需要判斷是否可安全飲用</p><p style=\"color:#666;font-size:13px;\"><b>所需材料：</b>無</p><table style=\"width:100%;border-collapse:collapse;font-size:13px;margin:8px 0;\"><tr style=\"background:#f0f0f0;\"><th style=\"padding:6px;\">項目</th><th>✅ 安全</th><th>⚠️ 注意</th><th>❌ 危險</th></tr><tr><td style=\"padding:6px;border:1px solid #ddd;\"><b>水的清澈度</b></td><td style=\"padding:6px;border:1px solid #ddd;background:#F0FFF4;\">清澈見底</td><td style=\"padding:6px;border:1px solid #ddd;background:#FFFBE6;\">微混濁</td><td style=\"padding:6px;border:1px solid #ddd;background:#FFF0F0;\">嚴重混濁或有顏色</td></tr><tr><td style=\"padding:6px;border:1px solid #ddd;\"><b>流動性</b></td><td style=\"padding:6px;border:1px solid #ddd;background:#F0FFF4;\">流動的溪水</td><td style=\"padding:6px;border:1px solid #ddd;background:#FFFBE6;\">緩慢流動</td><td style=\"padding:6px;border:1px solid #ddd;background:#FFF0F0;\">完全靜止的死水</td></tr><tr><td style=\"padding:6px;border:1px solid #ddd;\"><b>氣味</b></td><td style=\"padding:6px;border:1px solid #ddd;background:#F0FFF4;\">無氣味</td><td style=\"padding:6px;border:1px solid #ddd;background:#FFFBE6;\">輕微泥土味</td><td style=\"padding:6px;border:1px solid #ddd;background:#FFF0F0;\">臭味、化學味、腐敗味</td></tr><tr><td style=\"padding:6px;border:1px solid #ddd;\"><b>周圍環境</b></td><td style=\"padding:6px;border:1px solid #ddd;background:#F0FFF4;\">遠離人類活動區</td><td style=\"padding:6px;border:1px solid #ddd;background:#FFFBE6;\">附近有農田</td><td style=\"padding:6px;border:1px solid #ddd;background:#FFF0F0;\">附近有工廠、垃圾場、死動物</td></tr><tr><td style=\"padding:6px;border:1px solid #ddd;\"><b>水面</b></td><td style=\"padding:6px;border:1px solid #ddd;background:#F0FFF4;\">無漂浮物</td><td style=\"padding:6px;border:1px solid #ddd;background:#FFFBE6;\">少量落葉</td><td style=\"padding:6px;border:1px solid #ddd;background:#FFF0F0;\">油膜、泡沫、藻類大量繁殖</td></tr><tr><td style=\"padding:6px;border:1px solid #ddd;\"><b>水邊生物</b></td><td style=\"padding:6px;border:1px solid #ddd;background:#F0FFF4;\">有魚蝦等水生生物</td><td style=\"padding:6px;border:1px solid #ddd;background:#FFFBE6;\">只有少量生物</td><td style=\"padding:6px;border:1px solid #ddd;background:#FFF0F0;\">完全無生物或有死魚</td></tr></table><div style=\"padding:8px;margin:4px 0;background:#F5F5F5;border-radius:4px;border-left:4px solid #888;\"><b>全部 safe</b> → 低風險，建議煮沸後飲用</div><div style=\"padding:8px;margin:4px 0;background:#F5F5F5;border-radius:4px;border-left:4px solid #888;\"><b>有任何 warning</b> → 中風險，必須煮沸+過濾</div><div style=\"padding:8px;margin:4px 0;background:#F5F5F5;border-radius:4px;border-left:4px solid #888;\"><b>有任何 danger</b> → 高風險，避免飲用。若不得已，三層過濾+煮沸，且僅少量飲用</div><div style=\"margin-top:8px;padding:8px;background:#FFF3F3;border-radius:4px;\"><b>⚠️ 注意事項：</b><ul style=\"margin:4px 0;\"><li style=\"color:#CC0000;\">此 Checklist 僅供野外緊急參考</li><li style=\"color:#CC0000;\">任何有疑慮的水都應該煮沸</li><li style=\"color:#CC0000;\">化學污染無法用肉眼判斷</li></ul></div></div>"},"medical":{"title":"醫","icon":"🏥","html":"<div style=\"border:1px solid #ddd;border-radius:8px;padding:16px;margin:10px 0;background:#FAFAFA;\"><h3 style=\"margin:0 0 8px 0;\">傷口處理通則</h3><p style=\"color:#666;font-size:13px;\"><b>適用情境：</b>身體有開放性傷口（割傷、擦傷、刺傷等）</p><p style=\"color:#666;font-size:13px;\"><b>所需材料：</b>清水/布匹或紗布/（如有）消毒劑</p><ol style=\"line-height:2;\"><li><b>止血：用乾淨布料直接壓迫傷口 10-15 分鐘</b><br><span style=\"color:#666;font-size:13px;\">手臂抬高過心臟有助止血</span></li><li><b>清洗：用大量清水（煮沸冷卻後最佳）沖洗傷口</b><br><span style=\"color:#666;font-size:13px;\">目的是沖掉汙染物和細菌，沖洗比消毒更重要</span></li><li><b>去除異物：可見的砂石碎片用乾淨工具夾出</b><br><span style=\"color:#666;font-size:13px;\">深入的異物不要硬挖，避免加重傷害</span></li><li><b>敷藥（如有）：消毒藥水或辨識為消炎植物搗碎後敷上</b><br><span style=\"color:#666;font-size:13px;\">→ 見「藥用植物敷用法」</span></li><li><b>包紮：用乾淨布料包覆傷口，適度加壓但不影響血液循環</b><br><span style=\"color:#666;font-size:13px;\">檢查末端手指/腳趾是否發紫（太緊）</span></li><li><b>觀察：每 6-12 小時檢查傷口狀態</b><br><span style=\"color:#666;font-size:13px;\">→ 見「傷口發炎判斷 Checklist」</span></li></ol><div style=\"margin-top:8px;padding:8px;background:#FFF3F3;border-radius:4px;\"><b>⚠️ 注意事項：</b><ul style=\"margin:4px 0;\"><li style=\"color:#CC0000;\">大量出血（噴射狀）需要立即止血帶或壓迫止血</li><li style=\"color:#CC0000;\">動物咬傷需考慮狂犬病風險</li><li style=\"color:#CC0000;\">傷口超過 6 小時未處理感染風險大增</li></ul></div></div><div style=\"border:1px solid #ddd;border-radius:8px;padding:16px;margin:10px 0;background:#FAFAFA;\"><h3 style=\"margin:0 0 8px 0;\">傷口發炎判斷 Checklist</h3><p style=\"color:#666;font-size:13px;\"><b>適用情境：</b>傷口已處理一段時間，需要判斷是否感染發炎</p><p style=\"color:#666;font-size:13px;\"><b>所需材料：</b>無</p><table style=\"width:100%;border-collapse:collapse;font-size:13px;margin:8px 0;\"><tr style=\"background:#f0f0f0;\"><th style=\"padding:6px;\">檢查項目</th><th>✅ 正常</th><th>⚠️ 異常</th></tr><tr><td style=\"padding:8px;border:1px solid #ddd;\"><b>紅</b><br><span style=\"font-size:12px;color:#666;\">傷口周圍是否紅腫？紅腫範圍是否擴大？</span></td><td style=\"padding:8px;border:1px solid #ddd;background:#F0FFF4;\">輕微紅腫（正常癒合反應）</td><td style=\"padding:8px;border:1px solid #ddd;background:#FFF0F0;\">紅腫範圍擴大或出現紅色線條（向心臟方向）</td></tr><tr><td style=\"padding:8px;border:1px solid #ddd;\"><b>腫</b><br><span style=\"font-size:12px;color:#666;\">傷口周圍是否明顯腫脹？</span></td><td style=\"padding:8px;border:1px solid #ddd;background:#F0FFF4;\">輕微腫脹</td><td style=\"padding:8px;border:1px solid #ddd;background:#FFF0F0;\">嚴重腫脹且持續惡化</td></tr><tr><td style=\"padding:8px;border:1px solid #ddd;\"><b>熱</b><br><span style=\"font-size:12px;color:#666;\">用手背觸摸傷口周圍，是否明顯比其他皮膚燙？</span></td><td style=\"padding:8px;border:1px solid #ddd;background:#F0FFF4;\">略溫</td><td style=\"padding:8px;border:1px solid #ddd;background:#FFF0F0;\">明顯發燙</td></tr><tr><td style=\"padding:8px;border:1px solid #ddd;\"><b>痛</b><br><span style=\"font-size:12px;color:#666;\">疼痛是否加劇而非減輕？</span></td><td style=\"padding:8px;border:1px solid #ddd;background:#F0FFF4;\">疼痛逐漸減輕</td><td style=\"padding:8px;border:1px solid #ddd;background:#FFF0F0;\">疼痛加劇或出現跳動性疼痛</td></tr><tr><td style=\"padding:8px;border:1px solid #ddd;\"><b>膿</b><br><span style=\"font-size:12px;color:#666;\">是否有膿液（黃色/綠色黏稠液體）流出？</span></td><td style=\"padding:8px;border:1px solid #ddd;background:#F0FFF4;\">透明或微黃血清液體</td><td style=\"padding:8px;border:1px solid #ddd;background:#FFF0F0;\">黃色/綠色有臭味的膿液</td></tr><tr><td style=\"padding:8px;border:1px solid #ddd;\"><b>全身</b><br><span style=\"font-size:12px;color:#666;\">是否出現發燒、寒顫、全身無力？</span></td><td style=\"padding:8px;border:1px solid #ddd;background:#F0FFF4;\">無全身症狀</td><td style=\"padding:8px;border:1px solid #ddd;background:#FFF0F0;\">有 → 感染可能已擴散（嚴重）</td></tr></table><div style=\"padding:8px;margin:4px 0;background:#F0FFF4;border-radius:4px;border-left:4px solid #888;\"><b>全部 normal</b> → 傷口正常癒合中，持續保持清潔乾燥<br><span style=\"font-size:13px;\">👉 繼續每天換藥觀察</span></div><div style=\"padding:8px;margin:4px 0;background:#FFFBE6;border-radius:4px;border-left:4px solid #888;\"><b>1-2 項 warning</b> → 輕度感染徵兆<br><span style=\"font-size:13px;\">👉 增加換藥頻率（每 4-6 小時）、使用消炎植物敷用、保持傷口引流通暢</span></div><div style=\"padding:8px;margin:4px 0;background:#FFF0F0;border-radius:4px;border-left:4px solid #888;\"><b>3+ 項 warning</b> → 明顯感染<br><span style=\"font-size:13px;\">👉 積極清洗傷口、頻繁換藥、尋找更強效消炎植物、想辦法獲得醫療救助</span></div><div style=\"padding:8px;margin:4px 0;background:#FF0000;border-radius:4px;border-left:4px solid #888;\"><b>有全身症狀</b> → 嚴重感染（可能敗血症）<br><span style=\"font-size:13px;\">👉 ⚠️ 生命危險！必須盡快獲得醫療救助。持續清潔傷口、大量飲水、保持體力</span></div></div><div style=\"border:1px solid #ddd;border-radius:8px;padding:16px;margin:10px 0;background:#FAFAFA;\"><h3 style=\"margin:0 0 8px 0;\">扭傷骨折處置</h3><p style=\"color:#666;font-size:13px;\"><b>適用情境：</b>肢體扭傷腫脹或疑似骨折</p><p style=\"color:#666;font-size:13px;\"><b>所需材料：</b>冰水或冷水/布匹/直木棍（夾板用）</p><ol style=\"line-height:2;\"><li><b>R — Rest 休息：停止活動受傷部位</b><br><span style=\"color:#666;font-size:13px;\">不要試圖「走走看」</span></li><li><b>I — Ice 冰敷：用冷水浸濕布匹敷在患處 15-20 分鐘</b><br><span style=\"color:#666;font-size:13px;\">沒有冰塊時用溪水浸泡也有效，每小時一次</span></li><li><b>C — Compression 加壓：用布匹適度纏繞患處</b><br><span style=\"color:#666;font-size:13px;\">不要太緊，檢查末端是否發紫</span></li><li><b>E — Elevation 抬高：將受傷部位抬高過心臟</b><br><span style=\"color:#666;font-size:13px;\">減少腫脹</span></li></ol><div style=\"margin:8px 0;padding:10px;background:#F0F8FF;border-radius:6px;\"><b>🦴 夾板固定：</b><br>如疑似骨折：<br>1. 不要嘗試復位<br>2. 取兩根直木棍放在傷處兩側<br>3. 用布條在骨折處上下方各綁一道固定<br>4. 木棍長度要超過上下兩個關節<br>5. 固定後不要移動骨折部位</div><div style=\"margin-top:8px;padding:8px;background:#FFF3F3;border-radius:4px;\"><b>⚠️ 注意事項：</b><ul style=\"margin:4px 0;\"><li style=\"color:#CC0000;\">開放性骨折（骨頭外露）：先處理傷口再固定</li><li style=\"color:#CC0000;\">手指末端發白/發紫表示綁太緊</li><li style=\"color:#CC0000;\">骨折需要專業醫療，固定只是緊急處理</li></ul></div></div><div style=\"border:1px solid #ddd;border-radius:8px;padding:16px;margin:10px 0;background:#FAFAFA;\"><h3 style=\"margin:0 0 8px 0;\">藥用植物敷用法</h3><p style=\"color:#666;font-size:13px;\"><b>適用情境：</b>已辨識出藥用植物，需要正確處理後敷用在傷口</p><p style=\"color:#666;font-size:13px;\"><b>所需材料：</b>辨識過的藥用植物/清水/乾淨石頭/大葉或布匹</p><ol style=\"line-height:2;\"><li><b>確認植物已經 AI 辨識確認為藥用</b><br><span style=\"color:#666;font-size:13px;\">⚠️ 絕對不要使用未辨識的植物敷傷口</span></li><li><b>用清水徹底沖洗植物（去除泥土、蟲卵）</b><br><span style=\"color:#666;font-size:13px;\">至少沖洗 30 秒</span></li><li><b>取乾淨平坦石頭作為研磨面</b><br><span style=\"color:#666;font-size:13px;\">先用水沖洗石頭表面</span></li><li><b>將植物葉片/莖部放在石面上，用另一塊石頭反覆搗碎成泥狀</b><br><span style=\"color:#666;font-size:13px;\">越碎越好，讓汁液充分釋出</span></li><li><b>先用清水清洗傷口</b><br><span style=\"color:#666;font-size:13px;\">確保傷口表面乾淨</span></li><li><b>將植物泥均勻敷在傷口周圍（不要塞入深層傷口內）</b><br><span style=\"color:#666;font-size:13px;\">敷用厚度約 3-5mm</span></li><li><b>用大葉（月桃葉等）或乾淨布匹覆蓋固定</b><br><span style=\"color:#666;font-size:13px;\">固定但不加壓</span></li><li><b>每 4-6 小時更換新鮮植物泥</b><br><span style=\"color:#666;font-size:13px;\">更換時觀察傷口變化</span></li></ol><div style=\"margin-top:8px;padding:8px;background:#FFF3F3;border-radius:4px;\"><b>⚠️ 注意事項：</b><ul style=\"margin:4px 0;\"><li style=\"color:#CC0000;\">傷口如持續惡化（更紅更腫）→ 停止使用該植物</li><li style=\"color:#CC0000;\">深層傷口不適合敷用植物（感染風險）</li><li style=\"color:#CC0000;\">此為緊急措施，有機會獲得醫療時立即就醫</li></ul></div></div><div style=\"border:1px solid #ddd;border-radius:8px;padding:16px;margin:10px 0;background:#FAFAFA;\"><h3 style=\"margin:0 0 8px 0;\">燒燙傷處理</h3><p style=\"color:#666;font-size:13px;\"><b>適用情境：</b>被火焰、熱水、熱石板燙傷</p><p style=\"color:#666;font-size:13px;\"><b>所需材料：</b>大量清水/乾淨覆蓋物</p><ol style=\"line-height:2;\"><li><b>立即用大量清涼水沖洗患處至少 15-20 分鐘</b><br><span style=\"color:#666;font-size:13px;\">這是最重要的步驟，越快越好</span></li><li><b>移除患處的飾品、衣物（如未沾黏）</b><br><span style=\"color:#666;font-size:13px;\">腫脹前移除，否則之後無法取下</span></li><li><b>用乾淨濕布覆蓋患處</b><br><span style=\"color:#666;font-size:13px;\">保持濕潤減少疼痛</span></li><li><b>不要刺破水泡</b><br><span style=\"color:#666;font-size:13px;\">水泡是天然保護層</span></li><li><b>包紮時不要加壓</b><br><span style=\"color:#666;font-size:13px;\">輕鬆覆蓋即可</span></li></ol><div style=\"margin-top:8px;padding:8px;background:#FFF3F3;border-radius:4px;\"><b>⚠️ 注意事項：</b><ul style=\"margin:4px 0;\"><li style=\"color:#CC0000;\">❌ 勿塗抹牙膏、醬油、蛋白等偏方</li><li style=\"color:#CC0000;\">❌ 勿用冰塊直接接觸（會加重傷害）</li><li style=\"color:#CC0000;\">❌ 勿刺破水泡</li><li style=\"color:#CC0000;\">大面積燒傷（超過手掌大小）為嚴重傷害，需盡快就醫</li></ul></div></div><div style=\"border:1px solid #ddd;border-radius:8px;padding:16px;margin:10px 0;background:#FAFAFA;\"><h3 style=\"margin:0 0 8px 0;\">止血植物使用法</h3><p style=\"color:#666;font-size:13px;\"><b>適用情境：</b>傷口出血需要輔助止血</p><p style=\"color:#666;font-size:13px;\"><b>所需材料：</b>車前草或其他止血植物/壓迫用布匹</p><ol style=\"line-height:2;\"><li><b>先用手或布匹直接壓迫止血（這才是主要止血方式）</b><br><span style=\"color:#666;font-size:13px;\">持續壓迫 10-15 分鐘不放開</span></li><li><b>同時取得止血植物（車前草、艾草）</b><br><span style=\"color:#666;font-size:13px;\">必須是 AI 確認的種類</span></li><li><b>將植物葉片揉碎出汁</b><br><span style=\"color:#666;font-size:13px;\">不需完全搗泥，揉出汁液即可</span></li><li><b>直接將揉碎的葉片壓在傷口上</b><br><span style=\"color:#666;font-size:13px;\">植物汁液有輔助凝血作用</span></li><li><b>外層再用布匹加壓固定</b><br><span style=\"color:#666;font-size:13px;\">壓力才是止血的主力</span></li></ol><div style=\"margin-top:8px;padding:8px;background:#FFF3F3;border-radius:4px;\"><b>⚠️ 注意事項：</b><ul style=\"margin:4px 0;\"><li style=\"color:#CC0000;\">直接壓迫永遠是第一優先的止血方式</li><li style=\"color:#CC0000;\">植物只是輔助，不能替代壓迫</li><li style=\"color:#CC0000;\">噴射狀出血（動脈出血）需要止血帶</li></ul></div></div>"},"shelter":{"title":"住","icon":"🏕️","html":"<div style=\"border:1px solid #ddd;border-radius:8px;padding:16px;margin:10px 0;background:#FAFAFA;\"><h3 style=\"margin:0 0 8px 0;\">庇護所選址原則</h3><p style=\"color:#666;font-size:13px;\"><b>適用情境：</b>需要在野外找到適合搭建庇護所或休息的地點</p><p style=\"color:#666;font-size:13px;\"><b>所需材料：</b>無</p><ol style=\"line-height:2;\"><li><b>尋找高於周圍地面的位置</b><br><span style=\"color:#666;font-size:13px;\">避免低窪處（積水、冷空氣下沉）</span></li><li><b>遠離水源至少 50 公尺</b><br><span style=\"color:#666;font-size:13px;\">水邊蚊蟲多、有暴漲風險</span></li><li><b>避風但不要在風口或山稜線上</b><br><span style=\"color:#666;font-size:13px;\">利用地形或大樹作為擋風</span></li><li><b>檢查頭頂：無枯枝（寡婦製造者）、無蜂巢、無落石風險</b><br><span style=\"color:#666;font-size:13px;\">搖晃頭頂樹枝測試是否穩固</span></li><li><b>地面平坦、無尖石、無蟻丘</b><br><span style=\"color:#666;font-size:13px;\">清除地面落葉雜物後檢查</span></li><li><b>附近有建材可取（樹枝、大葉、藤蔓）</b><br><span style=\"color:#666;font-size:13px;\">減少搬運消耗體力</span></li><li><b>如需隱蔽：選擇自然遮蔽物旁（大石、倒木、密灌叢）</b><br><span style=\"color:#666;font-size:13px;\">避免在開闊制高點（容易被發現）</span></li></ol><div style=\"margin-top:8px;padding:8px;background:#FFF3F3;border-radius:4px;\"><b>⚠️ 注意事項：</b><ul style=\"margin:4px 0;\"><li style=\"color:#CC0000;\">暴風雨前不要在大樹正下方（雷擊）</li><li style=\"color:#CC0000;\">不要在乾涸河床搭帳（暴洪）</li><li style=\"color:#CC0000;\">熱帶地區注意蛇類棲息地（石縫、落葉堆）</li></ul></div></div><div style=\"border:1px solid #ddd;border-radius:8px;padding:16px;margin:10px 0;background:#FAFAFA;\"><h3 style=\"margin:0 0 8px 0;\">庇護所類型與搭建</h3><p style=\"color:#666;font-size:13px;\"><b>適用情境：</b>選好地點後，需要搭建遮蔽處</p><p style=\"color:#666;font-size:13px;\"><b>所需材料：</b>樹枝/大葉/藤蔓或繩索</p><div style=\"margin:8px 0;padding:10px;background:#F0F8FF;border-radius:6px;border-left:4px solid #4488CC;\"><b>傾斜式（Lean-to）</b><br>1. 找一根長橫木架在兩棵樹之間（或一端靠在樹幹上）<br>2. 斜靠多根樹枝在橫木上，間隔 10-15cm<br>3. 在斜面上鋪大量樹葉/大葉/樹皮（由下往上如屋瓦）<br>4. 開口面背風</div><div style=\"margin:8px 0;padding:10px;background:#F0F8FF;border-radius:6px;border-left:4px solid #4488CC;\"><b>A 形架</b><br>1. 長橫木一端架高（靠樹或三腳架支撐）<br>2. 兩側都斜靠樹枝<br>3. 覆蓋樹葉<br>4. 形成三角形通道</div><div style=\"margin:8px 0;padding:10px;background:#F0F8FF;border-radius:6px;border-left:4px solid #4488CC;\"><b>天然庇護改造</b><br>1. 尋找倒木、岩石凹處、樹根底部的天然空間<br>2. 清除內部雜物和蟲<br>3. 用樹枝和樹葉加固遮蔽不足處<br>4. 入口掛布或樹葉簾</div></div><div style=\"border:1px solid #ddd;border-radius:8px;padding:16px;margin:10px 0;background:#FAFAFA;\"><h3 style=\"margin:0 0 8px 0;\">保暖方法</h3><p style=\"color:#666;font-size:13px;\"><b>適用情境：</b>夜間氣溫下降需要保暖</p><p style=\"color:#666;font-size:13px;\"><b>所需材料：</b>落葉/乾草/火源</p><div style=\"margin:8px 0;padding:10px;background:#F0F8FF;border-radius:6px;border-left:4px solid #4488CC;\"><b>隔熱層</b><br>在地面和身體之間鋪至少 15cm 厚的乾燥落葉、乾草、或松針。地面散熱是最大熱量損失來源。</div><div style=\"margin:8px 0;padding:10px;background:#F0F8FF;border-radius:6px;border-left:4px solid #4488CC;\"><b>身體覆蓋層</b><br>在身上堆疊大量乾燥落葉和草，像被子一樣覆蓋。越厚越保暖。</div><div style=\"margin:8px 0;padding:10px;background:#F0F8FF;border-radius:6px;border-left:4px solid #4488CC;\"><b>火牆反射</b><br>在庇護所開口面升火，在火的另一側立一面木頭/石頭牆反射熱輻射回庇護所。</div><div style=\"margin:8px 0;padding:10px;background:#F0F8FF;border-radius:6px;border-left:4px solid #4488CC;\"><b>熱石頭</b><br>將石頭放在火中加熱 30 分鐘，用布包裹後放在睡覺處。注意別燙傷。</div><div style=\"margin:8px 0;padding:10px;background:#F0F8FF;border-radius:6px;border-left:4px solid #4488CC;\"><b>蜷縮姿勢</b><br>側臥蜷縮（胎兒姿勢），減少暴露面積。雙手夾在腋下。</div><div style=\"margin-top:8px;padding:8px;background:#FFF3F3;border-radius:4px;\"><b>⚠️ 注意事項：</b><ul style=\"margin:4px 0;\"><li style=\"color:#CC0000;\">地面隔熱比覆蓋更重要（地面散熱佔 80%）</li><li style=\"color:#CC0000;\">潮濕物質無法保暖（保持乾燥）</li><li style=\"color:#CC0000;\">火源需有人看守</li></ul></div></div><div style=\"border:1px solid #ddd;border-radius:8px;padding:16px;margin:10px 0;background:#FAFAFA;\"><h3 style=\"margin:0 0 8px 0;\">防蟲防蛇措施</h3><p style=\"color:#666;font-size:13px;\"><b>適用情境：</b>在蟲蛇較多的地區休息</p><p style=\"color:#666;font-size:13px;\"><b>所需材料：</b>火源/煙燻材料/長褲長袖</p><ol style=\"line-height:2;\"><li><b>清除庇護所周圍 2 公尺內的落葉和雜草</b><br><span style=\"color:#666;font-size:13px;\">蛇類喜歡躲在落葉堆下</span></li><li><b>庇護所入口升火（小火即可），產生煙霧驅蟲</b><br><span style=\"color:#666;font-size:13px;\">加入綠色葉子或苔蘚增加煙量</span></li><li><b>衣物褲管紮入襪子或靴子中</b><br><span style=\"color:#666;font-size:13px;\">阻擋蜱蟲和螞蝗爬入</span></li><li><b>用煙燻過的布匹掛在入口處</b><br><span style=\"color:#666;font-size:13px;\">煙味可驅趕蚊蟲</span></li><li><b>睡前檢查庇護所內部有無蛇蟲</b><br><span style=\"color:#666;font-size:13px;\">用棍子翻動落葉和角落</span></li><li><b>睡覺時頭部用衣物覆蓋防蚊</b><br><span style=\"color:#666;font-size:13px;\">蚊子主要叮咬暴露皮膚</span></li></ol><div style=\"margin-top:8px;padding:8px;background:#FFF3F3;border-radius:4px;\"><b>⚠️ 注意事項：</b><ul style=\"margin:4px 0;\"><li style=\"color:#CC0000;\">半夜起來注意腳邊（蛇可能靠近溫暖的人體）</li><li style=\"color:#CC0000;\">不要用手伸入看不見的縫隙或洞穴</li><li style=\"color:#CC0000;\">早晨起來先抖落鞋子和衣物（蠍子、蜘蛛可能躲入）</li></ul></div></div>"},"tools":{"title":"工具","icon":"🔧","html":"<div style=\"border:1px solid #ddd;border-radius:8px;padding:16px;margin:10px 0;background:#FAFAFA;\"><h3 style=\"margin:0 0 8px 0;\">繩結教學</h3><p style=\"color:#666;font-size:13px;\"><b>適用情境：</b>需要綁紮固定物品、搭建庇護所、製作陷阱</p><p style=\"color:#666;font-size:13px;\"><b>所需材料：</b>繩索/藤蔓/植物纖維繩</p><div style=\"margin:8px 0;padding:10px;background:#F0F8FF;border-radius:6px;border-left:4px solid #4488CC;\"><b>稱人結（Bowline）</b> — 製作不會收緊的固定環圈（救人、繫船）<br>1. 繩端在主繩上繞一個小圈（兔子洞）<br>2. 繩端從洞裡穿出（兔子出洞）<br>3. 繞過主繩背面（繞過樹）<br>4. 再穿回洞裡（兔子回洞）<br>5. 拉緊</div><div style=\"margin:8px 0;padding:10px;background:#F0F8FF;border-radius:6px;border-left:4px solid #4488CC;\"><b>雙套結（Clove Hitch）</b> — 快速將繩子固定在柱子或樹枝上<br>1. 繩子繞柱子一圈<br>2. 再繞第二圈（交叉在第一圈上方）<br>3. 將繩端從第二圈下穿過<br>4. 拉緊</div><div style=\"margin:8px 0;padding:10px;background:#F0F8FF;border-radius:6px;border-left:4px solid #4488CC;\"><b>八字結（Figure-8）</b> — 防止繩子從孔洞中滑脫（止退結）<br>1. 繩子做一個圈<br>2. 繩端再繞過主繩一次<br>3. 從圈中穿過<br>4. 拉緊後看起來像數字 8</div><div style=\"margin:8px 0;padding:10px;background:#F0F8FF;border-radius:6px;border-left:4px solid #4488CC;\"><b>收口結（Constrictor Knot）</b> — 綑綁包裹、封口袋子、固定夾板<br>1. 繩子繞物體一圈<br>2. 繩端交叉越過主繩<br>3. 再繞一圈並從交叉處下方穿過<br>4. 用力拉緊（此結越拉越緊）</div><div style=\"margin:8px 0;padding:10px;background:#F0F8FF;border-radius:6px;border-left:4px solid #4488CC;\"><b>方回結（Square Knot）</b> — 連接兩條等粗繩子<br>1. 右壓左，穿過<br>2. 左壓右，穿過<br>3. 記住：右壓左、左壓右<br>4. 拉緊後平整對稱</div></div><div style=\"border:1px solid #ddd;border-radius:8px;padding:16px;margin:10px 0;background:#FAFAFA;\"><h3 style=\"margin:0 0 8px 0;\">石刀製作</h3><p style=\"color:#666;font-size:13px;\"><b>適用情境：</b>沒有刀具，需要切割工具</p><p style=\"color:#666;font-size:13px;\"><b>所需材料：</b>石英岩/燧石/黑曜石 + 打擊石</p><ol style=\"line-height:2;\"><li><b>選石：尋找質地細密的石頭（燧石、石英岩、黑曜石最佳）</b><br><span style=\"color:#666;font-size:13px;\">敲擊時應該會產生尖銳碎片，而非碎成粉末</span></li><li><b>選擇一塊掌心大的原石和一塊圓形打擊石</b><br><span style=\"color:#666;font-size:13px;\">打擊石要比原石更硬</span></li><li><b>錘擊法：用打擊石對準原石邊緣 45 度角敲擊</b><br><span style=\"color:#666;font-size:13px;\">不是正面敲，而是斜著敲邊緣，讓碎片（石片）剝離</span></li><li><b>取得的薄石片即可作為切割工具</b><br><span style=\"color:#666;font-size:13px;\">石片邊緣非常鋒利，小心割傷</span></li><li><b>精修（可選）：用鹿角或骨頭輕壓石片邊緣修整</b><br><span style=\"color:#666;font-size:13px;\">讓刀刃更均勻鋒利</span></li><li><b>用藤蔓或樹脂將石片綁在木柄上</b><br><span style=\"color:#666;font-size:13px;\">增加使用安全性和力道</span></li></ol><div style=\"margin-top:8px;padding:8px;background:#FFF3F3;border-radius:4px;\"><b>⚠️ 注意事項：</b><ul style=\"margin:4px 0;\"><li style=\"color:#CC0000;\">製作過程碎片飛濺，保護眼睛</li><li style=\"color:#CC0000;\">石片邊緣極鋒利，拿取時小心</li><li style=\"color:#CC0000;\">石刀脆弱，不適合劈砍，只適合切割</li></ul></div></div><div style=\"border:1px solid #ddd;border-radius:8px;padding:16px;margin:10px 0;background:#FAFAFA;\"><h3 style=\"margin:0 0 8px 0;\">容器製作</h3><p style=\"color:#666;font-size:13px;\"><b>適用情境：</b>需要盛裝水或食物的容器</p><p style=\"color:#666;font-size:13px;\"><b>所需材料：</b>竹子/樹皮/大葉/泥土</p><div style=\"margin:8px 0;padding:10px;background:#F0F8FF;border-radius:6px;border-left:4px solid #4488CC;\"><b>竹筒容器</b><br>1. 找一根直徑 8cm 以上的竹子<br>2. 在竹節處鋸斷（保留竹節作為底部）<br>3. 上方開口即為容器<br>4. 可直接盛水、煮水（竹筒不會在水燒乾前燒穿）</div><div style=\"margin:8px 0;padding:10px;background:#F0F8FF;border-radius:6px;border-left:4px solid #4488CC;\"><b>樹皮容器</b><br>1. 從活樹或新鮮倒木取一大片樹皮<br>2. 四角折起用藤蔓或木釘固定<br>3. 形成方形淺盆</div><div style=\"margin:8px 0;padding:10px;background:#F0F8FF;border-radius:6px;border-left:4px solid #4488CC;\"><b>大葉容器</b><br>1. 取姑婆芋或月桃的大葉<br>2. 折疊成杯狀或漏斗狀<br>3. 用細枝條釘住</div><div style=\"margin:8px 0;padding:10px;background:#F0F8FF;border-radius:6px;border-left:4px solid #4488CC;\"><b>泥土容器（進階）</b><br>1. 取黏土（河岸常見）<br>2. 塑形成碗或罐<br>3. 在火旁慢慢烘乾 2-3 小時<br>4. 最後放入火中燒製 1 小時</div></div><div style=\"border:1px solid #ddd;border-radius:8px;padding:16px;margin:10px 0;background:#FAFAFA;\"><h3 style=\"margin:0 0 8px 0;\">天然繩索製作</h3><p style=\"color:#666;font-size:13px;\"><b>適用情境：</b>需要繩索但身上沒有</p><p style=\"color:#666;font-size:13px;\"><b>所需材料：</b>植物纖維（芒草、構樹皮、棕櫚纖維）</p><ol style=\"line-height:2;\"><li><b>取得纖維：撕取構樹內皮、芒草葉、棕櫚葉纖維</b><br><span style=\"color:#666;font-size:13px;\">越長越好，較乾的纖維更強韌</span></li><li><b>將纖維分成兩束（等量）</b><br><span style=\"color:#666;font-size:13px;\">每束約 3-5 根纖維</span></li><li><b>兩束同時順時針搓捻（各自擰緊）</b><br><span style=\"color:#666;font-size:13px;\">搓到每束都自然想要回彈</span></li><li><b>兩束互相逆時針纏繞（交錯編織）</b><br><span style=\"color:#666;font-size:13px;\">搓捻力和纏繞力互相鎖定=強韌</span></li><li><b>接繩：快到繩端時，插入新纖維一起搓入</b><br><span style=\"color:#666;font-size:13px;\">接合處錯開，不要兩束同時接</span></li><li><b>完成後末端打結固定</b><br><span style=\"color:#666;font-size:13px;\">粗繩=多股細繩再次搓合</span></li></ol><div style=\"margin-top:8px;padding:8px;background:#FFF3F3;border-radius:4px;\"><b>⚠️ 注意事項：</b><ul style=\"margin:4px 0;\"><li style=\"color:#CC0000;\">新鮮植物纖維乾燥後會收縮（綁東西要預留餘量）</li><li style=\"color:#CC0000;\">潮濕纖維較弱，盡量使用乾燥的</li><li style=\"color:#CC0000;\">製作需要耐心，但一條好繩子價值極高</li></ul></div></div><div style=\"border:1px solid #ddd;border-radius:8px;padding:16px;margin:10px 0;background:#FAFAFA;\"><h3 style=\"margin:0 0 8px 0;\">簡易武器/工具</h3><p style=\"color:#666;font-size:13px;\"><b>適用情境：</b>需要防身或狩獵工具</p><p style=\"color:#666;font-size:13px;\"><b>所需材料：</b>硬木棍/石頭/繩索</p><div style=\"margin:8px 0;padding:10px;background:#F0F8FF;border-radius:6px;border-left:4px solid #4488CC;\"><b>木矛</b><br>1. 取一根筆直硬木（約 150-180cm）<br>2. 用石刀削尖一端<br>3. 將尖端放在火上烘烤（碳化硬化，不要燒焦）<br>4. 最後在粗糙石面上磨尖</div><div style=\"margin:8px 0;padding:10px;background:#F0F8FF;border-radius:6px;border-left:4px solid #4488CC;\"><b>棍棒</b><br>取一根手臂粗、約 60cm 的硬木。根端（較粗重）作為打擊端。可在打擊端綁上石頭增加重量。</div><div style=\"margin:8px 0;padding:10px;background:#F0F8FF;border-radius:6px;border-left:4px solid #4488CC;\"><b>投石索</b><br>1. 取約 60cm 繩索<br>2. 一端打結做指環<br>3. 中間做一個小兜（放石頭）<br>4. 甩動加速後釋放一端投出</div><div style=\"margin-top:8px;padding:8px;background:#FFF3F3;border-radius:4px;\"><b>⚠️ 注意事項：</b><ul style=\"margin:4px 0;\"><li style=\"color:#CC0000;\">武器主要用於防身和狩獵，謹慎使用</li><li style=\"color:#CC0000;\">矛尖朝外行走時注意安全</li><li style=\"color:#CC0000;\">手持武器靠近野生動物時，動物可能因恐懼而攻擊</li></ul></div></div>"},"navigation":{"title":"行","icon":"🧭","html":"<div style=\"border:1px solid #ddd;border-radius:8px;padding:16px;margin:10px 0;background:#FAFAFA;\"><h3 style=\"margin:0 0 8px 0;\">方位判斷</h3><p style=\"color:#666;font-size:13px;\"><b>適用情境：</b>沒有指南針，需要判斷東南西北</p><p style=\"color:#666;font-size:13px;\"><b>所需材料：</b>太陽/星星/手錶/樹枝</p><div style=\"margin:8px 0;padding:10px;background:#F0F8FF;border-radius:6px;border-left:4px solid #4488CC;\"><b>影子法（太陽）</b><br>1. 找一根直棍（約 1 公尺），垂直插在地上<br>2. 在影子尖端放一塊石頭（標記 A）<br>3. 等待 15-20 分鐘<br>4. 在新的影子尖端放另一塊石頭（標記 B）<br>5. A→B 的方向即為「東方」<br>6. 站在 AB 線上，A 在左腳 B 在右腳，面朝的方向為「北方」</div><div style=\"margin:8px 0;padding:10px;background:#F0F8FF;border-radius:6px;border-left:4px solid #4488CC;\"><b>手錶法</b><br>1. 將手錶水平放置<br>2. 時針對準太陽方向<br>3. 時針和 12 點之間的角平分線方向即為「南方」（北半球）<br>— 南半球：12 點對準太陽，12 點和時針的角平分線為「北方」</div><div style=\"margin:8px 0;padding:10px;background:#F0F8FF;border-radius:6px;border-left:4px solid #4488CC;\"><b>北極星法（夜間）</b><br>1. 找到北斗七星（像勺子的七顆星）<br>2. 勺口兩顆星的連線延伸約 5 倍距離<br>3. 那裡最亮的星就是北極星<br>4. 北極星的方向即為正北方</div><div style=\"margin:8px 0;padding:10px;background:#F0F8FF;border-radius:6px;border-left:4px solid #4488CC;\"><b>苔蘚觀察法</b><br>樹幹上苔蘚/青苔較多的一面「傾向」為北面（北半球），因為北面日照少、較潮濕。但這只是傾向，不是絕對！要觀察多棵樹取多數一致的方向。</div><div style=\"margin:8px 0;padding:10px;background:#F0F8FF;border-radius:6px;border-left:4px solid #4488CC;\"><b>南十字星法（南半球）</b><br>1. 找到南十字星座（四顆星組成十字）<br>2. 沿長軸方向延伸 4.5 倍距離<br>3. 該點垂直投影到地平線的方向為「正南方」</div></div><div style=\"border:1px solid #ddd;border-radius:8px;padding:16px;margin:10px 0;background:#FAFAFA;\"><h3 style=\"margin:0 0 8px 0;\">時間判斷</h3><p style=\"color:#666;font-size:13px;\"><b>適用情境：</b>手錶壞了或手機沒電，需要估計目前時間</p><p style=\"color:#666;font-size:13px;\"><b>所需材料：</b>太陽/自己的手</p><div style=\"margin:8px 0;padding:10px;background:#F0F8FF;border-radius:6px;border-left:4px solid #4488CC;\"><b>拳頭法</b><br>1. 面向太陽，手臂伸直<br>2. 將拳頭疊在地平線和太陽之間<br>3. 每一個拳頭寬度約等於 1 小時<br>4. 從地平線數到太陽 = 距離日落的小時數<br>— 例：太陽距地平線 3 拳 = 約 3 小時後日落</div><div style=\"margin:8px 0;padding:10px;background:#F0F8FF;border-radius:6px;border-left:4px solid #4488CC;\"><b>影長法</b><br>正午時太陽最高，影子最短。<br>影子開始變長 = 過了正午。<br>影子長度 = 物體高度時，大約是下午 3-4 點（視緯度和季節）。</div></div><div style=\"border:1px solid #ddd;border-radius:8px;padding:16px;margin:10px 0;background:#FAFAFA;\"><h3 style=\"margin:0 0 8px 0;\">求救信號</h3><p style=\"color:#666;font-size:13px;\"><b>適用情境：</b>需要向搜救隊或過路者發出求救</p><p style=\"color:#666;font-size:13px;\"><b>所需材料：</b>火源/鏡子或反光物/哨子/明顯材料</p><div style=\"margin:8px 0;padding:10px;background:#F0F8FF;border-radius:6px;border-left:4px solid #4488CC;\"><b>國際三聯信號</b><br>任何信號重複三次代表求救：<br>• 三堆火（三角形排列，間隔約 30 公尺）<br>• 三聲哨音（間隔 1 秒）<br>• 三次閃光</div><div style=\"margin:8px 0;padding:10px;background:#F0F8FF;border-radius:6px;border-left:4px solid #4488CC;\"><b>煙霧信號</b><br>白天：在火上加綠色植物/苔蘚產生白煙（晴天對比明顯）<br>陰天：燒橡膠/塑膠產生黑煙（灰色天空下更顯眼）</div><div style=\"margin:8px 0;padding:10px;background:#F0F8FF;border-radius:6px;border-left:4px solid #4488CC;\"><b>反光信號</b><br>用鏡子/金屬片/手機螢幕反射陽光向飛機或遠處人員閃爍。三短三長三短（SOS）。</div><div style=\"margin:8px 0;padding:10px;background:#F0F8FF;border-radius:6px;border-left:4px solid #4488CC;\"><b>地面標記</b><br>在開闊地用石頭/樹枝排列大型 SOS 或 X 標記（至少 3 公尺高）。使用與背景對比強烈的顏色。</div><div style=\"margin:8px 0;padding:10px;background:#F0F8FF;border-radius:6px;border-left:4px solid #4488CC;\"><b>哨音求救</b><br>每分鐘吹三聲短促哨音，休息一分鐘後重複。沒有哨子可以用手指或葉子吹口哨。</div></div><div style=\"border:1px solid #ddd;border-radius:8px;padding:16px;margin:10px 0;background:#FAFAFA;\"><h3 style=\"margin:0 0 8px 0;\">地形判讀與移動原則</h3><p style=\"color:#666;font-size:13px;\"><b>適用情境：</b>需要在野外移動尋找安全地帶或文明</p><p style=\"color:#666;font-size:13px;\"><b>所需材料：</b>無</p><div style=\"margin-top:8px;\"><b>⚠️ 危險地形：</b></div><table style=\"width:100%;border-collapse:collapse;font-size:13px;margin:4px 0;\"><tr style=\"background:#FFF0F0;\"><th style=\"padding:6px;\">地形</th><th>應對</th></tr><tr><td style=\"padding:6px;border:1px solid #ddd;\"><b>陡峭懸崖</b></td><td style=\"padding:6px;border:1px solid #ddd;\">繞行，絕不冒險攀爬</td></tr><tr><td style=\"padding:6px;border:1px solid #ddd;\"><b>沼澤/軟泥地</b></td><td style=\"padding:6px;border:1px solid #ddd;\">繞行或用棍子探路，一步一試</td></tr><tr><td style=\"padding:6px;border:1px solid #ddd;\"><b>河流急流</b></td><td style=\"padding:6px;border:1px solid #ddd;\">不要嘗試涉水，尋找窄處或搭便橋</td></tr><tr><td style=\"padding:6px;border:1px solid #ddd;\"><b>密林無路區</b></td><td style=\"padding:6px;border:1px solid #ddd;\">用刀開路或找動物小徑</td></tr></table><ol style=\"line-height:2;\"><li><b>沿溪流下行通常能找到人類聚落</b><br><span style=\"color:#666;font-size:13px;\">水流方向是天然路標，但注意瀑布和峽谷</span></li><li><b>山稜線視野好但暴露，谷底有水但可能迷路</b><br><span style=\"color:#666;font-size:13px;\">視情況選擇：需要方向感走稜線，需要水和遮蔽走谷底</span></li><li><b>每走 30 分鐘停下來回頭看一次路</b><br><span style=\"color:#666;font-size:13px;\">記住返回方向，避免迷失</span></li><li><b>做標記：折斷樹枝、疊石頭、在樹皮上刻劃</b><br><span style=\"color:#666;font-size:13px;\">幫助你不重複走路和讓搜救隊追蹤</span></li><li><b>聽人造聲音：車聲、狗叫、機器聲</b><br><span style=\"color:#666;font-size:13px;\">這些聲音代表文明很近</span></li><li><b>觀察電線、飛機航線、人造垃圾</b><br><span style=\"color:#666;font-size:13px;\">越多人造物品表示越接近人類活動區</span></li></ol></div>"}}''')

GUIDE_ORDER = ["food","water","medical","shelter","tools","navigation"]
print(f"\u2705 Survival guide: {list(GUIDE_RENDERED.keys())}")

## 🔄 Pipeline — 完整辨識流程Phase 0: 載入 KB → Phase 1: LLM 特徵萃取 → Phase 2: 特徵合併 → Phase 3: 比對計分 → Phase 4: 後處理

In [ ]:
# Cell 10: Pipeline (Notebook 版)

class PipelineState:
    """Tracks state across iterative photo sessions."""
    def __init__(self):
        self.accumulated_features = {}
        self.photo_count = 0
        self.user_overrides = {}
        self.extraction_logs = []
        self.identification_logs = []
        self.latest_result = None


class JungleSurvivorV2:
    """Main v2 pipeline controller — Notebook version (no file I/O)."""

    def __init__(self):
        self.plants = PLANTS_DB
        self.schema = FEATURE_SCHEMA
        self.confusion_db = CONFUSION_DB
        self.enums = DERIVED_ENUMS
        self.prompt_template = PROMPT_TEMPLATE
        self.state = PipelineState()
        self._plants_by_id = {sp["id"]: sp for sp in self.plants}

    def reset(self):
        self.state = PipelineState()

    def get_prompt(self) -> str:
        return self.prompt_template

    # ── Phase 1: Feature Extraction ──

    def extract_features_from_response(self, llm_response: str) -> ExtractionResult:
        result = parse_llm_response(llm_response, self.schema, self.enums)
        if result.success and result.features:
            self.state.accumulated_features = merge_features(
                self.state.accumulated_features,
                result.features,
                self.schema,
            )
            self.state.photo_count += 1
        return result

    def log_extraction(self, llm_time_sec, response_len, result):
        feat_count = 0
        if result.success and result.features:
            for sec, data in result.features.items():
                feat_count += len(data) if isinstance(data, dict) else 1
        self.state.extraction_logs.append({
            "photo_num": self.state.photo_count,
            "time_sec": llm_time_sec,
            "success": result.success,
            "resp_len": response_len,
            "feature_count": feat_count,
            "warning_count": len(result.warnings),
        })

    # ── Phase 2: Feature Management ──

    def get_current_features(self) -> dict:
        if self.state.user_overrides:
            return apply_user_input(
                self.state.accumulated_features,
                self.state.user_overrides,
            )
        return self.state.accumulated_features

    def get_feature_summary(self) -> dict:
        features = self.get_current_features()
        return get_feature_summary(features, self.schema)

    def set_user_feature(self, section: str, attr: str, value):
        if section not in self.state.user_overrides:
            self.state.user_overrides[section] = {}
        self.state.user_overrides[section][attr] = value

    def remove_user_override(self, section: str, attr: str):
        if section in self.state.user_overrides:
            self.state.user_overrides[section].pop(attr, None)
            if not self.state.user_overrides[section]:
                del self.state.user_overrides[section]

    def get_override_list(self):
        items = []
        for sec, attrs in self.state.user_overrides.items():
            for attr, val in attrs.items():
                items.append((sec, attr, val))
        return items

    # ── Phase 3 + 4: Identify ──

    def identify(self, top_n: int = 3) -> ProcessedResult:
        features = self.get_current_features()
        has_photo = self.state.photo_count > 0
        top_results = match_top_n(features, self.plants, self.schema, has_photo=has_photo, top_n=top_n)
        result = process_results(top_results, self.confusion_db, self.plants)
        self.state.latest_result = result
        return result

    def log_identification(self, time_sec, processed):
        top1 = ""
        top1_score = ""
        warning_level = ""
        if processed.top_results:
            r0 = processed.top_results[0]
            sp = self._plants_by_id.get(r0.species_id, {})
            top1 = sp.get("common_names", {}).get("zh-TW", r0.species_name)
            top1_score = f"{r0.confidence:.1f}%"
        warning_level = processed.warning.level if processed.warning else ""
        ovr_count = sum(len(v) for v in self.state.user_overrides.values())
        self.state.identification_logs.append({
            "time_sec": time_sec,
            "photo_count": self.state.photo_count,
            "override_count": ovr_count,
            "top1": top1,
            "top1_score": top1_score,
            "warning_level": warning_level,
        })

    def format_display(self, result=None) -> str:
        r = result or self.state.latest_result
        if r is None:
            return "尚未進行辨識。請先上傳照片。"
        return format_result_display(r, self.plants)

    def get_species_info(self, species_id: str):
        return self._plants_by_id.get(species_id)

    def get_schema_enums_for_attr(self, section: str, attr: str) -> list:
        return self.enums.get(section, {}).get(attr, [])


pipeline = JungleSurvivorV2()
print(f"\u2705 Pipeline 初始化完成 — {len(pipeline.plants)} species loaded")

## 🤖 Gemma 4 模型載入使用 `google/gemma-4-E4B-it` 多模態模型，專門用於看照片萃取結構化特徵。

In [ ]:
# Cell 11: 載入 Gemma 4 多模態模型
from transformers import AutoProcessor, AutoModelForImageTextToText

processor = AutoProcessor.from_pretrained(MODEL_ID)
processor.image_seq_length = 560  # 280(預設)→560: 圖片細節更豐富，利於辨識葉面質感等

# E4B BF16 ~16GB，超過單張 T4 (14.56 GiB)，需要 device_map="auto" 分散到兩張 GPU
# transformers 會自動在前向傳遞時處理跨 GPU 的 tensor 搬移
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    dtype=DTYPE,
    device_map="auto",
)
print(f"\u2705 Model loaded: {MODEL_ID}")
print(f"   Device map: {model.hf_device_map}")
print(f"   Dtype: {model.dtype}")
for i in range(torch.cuda.device_count()):
    print(f"   GPU {i} Memory: {torch.cuda.memory_allocated(i) / 1024**3:.1f} GB used")

## 📸 辨識函式 — 從照片到結果1. 將 prompt_template + 照片送入 Gemma 42. LLM 回傳結構化 JSON 特徵3. 特徵萃取 + 驗證 + 合併4. 演算法比對 KB → Top 3 結果5. 後處理：警告等級 + 混淆物種

In [ ]:
# Cell 12: 辨識函式

IMG_MAX_SIDE = 1600
IMG_MIN_SIDE = 256

def preprocess_image(image: Image.Image) -> Image.Image:
    """Resize image so the long side is at most IMG_MAX_SIDE pixels (LANCZOS)."""
    w, h = image.size
    if max(w, h) <= IMG_MAX_SIDE and min(w, h) >= IMG_MIN_SIDE:
        return image
    if max(w, h) > IMG_MAX_SIDE:
        ratio = IMG_MAX_SIDE / max(w, h)
        new_size = (int(w * ratio), int(h * ratio))
        image = image.resize(new_size, Image.LANCZOS)
    return image


def _get_input_device():
    """Get the device for the model's first parameter (for multi-GPU device_map='auto')."""
    try:
        return next(model.parameters()).device
    except StopIteration:
        return torch.device("cuda:0")

def call_gemma(image: Image.Image, prompt: str) -> str:
    """Send image + prompt to Gemma 4, return raw text response."""
    image = preprocess_image(image)
    content = [{"type": "image", "image": image}, {"type": "text", "text": prompt}]
    messages = [{"role": "user", "content": content}]

    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        add_generation_prompt=True,
    ).to(_get_input_device())

    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)

    input_len = inputs["input_ids"].shape[1]
    response = processor.decode(output_ids[0][input_len:], skip_special_tokens=True)
    return response


def identify_from_image(image: Image.Image, session_pipeline=None) -> str:
    """Full pipeline: image → Gemma 4 → features → matching → result display."""
    if session_pipeline is None:
        session_pipeline = pipeline

    prompt = session_pipeline.get_prompt()
    print("📸 正在呼叫 Gemma 4 萃取特徵...")
    llm_response = call_gemma(image, prompt)
    print(f"   LLM 回應長度: {len(llm_response)} chars")

    result = session_pipeline.extract_features_from_response(llm_response)
    if not result.success:
        return f"❌ 特徵萃取失敗: {result.error}\n\n原始回應:\n{llm_response[:500]}"

    features = session_pipeline.get_current_features()
    feature_count = sum(
        len(v) if isinstance(v, dict) else 1
        for v in features.values()
    )
    print(f"   ✅ 萃取成功 — {feature_count} attributes across {len(features)} sections")
    if result.warnings:
        for w in result.warnings:
            print(f"   ⚠️ {w}")

    processed = session_pipeline.identify(top_n=3)
    display = session_pipeline.format_display(processed)
    return display


def identify_from_url(url: str) -> str:
    """Download image from URL and identify."""
    response = requests.get(url, timeout=30)
    image = Image.open(BytesIO(response.content)).convert("RGB")
    return identify_from_image(image)


print("✅ 辨識函式就緒")

## 🧪 Demo 測試### 測試 1: 純演算法測試（不需要 GPU）模擬 LLM 已萃取好的特徵，直接測試計分引擎。

In [ ]:
# Cell 13: 演算法測試 — 模擬已萃取的特徵

test_pipeline = JungleSurvivorV2()

# 模擬姑婆芋特徵 (LLM 萃取結果)
alocasia_features = {
    "growth_form": "herb",
    "height_estimate": "1-2m",
    "habitat": "forest_floor",
    "leaf": {
        "shape": "heart",
        "edge": "entire",
        "colors": ["dark_green"],
        "surface_top": "glossy",
        "size": "very_large_>50cm",
        "arrangement": "clustered",
        "venation": "pinnate",
        "texture": "leathery"
    },
    "stem": {
        "type": "erect",
        "colors": ["green"],
        "surface": "smooth"
    },
    "flower": {
        "special_shape": "spathe",
        "arrangement": "spathe_spadix",
        "colors": ["green", "yellow"]
    }
}

mock_response = json.dumps(alocasia_features, ensure_ascii=False)
result = test_pipeline.extract_features_from_response(mock_response)
print(f"特徵萃取: {'✅ 成功' if result.success else '❌ 失敗'}")
if result.warnings:
    for w in result.warnings:
        print(f"  ⚠️ {w}")

processed = test_pipeline.identify(top_n=3)
print()
print(test_pipeline.format_display(processed))
print()
print("=" * 60)

# 測試 2: 模擬魚腥草特徵
test_pipeline.reset()
houttuynia_features = {
    "growth_form": "herb",
    "height_estimate": "<30cm",
    "smell": "fishy",
    "habitat": "streamside",
    "leaf": {
        "shape": "heart",
        "edge": "entire",
        "colors": ["green", "red_underside"],
        "surface_top": "matte",
        "arrangement": "alternate",
        "size": "medium_5-15cm"
    },
    "stem": {
        "type": "creeping",
        "colors": ["purple"]
    },
    "flower": {
        "colors": ["white"],
        "petal_count": "4",
        "arrangement": "spike"
    }
}

mock_response2 = json.dumps(houttuynia_features, ensure_ascii=False)
result2 = test_pipeline.extract_features_from_response(mock_response2)
print(f"特徵萃取: {'✅ 成功' if result2.success else '❌ 失敗'}")
processed2 = test_pipeline.identify(top_n=3)
print()
print(test_pipeline.format_display(processed2))

### 測試 2: 真實照片辨識（需要 GPU + Gemma 4）上傳照片或貼入 URL 進行測試。

In [ ]:
# Cell 14: 真實照片辨識（取消註解並替換 URL 即可使用）

# pipeline.reset()
# result = identify_from_url("https://upload.wikimedia.org/wikipedia/commons/thumb/X/XX/example.jpg/800px-example.jpg")
# print(result)

# 或使用本機上傳：
# from google.colab import files  # Colab
# uploaded = files.upload()
# for name, data in uploaded.items():
#     img = Image.open(BytesIO(data)).convert("RGB")
#     pipeline.reset()
#     print(identify_from_image(img))

print("💡 取消上方註解並替換為實際圖片 URL 來測試真實照片辨識")

### 測試 3: 多照片特徵合併

In [ ]:
# Cell 15: 多照片特徵合併測試

test_multi = JungleSurvivorV2()

# 第一張照片 — 只看到葉子
photo1 = json.dumps({
    "growth_form": "herb",
    "height_estimate": "1-2m",
    "leaf": {
        "shape": "heart",
        "edge": "entire",
        "colors": ["dark_green"],
        "surface_top": "glossy",
        "size": "very_large_>50cm",
        "venation": "pinnate"
    },
    "stem": {"type": "not_visible"},
    "flower": {"colors": "not_visible"}
})

r1 = test_multi.extract_features_from_response(photo1)
print(f"📸 Photo 1: {'✅' if r1.success else '❌'} — {test_multi.state.photo_count} photo(s)")

# 第二張照片 — 看到花和莖
photo2 = json.dumps({
    "flower": {
        "special_shape": "spathe",
        "arrangement": "spathe_spadix",
        "colors": ["green", "yellow"]
    },
    "stem": {
        "type": "erect",
        "colors": ["green"],
        "surface": "smooth"
    }
})

r2 = test_multi.extract_features_from_response(photo2)
print(f"📸 Photo 2: {'✅' if r2.success else '❌'} — {test_multi.state.photo_count} photo(s)")

# 合併結果
features = test_multi.get_current_features()
summary = test_multi.get_feature_summary()
print(f"\n📊 特徵完整度:")
for section, info in summary.items():
    print(f"   {section}: {info['filled']}/{info['total']}")

processed = test_multi.identify(top_n=3)
print()
print(test_multi.format_display(processed))

## 🎨 Gradio 互動介面提供完整的互動式植物辨識介面：- 上傳照片 → AI 自動萃取特徵- Tab 分頁顯示每個特徵的下拉選單，可直接編輯- 多照片特徵自動聯集合併至下拉選單- 一鍵辨識 → 安全警告 + Top 3 結果

In [ ]:
# Cell 16: Gradio 互動介面
import gradio as gr

SECTION_NAMES = {
    "overall": "整株特徵", "leaf": "葉", "stem": "莖",
    "flower": "花", "fruit": "果實", "root": "根/地下部",
}
SECTION_ORDER = ["overall", "leaf", "stem", "flower", "fruit", "root"]

ATTR_NAMES = {
    "growth_form": "生長型態", "height_estimate": "高度估計",
    "latex": "汁液", "smell": "氣味", "habitat": "棲地",
    "water_droplet_test": "水珠測試", "leaf_type": "葉型",
    "shape": "形狀", "edge": "葉緣", "tip": "葉尖", "base": "葉基",
    "colors": "顏色", "color_pattern": "色彩花紋",
    "surface_top": "葉面質感", "surface_bottom": "葉背質感",
    "arrangement": "排列", "size": "大小", "venation": "脈序",
    "texture": "質地", "petiole_attach": "葉柄著生",
    "type": "類型", "cross_section": "橫截面", "surface": "表面",
    "interior": "內部", "has_thorns": "有刺",
    "petal_count": "花瓣數", "symmetry": "對稱性",
    "position": "位置", "orientation": "朝向",
    "special_shape": "特殊形態", "fragrant": "有香味",
}

VALUE_ZH = {
    "herb": "草本", "shrub": "灌木", "tree": "喬木", "vine": "藤蔓",
    "fern": "蕨類", "fungus": "菇菌類", "grass": "禾草", "succulent": "多肉",
    "aquatic": "水生", "moss": "苔蘚", "palm_like": "棕櫚狀",
    "<30cm": "< 30cm", "30-100cm": "30-100cm", "1-2m": "1-2m",
    "2-5m": "2-5m", ">5m": "> 5m",
    "none": "無", "yes_white": "有（白色乳汁）", "yes_yellow": "有（黃色）",
    "yes_red": "有（紅色）", "yes_clear": "有（透明）",
    "aromatic": "芳香", "pungent": "刺鼻", "fishy": "魚腥味",
    "minty": "薄荷味", "spicy": "辛辣味", "rotten": "腐臭", "sweet": "甜香",
    "roadside": "路邊", "forest_floor": "林下", "streamside": "溪邊",
    "cliff_rock": "岩壁", "open_field": "空曠地", "wetland": "濕地",
    "epiphytic": "附生", "parasitic": "寄生", "urban": "都市", "coastal": "海岸",
    "beading": "水珠成珠", "flat": "水珠攤平",
    "simple": "單葉", "trifoliate": "三出複葉",
    "pinnate_compound": "羽狀複葉", "bipinnate_compound": "二回羽狀複葉",
    "palmate_compound": "掌狀複葉",
    "heart": "心形", "arrow": "箭形", "oval": "卵形", "elliptic": "橢圓",
    "lanceolate": "披針形", "linear": "線形", "needle": "針形",
    "scale": "鱗片狀", "round": "圓形", "spatulate": "匙形",
    "obovate": "倒卵形", "rhombic": "菱形", "fan": "扇形", "kidney": "腎形",
    "entire": "全緣", "serrated": "鋸齒", "double_serrated": "重鋸齒",
    "crenate": "圓齒", "lobed": "裂片", "deeply_lobed": "深裂",
    "wavy": "波狀", "spiny": "刺狀",
    "acute": "銳尖", "acuminate": "漸尖", "rounded": "圓鈍",
    "emarginate": "凹缺", "truncate": "截形",
    "cordate": "心形基", "cuneate": "楔形基", "sagittate": "箭形基",
    "peltate": "盾狀基",
    "light_green": "淺綠", "green": "綠色", "dark_green": "深綠",
    "yellow_green": "黃綠", "red": "紅色", "purple": "紫色",
    "variegated": "斑葉", "silver": "銀色", "brown": "褐色",
    "red_underside": "葉背紅", "white": "白色", "yellow": "黃色",
    "orange": "橘色", "pink": "粉紅", "blue": "藍色",
    "gray": "灰色", "white_powdery": "白粉狀", "black": "黑色",
    "white_waxy": "白蠟質",
    "solid": "單色", "gradient": "漸層", "spotted": "斑點",
    "striped": "條紋", "bicolor": "雙色", "center_different": "花心異色",
    "glossy": "光滑亮澤", "matte": "霧面", "hairy": "有毛",
    "rough": "粗糙", "waxy": "蠟質", "velvety": "天鵝絨",
    "pubescent": "柔毛", "scaly": "鱗片狀", "sandpaper": "砂紙狀",
    "alternate": "互生", "opposite": "對生", "whorled": "輪生",
    "rosette": "蓮座", "basal_rosette": "基生蓮座", "clustered": "叢生",
    "spiral": "螺旋", "two_ranked": "二列", "bird_nest_radial": "鳥巢放射狀",
    "tiny_<2cm": "極小 (< 2cm)", "small_2-5cm": "小 (2-5cm)",
    "medium_5-15cm": "中 (5-15cm)", "large_15-50cm": "大 (15-50cm)",
    "very_large_>50cm": "特大 (> 50cm)",
    "parallel": "平行脈", "pinnate": "羽狀脈", "palmate": "掌狀脈",
    "reticulate": "網狀脈", "single_midrib": "單主脈",
    "papery": "紙質", "leathery": "革質", "fleshy": "肉質",
    "membranous": "膜質",
    "normal": "一般著生", "peltate_shield": "盾狀",
    "sheathing": "鞘狀", "sessile": "無柄",
    "erect": "直立", "creeping": "匍匐", "climbing": "攀緣",
    "twining": "纏繞", "prostrate": "伏地", "rhizome": "根莖",
    "pseudostem": "假莖",
    "square": "方形", "triangular": "三角形",
    "flattened": "扁平", "ridged": "有稜",
    "smooth": "光滑", "thorny": "有刺", "bark_rough": "樹皮粗糙",
    "bark_smooth": "樹皮光滑",
    "hollow": "中空", "pithy": "有髓",
    "yes": "是", "no": "否",
    "0": "無花瓣", "3": "3 瓣", "4": "4 瓣", "5": "5 瓣",
    "6": "6 瓣", "many": "多瓣", "fused_tubular": "合瓣筒狀",
    "radial": "輻射對稱", "bilateral": "兩側對稱", "irregular": "不規則",
    "tiny_<5mm": "極小 (< 5mm)", "small_5-15mm": "小 (5-15mm)",
    "medium_15-30mm": "中 (15-30mm)", "large_30-60mm": "大 (30-60mm)",
    "very_large_>60mm": "特大 (> 60mm)",
    "solitary": "單生", "raceme": "總狀", "spike": "穗狀",
    "umbel": "繖形", "head_composite": "頭狀/菊科", "panicle": "圓錐",
    "cyme": "聚繖", "catkin": "柔荑", "spathe_spadix": "佛焰苞+肉穗",
    "terminal": "頂生", "axillary": "腋生", "basal": "基生",
    "cauliflorous": "莖生花",
    "upright": "向上", "horizontal": "水平", "drooping": "下垂",
    "lip_labellum": "唇瓣", "spur": "距", "spathe": "佛焰苞",
    "butterfly_shape": "蝶形", "bell_tubular": "鐘形/筒狀", "trumpet": "喇叭形",
    "berry": "漿果", "drupe": "核果", "capsule": "蒴果",
    "pod_legume": "莢果", "achene": "瘦果", "samara": "翅果",
    "nut": "堅果", "aggregate": "聚合果", "fig": "隱花果",
    "cone": "毬果", "spore": "孢子",
    "large_>30mm": "大 (> 30mm)", "medium_15-30mm": "中 (15-30mm)",
    "small_5-15mm": "小 (5-15mm)", "tiny_<5mm": "極小 (< 5mm)",
    "warty": "疣狀", "hooked_bristles": "鉤刺",
    "fibrous": "鬚根", "taproot": "主根", "tuberous": "塊根",
    "bulb": "鱗莖", "corm": "球莖", "aerial_root": "氣生根",
    "storage_tuber": "儲藏塊莖",
    "not_visible": "看不到", "uncertain": "不確定", "not_checkable": "無法檢查",
}


def val_to_zh(v):
    return VALUE_ZH.get(v, v)


ui_pipeline = JungleSurvivorV2()

# Build ordered attribute list from schema (used to map dropdowns ↔ features)
_schema_clean = {k: v for k, v in FEATURE_SCHEMA.items() if not k.startswith("_")}
ATTR_ORDER = []
for _s in SECTION_ORDER:
    for _a, _d in _schema_clean.get(_s, {}).items():
        if _d["type"] != "boolean":
            ATTR_ORDER.append((_s, _a, _d))
N_ATTRS = len(ATTR_ORDER)


def _no_change():
    """Return gr.update() for every dropdown (no visual change)."""
    return tuple(gr.update() for _ in range(N_ATTRS))


def _get_dropdown_updates():
    """Return gr.update(value=...) for every dropdown from current features."""
    features = ui_pipeline.get_current_features()
    updates = []
    for sec, attr, adef in ATTR_ORDER:
        val = features.get(sec, {}).get(attr)
        if val is None:
            updates.append(gr.update(value=[] if adef["type"] == "array" else None))
        else:
            updates.append(gr.update(value=val))
    return tuple(updates)


def _clear_dropdowns():
    return tuple(
        gr.update(value=[] if ad["type"] == "array" else None)
        for _, _, ad in ATTR_ORDER
    )


def _sync_overrides(dd_vals):
    """Compare dropdown values with AI accumulated features; differences become overrides."""
    if not ui_pipeline.state.accumulated_features:
        return
    for i, (sec, attr, adef) in enumerate(ATTR_ORDER):
        val = dd_vals[i]
        ai_val = ui_pipeline.state.accumulated_features.get(sec, {}).get(attr)
        if adef["type"] == "array":
            v_list = val if isinstance(val, list) else ([] if val is None else [val])
            a_list = ai_val if isinstance(ai_val, list) else ([] if ai_val is None else [ai_val])
            if sorted(str(x) for x in v_list) != sorted(str(x) for x in a_list):
                if v_list:
                    ui_pipeline.set_user_feature(sec, attr, v_list)
                else:
                    ui_pipeline.remove_user_override(sec, attr)
            else:
                ui_pipeline.remove_user_override(sec, attr)
        else:
            if val != ai_val and val is not None:
                ui_pipeline.set_user_feature(sec, attr, val)
            else:
                ui_pipeline.remove_user_override(sec, attr)


def format_history_md():
    ext = ui_pipeline.state.extraction_logs
    ids = ui_pipeline.state.identification_logs
    lines = []
    if ext:
        lines.append("### LLM 特徵萃取紀錄")
        lines.append("| # | 耗時 | 結果 | 回應長度 | 特徵數 | 警告 |")
        lines.append("|---|------|------|----------|--------|------|")
        for i, log in enumerate(ext, 1):
            st = "\u2705" if log["success"] else "\u274c"
            lines.append(
                f"| {i} | {log['time_sec']:.1f}s | {st} "
                f"| {log['resp_len']} | {log['feature_count']} | {log['warning_count']} |"
            )
        lines.append("")
    if ids:
        lines.append("### 辨識紀錄")
        lines.append("| # | 耗時 | 照片 | Top 1 | 信心度 | 警告等級 |")
        lines.append("|---|------|------|-------|--------|----------|")
        for i, log in enumerate(ids, 1):
            lines.append(
                f"| {i} | {log['time_sec']:.2f}s | {log['photo_count']} "
                f"| {log['top1']} | {log['top1_score']} | {log['warning_level']} |"
            )
        lines.append("")
    if not lines:
        return "尚無紀錄。"
    return "\n".join(lines)


def _get_final_features_json():
    """Return the merged features (accumulated + overrides) for the JSON display."""
    features = ui_pipeline.get_current_features()
    return features if features else {}

# ── Event Handlers ──

def on_upload_photo(image, *dd_vals):
    _sync_overrides(dd_vals)
    if image is None:
        yield ("請上傳照片",) + _no_change() + (gr.update(), gr.update())
        return
    yield ("\u23f3 AI 正在萃取特徵...",) + _no_change() + (gr.update(), gr.update())
    pil_image = Image.fromarray(image).convert("RGB")
    prompt = ui_pipeline.get_prompt()
    try:
        t0 = time.time()
        llm_response = call_gemma(pil_image, prompt)
        llm_time = time.time() - t0
        result = ui_pipeline.extract_features_from_response(llm_response)
        ui_pipeline.log_extraction(llm_time, len(llm_response), result)
        if not result.success:
            yield (f"\u274c 萃取失敗: {result.error}",) + _no_change() + (format_history_md(), _get_final_features_json())
            return
        status = f"\u2705 第 {ui_pipeline.state.photo_count} 張照片（{llm_time:.1f}s）"
        if result.warnings:
            status += "\n" + "\n".join(f"\u26a0\ufe0f {w}" for w in result.warnings)
        yield (status,) + _get_dropdown_updates() + (format_history_md(), _get_final_features_json())
    except Exception as e:
        yield (f"\u274c 錯誤: {str(e)}",) + _no_change() + (format_history_md(), _get_final_features_json())


def on_paste_json(json_text, *dd_vals):
    _sync_overrides(dd_vals)
    if not json_text.strip():
        return ("請貼入 JSON",) + _no_change() + (gr.update(), gr.update())
    t0 = time.time()
    result = ui_pipeline.extract_features_from_response(json_text)
    parse_time = time.time() - t0
    ui_pipeline.log_extraction(parse_time, len(json_text), result)
    if not result.success:
        return (f"\u274c 萃取失敗: {result.error}",) + _no_change() + (format_history_md(), _get_final_features_json())
    status = f"\u2705 第 {ui_pipeline.state.photo_count} 張（{parse_time:.2f}s）"
    if result.warnings:
        status += "\n" + "\n".join(f"\u26a0\ufe0f {w}" for w in result.warnings)
    return (status,) + _get_dropdown_updates() + (format_history_md(), _get_final_features_json())


def on_identify(*dd_vals):
    _sync_overrides(dd_vals)
    features = ui_pipeline.get_current_features()
    if not features:
        return "請先上傳照片或輸入特徵。", format_history_md(), _get_final_features_json()
    t0 = time.time()
    processed = ui_pipeline.identify(top_n=3)
    id_time = time.time() - t0
    ui_pipeline.log_identification(id_time, processed)
    display = ui_pipeline.format_display(processed)
    display = display.replace("<", "&lt;")
    return display, format_history_md(), _get_final_features_json()


def on_reset():
    old_ext = ui_pipeline.state.extraction_logs
    old_ids = ui_pipeline.state.identification_logs
    ui_pipeline.reset()
    ui_pipeline.state.extraction_logs = old_ext
    ui_pipeline.state.identification_logs = old_ids
    return ("\u2705 已重置",) + _clear_dropdowns() + ("等待辨識...", format_history_md(), {})


# ── Build Gradio UI ──

with gr.Blocks(title="JungleSurvivor v2", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# \U0001f33f JungleSurvivor v2 — 野外植物辨識系統")
    gr.Markdown(
        "### \U0001f4f7 拍攝與使用指南\n"
        "1. **拍照**：對準植物拍攝 **3 張或以上**不同特徵的正面照片：\n"
        "   - \u2460 整株植物全貌（須清楚拍到莖部）\n"
        "   - \u2461 葉子正上方特寫（展示葉形、葉脈、葉緣）\n"
        "   - \u2462 花朵或果實正面特寫（若有的話）\n"
        "2. **上傳** \u2192 逐張點選「AI 萃取特徵」，每上傳一張會自動合併\n"
        "3. **手動修正**：查看下方特徵選單，修正 AI 判斷有誤的項目\n"
        "4. **辨識** \u2192 查看結果中的生存用途資訊 \u2192 翻閱下方「叢林生存指南」\n\n"
        "> \U0001f4a1 手動輸入越多正確特徵，辨識結果越準確。"
    )

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("## \U0001f4f8 輸入")
            image_input = gr.Image(label="上傳植物照片", type="numpy")
            upload_btn = gr.Button("\U0001f4f7 AI 萃取特徵", variant="primary")
            status_output = gr.Textbox(label="萃取狀態", interactive=False, lines=4)

            with gr.Accordion("\U0001f4cb JSON 測試", open=False):
                json_input = gr.Textbox(label="JSON 特徵", lines=4,
                    placeholder='{"growth_form":"herb","leaf":{"shape":"heart",...}}')
                json_btn = gr.Button("\U0001f4cb 載入 JSON")

        with gr.Column(scale=2):
            gr.Markdown("## \U0001f4cb 特徵（直接編輯下拉選單即可覆蓋）")

            dropdowns = []
            with gr.Tabs():
                for _sec in SECTION_ORDER:
                    _sec_schema = _schema_clean.get(_sec, {})
                    _sec_attrs = [(_a, _d) for _a, _d in _sec_schema.items() if _d["type"] != "boolean"]
                    if not _sec_attrs:
                        continue
                    with gr.TabItem(SECTION_NAMES.get(_sec, _sec)):
                        for _a, _d in _sec_attrs:
                            _is_multi = _d["type"] == "array"
                            _enum_vals = DERIVED_ENUMS.get(_sec, {}).get(_a, [])
                            _choices = [(val_to_zh(v), v) for v in _enum_vals]
                            _dd = gr.Dropdown(
                                choices=_choices,
                                multiselect=_is_multi,
                                label=ATTR_NAMES.get(_a, _a),
                                interactive=True,
                            )
                            dropdowns.append(_dd)

            with gr.Row():
                identify_btn = gr.Button("\U0001f50d 開始辨識", variant="primary", size="lg")
                reset_btn = gr.Button("\U0001f504 重新開始", variant="stop")

            gr.Markdown("## 結果")
            result_output = gr.Markdown("等待辨識...")

            with gr.Accordion("\U0001f4ca 歷史紀錄", open=False):
                history_display = gr.Markdown("尚無紀錄。")

            with gr.Accordion("\U0001f50d 目前特徵 JSON", open=False):
                feature_json_display = gr.JSON(label="合併後特徵（AI + 使用者覆蓋）")

    gr.Markdown("---")
    gr.Markdown("## \U0001f33f 叢林生存指南")
    gr.Markdown("辨識植物後，可在此查詢烹調處理方式、藥用方法、求生技巧等實用資訊。")
    with gr.Tabs():
        for _gcat in GUIDE_ORDER:
            _ginfo = GUIDE_RENDERED.get(_gcat, {})
            if _ginfo:
                _gt = f'{_ginfo.get("icon","")} {_ginfo.get("title", _gcat)}'
                with gr.TabItem(_gt):
                    gr.HTML(_ginfo.get("html", ""))

    # ── Event Bindings ──
    _up_outs = [status_output] + dropdowns + [history_display, feature_json_display]
    upload_btn.click(on_upload_photo, [image_input] + dropdowns, _up_outs)
    json_btn.click(on_paste_json, [json_input] + dropdowns, _up_outs)
    identify_btn.click(on_identify, dropdowns, [result_output, history_display, feature_json_display])
    _rst_outs = [status_output] + dropdowns + [result_output, history_display, feature_json_display]
    reset_btn.click(on_reset, [], _rst_outs)

demo.launch(share=True, debug=True)